# B5 · Modèles de diffusion : un DDPM minimal — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b5_diffusion/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B5.Q1 à B5.Q11, B5.R1 à B5.R3, B5.1 à B5.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B5.12 | linear_beta_schedule, cosine_beta_schedule et noise_schedule | 🔨 | ★★ | 20 |
|  | B5.13 | Prédire l'allure d'un chiffre bruité aux pas 100, 300 et 600, planning linéaire puis cosinus | 🔮 | ★ | 10 |
|  | B5.14 | q_sample : bruiter un batch d'images à des pas différents | 🔨 | ★★ | 20 |
|  | B5.15 | Lire ᾱ_t, le rapport signal sur bruit et la grille de bruitage des deux plannings | 📈 | ★ | 15 |
|  | B5.16 | q_posterior_mean_variance et predict_x0_from_eps | 🔨 | ★★ | 30 |
|  | B5.17 | ddpm_step et ddim_step | 🔨 | ★★★ | 40 |
|  | B5.18 | Avec le vrai bruit, DDPM et DDIM retrouvent-ils x_0 ? Contrôle sur des données 2D | 🔬 | ★★ | 25 |
| B | B5.19 | Diffusion en 2D : un MLP débruiteur conditionné par le pas de temps, entraîné puis échantillonné | 📦 | ★★ | 30 |
|  | B5.20 | Flow matching en 2D : même MLP, cible « vitesse » et intégration d'Euler | 🔬 | ★★★ | 40 |
| C | B5.21 | Un U-Net minuscule en PyTorch : blocs convolutifs, connexions de saut, encodage du pas de temps | 📦 | ★★★ | 50 |
|  | B5.22 | Le DDPM qui ne produit que du gris : quatre bugs d'indices, de racines et d'échelle | 🐛 | ★★ | 25 |
|  | B5.23 | Entraîner un DDPM sur MNIST : boucle, moyenne exponentielle des poids (EMA), courbe de loss 🚀 | 📦 | ★★★ | 60 |
|  | B5.24 | Échantillonner : DDPM à 1 000 pas contre DDIM à 50 pas, qualité et temps | 📦 | ★★ | 30 |
|  | B5.25 | Nombre de pas DDIM, planning linéaire ou cosinus : distance de Fréchet contre temps de calcul 🚀 | 🔬 | ★★★ | 45 |
|  | B5.26 | Prédire l'effet de w = 0, 1, 3 et 7 sur la netteté et la diversité | 🔮 | ★ | 10 |
|  | B5.27 | Diffusion conditionnelle et guidance sans classifieur sur Fashion-MNIST 🚀 | 📦 | ★★★ | 60 |
|  | B5.28 | Mémorisation : plus proches voisins des échantillons, entraînement sur 500 images contre 60 000 🚀 | 🔬 | ★★★ | 40 |
|  | B5.29 | Distance de Fréchet : situer le DDPM face au VAE (ch. 25) et au DCGAN (ch. 27) sur MNIST, avec le code d'évaluation du mini-projet MP6 | 📦 | ★★★ | 30 |
|  | B5.30 | Un échantillonneur autonome : sample.py avec --steps, --seed et --guidance, images et paramètres sauvegardés | 🛠️ | ★★ | 25 |
|  | B5.31 | Défi : la plus petite distance de Fréchet sur Fashion-MNIST en au plus 50 pas d'échantillonnage 🚀 | 🏆 | ★★★★ | 120 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

## Objectifs et rappel express

- Dériver la forme fermée du bruitage et le posterior d'un pas, et calculer à la main un pas DDPM et un pas DDIM.
- Implémenter en NumPy les plannings de bruit, le bruitage et les pas d'échantillonnage, et les contrôler par l'aller-retour avec le vrai bruit.
- Écrire et entraîner un petit U-Net conditionné par le pas, l'échantillonner par DDPM et DDIM, et le conditionner par la classe avec la guidance sans classifieur.
- Évaluer un générateur (distance de Fréchet, plus proches voisins) et le situer face au VAE et au GAN.

**Rappel express.** Pas de l'article $t = 1..T$, indice du code $t - 1$. $\alpha_t = 1 - \beta_t$, $\bar\alpha_t = \prod_{s \le t} \alpha_s$, $\mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1 - \bar\alpha_t}\,\boldsymbol{\varepsilon}$, $\operatorname{SNR}(t) = \bar\alpha_t / (1 - \bar\alpha_t)$. Posterior : $\tilde\beta_t = \frac{1 - \bar\alpha_{t-1}}{1 - \bar\alpha_t}\beta_t$. DDPM : $\mathbf{x}_{t-1} = \frac{1}{\sqrt{\alpha_t}}\big(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1 - \bar\alpha_t}}\boldsymbol{\varepsilon}_\theta\big) + \sigma_t\mathbf{z}$, sans bruit au dernier pas. DDIM ($\eta = 0$) : $\hat{\mathbf{x}}_0 = (\mathbf{x}_t - \sqrt{1 - \bar\alpha_t}\,\boldsymbol{\varepsilon}_\theta) / \sqrt{\bar\alpha_t}$, puis $\mathbf{x}_{t'} = \sqrt{\bar\alpha_{t'}}\,\hat{\mathbf{x}}_0 + \sqrt{1 - \bar\alpha_{t'}}\,\boldsymbol{\varepsilon}_\theta$. Guidance : $\tilde{\boldsymbol{\varepsilon}} = (1 + w)\,\boldsymbol{\varepsilon}_\theta(\mathbf{x}_t, c) - w\,\boldsymbol{\varepsilon}_\theta(\mathbf{x}_t, \varnothing)$, `guidance_scale` $= 1 + w$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B5.1 à B5.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; un vecteur : une liste de nombres (`[0.5, -1.25]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ B5.6 et B5.7, l'estimation 🧮 B5.8, la réflexion (🗣️ B5.9, ⚖️ B5.10, 📄 B5.11) et l'entretien se corrigent avec `05_solutions.md`.

In [ ]:
# The short answers only use the numbers of their statements (02_exercices.md). Steps: t = 1..T in the
# formulas (Ho et al.), indices 0..T-1 in the code (step t is index t - 1).
import math

import numpy as np

# ✏️ B5.1: classifier-free guidance on two pixels, (1 + w) eps_c - w eps_uncond
EC_B51, EU_B51 = np.array([0.8, -0.5]), np.array([0.5, 0.1])


def cfg_b51(w, cond=EC_B51, uncond=EU_B51):
    return [float(v) for v in (1 + w) * cond - w * uncond]


W0_B51, W1_B51, W3_B51 = cfg_b51(0), cfg_b51(1), cfg_b51(3)
GAP_B51 = 3 * float(np.linalg.norm(EC_B51 - EU_B51))                # || eps_tilde - eps_c || for w = 3
EVALS_B51 = 2 * 50                                                  # two predictions per step, 50 DDIM steps

# ✏️ B5.2: toy linear schedule T = 4, beta from 0.05 to 0.35 (it serves B5.4 to B5.7 too)
BETAS_B52 = np.linspace(0.05, 0.35, 4)
ALPHAS_B52 = 1 - BETAS_B52
AB_B52 = np.cumprod(ALPHAS_B52)                                     # alpha_bar, indices 0..3 (steps 1..4)
BETA2_B52, AB2_B52, AB4_B52 = float(BETAS_B52[1]), float(AB_B52[1]), float(AB_B52[3])
KEEP4_B52 = math.sqrt(AB4_B52)
APPROX4_B52 = math.exp(-float(BETAS_B52.sum()))
SUM1000_B52 = 1000 * (1e-4 + 0.02) / 2                              # arithmetic series: T (beta_1 + beta_T) / 2
AB1000_E5_B52 = math.exp(-SUM1000_B52) * 1e5
KEEP1000_PCT_B52 = 100 * math.sqrt(AB1000_E5_B52 * 1e-5)
SUM400_B52 = 400 * (1e-4 + 0.02) / 2
SUM400_SCALED_B52 = 400 * (1e-4 + 0.02) * (1000 / 400) / 2

# ✏️ B5.3: signal-to-noise ratio of the linear schedule (approximation exp(-S(t))) and of the cosine schedule (exact)
T_B53, B1_B53, BT_B53, S_B53 = 1000, 1e-4, 0.02, 0.008


def lin_sum_b53(t, square=False):
    """S(t), the sum of the first t betas; square=True: the t**2 slip instead of t (t - 1)."""
    return t * B1_B53 + (BT_B53 - B1_B53) * (t * t if square else t * (t - 1)) / (2 * (T_B53 - 1))


def snr_b53(ab):
    return ab / (1 - ab)


def cos_ab_b53(u, s=S_B53):
    f = lambda v: math.cos((v + s) / (1 + s) * math.pi / 2) ** 2
    return f(u) / f(0)


def lin_cross_b53(target_sum):
    """The t (real) where S(t) = target_sum: a t**2 + b t - target_sum = 0, then the fraction t / T."""
    a = (BT_B53 - B1_B53) / (2 * (T_B53 - 1))
    b = B1_B53 - a
    return (-b + math.sqrt(b * b + 4 * a * target_sum)) / (2 * a) / T_B53


def cos_cross_b53(alpha_bar, s=S_B53):
    """The fraction u where the cosine alpha_bar(u) equals alpha_bar."""
    f0 = math.cos(s / (1 + s) * math.pi / 2) ** 2
    theta = math.acos(math.sqrt(alpha_bar * f0))
    return theta * 2 / math.pi * (1 + s) - s


S500_B53 = lin_sum_b53(500)
LIN500_B53 = snr_b53(math.exp(-S500_B53))
COS500_B53 = snr_b53(cos_ab_b53(0.5))
LIN1_B53, COS1_B53 = lin_cross_b53(math.log(2)), cos_cross_b53(0.5)           # SNR = 1: alpha_bar = 1/2
LIN01_B53, COS01_B53 = lin_cross_b53(math.log(11)), cos_cross_b53(1 / 11)     # SNR = 0.1: alpha_bar = 1/11

# ✏️ B5.4: one DDPM step at step t = 3 (index 2) of the toy schedule
I_B54, X3_B54, EPS_B54, Z_B54 = 2, 0.8, 0.5, -1.0


def ddpm_mean_b54(x, eps, i=I_B54, sign=-1.0, root=True):
    """Mean of Algorithm 2; sign=+1: the sign slip; root=False: beta / (1 - alpha_bar) without the square root."""
    ab = float(AB_B52[i])
    coef = BETAS_B52[i] / (math.sqrt(1 - ab) if root else (1 - ab))
    return float((x + sign * coef * eps) / math.sqrt(ALPHAS_B52[i]))


def posterior_b54(i=I_B54):
    """coef1, coef2 and beta tilde of the posterior at index i (alpha_bar_prev = 1 at index 0)."""
    ab, ab_prev = float(AB_B52[i]), (float(AB_B52[i - 1]) if i > 0 else 1.0)
    beta = float(BETAS_B52[i])
    return (beta * math.sqrt(ab_prev) / (1 - ab), (1 - ab_prev) * math.sqrt(ALPHAS_B52[i]) / (1 - ab),
            beta * (1 - ab_prev) / (1 - ab))


COEF_B54 = float(BETAS_B52[I_B54] / math.sqrt(1 - AB_B52[I_B54]))
MEAN_B54 = ddpm_mean_b54(X3_B54, EPS_B54)
C1_B54, C2_B54, BTILDE_B54 = posterior_b54()
X2_B54 = MEAN_B54 + math.sqrt(BTILDE_B54) * Z_B54
X2_BETA_B54 = MEAN_B54 + math.sqrt(BETAS_B52[I_B54]) * Z_B54
X3C_B54, EPSC_B54 = 1.5, -0.6
X0HAT_C_B54 = (X3C_B54 - math.sqrt(1 - AB_B52[I_B54]) * EPSC_B54) / math.sqrt(AB_B52[I_B54])
CLIP_MEAN_B54 = C1_B54 * float(np.clip(X0HAT_C_B54, -1, 1)) + C2_B54 * X3C_B54
NOCLIP_MEAN_B54 = C1_B54 * X0HAT_C_B54 + C2_B54 * X3C_B54
LAST_B54 = ddpm_mean_b54(0.5, 0.2, i=0)                                         # step 1 (index 0): no noise

# ✏️ B5.5: DDIM from step 4 (index 3), x_4 = -0.5, eps_theta = -1.2
X4_B55, EPS_B55 = -0.5, -1.2


def x0hat_b55(x=X4_B55, eps=EPS_B55, i=3):
    return float((x - math.sqrt(1 - AB_B52[i]) * eps) / math.sqrt(AB_B52[i]))


def ab_b55(i):
    return 1.0 if i == -1 else float(AB_B52[i])


def ddim_b55(i, i_prev, eta=0.0, x=X4_B55, eps=EPS_B55, noise_term=True):
    """The deterministic part of a DDIM step (z = 0), and sigma."""
    a, ap = ab_b55(i), ab_b55(i_prev)
    sigma = eta * math.sqrt((1 - ap) / (1 - a)) * math.sqrt(1 - a / ap)
    x0 = (x - math.sqrt(1 - a) * eps) / math.sqrt(a)
    return math.sqrt(ap) * x0 + (math.sqrt(1 - ap - sigma ** 2) * eps if noise_term else 0.0), sigma


X0HAT_B55 = x0hat_b55()
X1_B55 = ddim_b55(3, 0)[0]
X2_B55 = ddim_b55(3, 1)[0]
SIGMA2_43_B55 = ddim_b55(3, 2, eta=1.0)[1] ** 2
SIGMA_41_B55 = ddim_b55(3, 0, eta=1.0)[1]
DIR_41_B55 = math.sqrt(1 - AB_B52[0] - SIGMA_41_B55 ** 2)

# quizzes and recalls with a computation
Q1E = 1000 * 0.02
Q2B = float(np.mean([-1, 0, 2]))
Q2C = float(np.mean((np.array([-1, 0, 2]) - Q2B) ** 2))
Q3A, Q3B, Q3D = math.sqrt(0.64), math.sqrt(1 - 0.64), 0.64 / 0.36
Q4B = (0.5 - math.sqrt(1 - 0.64) * 0.2) / math.sqrt(0.64)
Q5C = 128 // 2
Q6B, Q6D = 128 + 64, 32 // 2 ** 3
Q7A = 1000 * 0.05
Q9A = 3 * 0.3 - 2 * 0.5
Q11A = (256 // 4) ** 2 * 3
R1B = 2 * math.pi
R2A = (0 - 1) ** 2 + (1 + 4 - 2 * math.sqrt(1 * 4))
R2E = 0 ** 2 + (1 + 0 - 2 * math.sqrt(1 * 0))
R3A = 0.5 + math.exp(0.5 * math.log(0.25)) * (-2)
R3B = 0.5 * (0.5 ** 2 + 0.25 - 1 - math.log(0.25))

# The same answers from the libraries (no download): mylearn (reference) and, if installed, diffusers
D = mylearn.diffusion
toy = D.noise_schedule(D.linear_beta_schedule(4, 0.05, 0.35))
assert np.allclose(toy["betas"], BETAS_B52) and np.allclose(toy["alpha_bars"], AB_B52)

# ✏️ B5.2: the sums of the betas, and the exact alpha_bar_T that the approximation overestimates
lin1000 = D.noise_schedule(D.linear_beta_schedule(1000))
assert math.isclose(lin1000["betas"].sum(), SUM1000_B52)
assert math.isclose(D.linear_beta_schedule(400).sum(), SUM400_B52)
assert math.isclose(D.linear_beta_schedule(400, 1e-4 * 2.5, 0.02 * 2.5).sum(), SUM400_SCALED_B52)
assert lin1000["alpha_bars"][-1] < AB1000_E5_B52 * 1e-5 and toy["alpha_bars"][-1] < APPROX4_B52

# ✏️ B5.3: the exact schedules of the library (index i = step i + 1) cross the thresholds at the same 2 decimals
cos1000 = D.noise_schedule(D.cosine_beta_schedule(1000))
for sched, answers in ((lin1000, (LIN1_B53, LIN01_B53)), (cos1000, (COS1_B53, COS01_B53))):
    snr = sched["alpha_bars"] / (1 - sched["alpha_bars"])
    for threshold, answer in zip((1.0, 0.1), answers):
        first_step = int(np.argmax(snr < threshold)) + 1
        assert round(first_step / 1000, 2) == round(answer, 2), (threshold, first_step, answer)
assert math.isclose(cos1000["alpha_bars"][499], cos_ab_b53(0.5), rel_tol=1e-9)
assert round(float(lin1000["alpha_bars"][499] / (1 - lin1000["alpha_bars"][499])), 2) == round(LIN500_B53, 2)

# ✏️ B5.4: ddpm_step; its mean is step(x) - step(0) with the same seed, its variance posterior_variance
def mean_of_ddpm_step(x, i, eps, **kw):
    one = D.ddpm_step(np.array([[x]]), i, np.array([[eps]]), toy, rng=np.random.default_rng(0), **kw)
    zero = D.ddpm_step(np.array([[0.0]]), i, np.array([[0.0]]), toy, rng=np.random.default_rng(0),
                       variance=kw.get("variance", "posterior"))
    return float((one - zero)[0, 0])


assert math.isclose(mean_of_ddpm_step(X3_B54, I_B54, EPS_B54), MEAN_B54)
assert math.isclose(mean_of_ddpm_step(X3_B54, I_B54, EPS_B54, variance="beta"), MEAN_B54)
assert math.isclose(mean_of_ddpm_step(X3C_B54, I_B54, EPSC_B54, clip_x0=1.0), CLIP_MEAN_B54)
assert math.isclose(toy["posterior_variance"][I_B54], BTILDE_B54)
assert math.isclose(toy["posterior_mean_coef1"][I_B54], C1_B54) and math.isclose(toy["posterior_mean_coef2"][I_B54], C2_B54)
for variance in ("posterior", "beta"):                                 # step 1 (index 0): no noise, whatever the seed
    assert math.isclose(float(D.ddpm_step(np.array([[0.5]]), 0, np.array([[0.2]]), toy, rng=7, variance=variance)[0, 0]),
                        LAST_B54)

# ✏️ B5.5: predict_x0_from_eps and ddim_step (eta = 0: deterministic)
assert math.isclose(float(D.predict_x0_from_eps(np.array([X4_B55]), [3], np.array([EPS_B55]), AB_B52)[0]), X0HAT_B55)
x4 = np.array([[X4_B55]]); e4 = np.array([[EPS_B55]])
assert math.isclose(float(D.ddim_step(x4, 3, 0, e4, AB_B52)[0, 0]), X1_B55)
assert math.isclose(float(D.ddim_step(x4, 3, 1, e4, AB_B52)[0, 0]), X2_B55)
assert math.isclose(float(D.ddim_step(D.ddim_step(x4, 3, 1, e4, AB_B52), 1, 0, e4, AB_B52)[0, 0]), X1_B55)
assert math.isclose(float(D.ddim_step(x4, 3, -1, e4, AB_B52)[0, 0]), X0HAT_B55)
assert math.isclose(SIGMA2_43_B55, toy["posterior_variance"][3])

# recalls: positional_encoding (B3), frechet_distance (ch. 27), reparameterize and gaussian_kl (ch. 25)
pe = mylearn.attention.positional_encoding(1000, 128)
assert pe[0, 1] == 1.0 and pe.shape[1] // 2 == Q5C
assert np.allclose(pe[:, 0], np.sin(np.arange(1000)))                 # column 0: sin(p), period 2 pi
assert 2 * math.pi * 10000 ** (126 / 128) > 1000                      # the slowest pair: less than one turn
fd = mylearn.gen_metrics.frechet_distance
assert math.isclose(fd([0.0], [[1.0]], [1.0], [[4.0]]), R2A) and math.isclose(fd([0.0], [[1.0]], [0.0], [[0.0]]), R2E)
z, _ = mylearn.vae.reparameterize([[0.5]], [[math.log(0.25)]], rng=0)
assert z.shape == (1, 1)
assert math.isclose(float(mylearn.vae.gaussian_kl([[0.5]], [[math.log(0.25)]])[0]), R3B)
print("✅ the paper answers agree with mylearn (reference): diffusion, attention, gen_metrics, vae")

# Optional: diffusers' schedulers on the toy schedule (float64 tensors, noise set to zero)
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")   # tqdm, imported by diffusers
try:
    import torch
    import diffusers.schedulers.scheduling_ddpm as ddpm_module
    from diffusers import DDIMScheduler, DDPMScheduler
except ImportError:
    print("ℹ️ diffusers is not installed: the scheduler comparison is skipped (the closed forms above are checked)")
else:
    def as_double(scheduler, names):
        for name in names:
            setattr(scheduler, name, getattr(scheduler, name).double())
        return scheduler

    def diffusers_ddpm(x, i, eps, clip=None):
        sch = as_double(DDPMScheduler(num_train_timesteps=4, trained_betas=BETAS_B52.tolist(),
                                      clip_sample=clip is not None, clip_sample_range=clip or 1.0),
                        ("betas", "alphas", "alphas_cumprod", "one"))
        saved = ddpm_module.randn_tensor
        ddpm_module.randn_tensor = lambda shape, **kw: torch.zeros(shape, dtype=torch.float64)
        try:
            out = sch.step(torch.tensor([[eps]], dtype=torch.float64), i, torch.tensor([[x]], dtype=torch.float64))
        finally:
            ddpm_module.randn_tensor = saved
        return float(out.prev_sample), float(sch._get_variance(i))

    def diffusers_ddim(x, i, eps, n_steps):
        sch = as_double(DDIMScheduler(num_train_timesteps=4, trained_betas=BETAS_B52.tolist(), clip_sample=False,
                                      set_alpha_to_one=True), ("alphas_cumprod", "final_alpha_cumprod"))
        sch.set_timesteps(n_steps)
        out = sch.step(torch.tensor([[eps]], dtype=torch.float64), i, torch.tensor([[x]], dtype=torch.float64),
                       eta=0.0)
        return float(out.prev_sample), float(out.pred_original_sample), sch

    mean, var = diffusers_ddpm(X3_B54, I_B54, EPS_B54)
    assert math.isclose(mean, MEAN_B54, rel_tol=1e-6) and math.isclose(var, BTILDE_B54, rel_tol=1e-6)
    assert math.isclose(diffusers_ddpm(X3C_B54, I_B54, EPSC_B54, clip=1.0)[0], CLIP_MEAN_B54, rel_tol=1e-6)
    assert math.isclose(diffusers_ddpm(0.5, 0, 0.2)[0], LAST_B54, rel_tol=1e-6)
    prev, x0_pred, ddim = diffusers_ddim(X4_B55, 3, EPS_B55, 2)          # 4 // 2 = 2: index 3 -> index 1
    assert math.isclose(prev, X2_B55, rel_tol=1e-6) and math.isclose(x0_pred, X0HAT_B55, rel_tol=1e-6)
    assert math.isclose(diffusers_ddim(X4_B55, 3, EPS_B55, 1)[0], X0HAT_B55, rel_tol=1e-6)   # index 3 -> clean
    assert math.isclose(float(ddim._get_variance(3, 2)), SIGMA2_43_B55, rel_tol=1e-6)
    assert math.isclose(float(ddim._get_variance(3, 0)), SIGMA_41_B55 ** 2, rel_tol=1e-6)
    print("✅ ✏️ B5.4 and B5.5 also agree with diffusers' DDPMScheduler and DDIMScheduler")

**B5.Q1 — Diffusion, VAE, GAN : comment chacun fabrique-t-il une image ?**

In [ ]:
wb.record("B5.Q1a", "A", mistakes={"relis l'encadré 🔗 « Lien avec les ch. 25 et 27 » : combien de passes du décodeur pour générer avec un VAE ?": "B", "pour générer, part-on d'une image du dataset ? Relis l'encadré 🔗 de 205.1": "C"})
wb.record("B5.Q1b", False, mistakes={"relis « Détruire est facile » (§205.1) : le processus direct s'apprend-il ?": True})
wb.record("B5.Q1c", "C", mistakes={"relis l'encadré 🔗 de 205.1 : la diffusion s'entraîne-t-elle contre un adversaire ?": "A", "le VAE a-t-il un discriminateur ? Relis l'encadré 🔗 de 205.1": "B"})
wb.record("B5.Q1d", True, mistakes={"relis l'encadré 🔗 « Lien avec les ch. 25 et 27 » : quelle taille ont les codes de la diffusion ?": False})
wb.record("B5.Q1e", round(Q1E), mistakes={"c'est le nombre de passes : on demande des secondes": 1000,
          "vérifie la multiplication par 1 000": 2})

**B5.Q2 — Pourquoi détruire l'image en mille petits pas de bruit plutôt qu'en un seul ?**

In [ ]:
wb.record("B5.Q2a", "B", mistakes={"relis l'encadré 🧮 « la MSE apprend une moyenne » (§205.1) : que rend un réseau entraîné par MSE quand plusieurs cibles sont possibles ?": "A", "un réseau entraîné par MSE recopie-t-il un exemple ? Relis l'encadré 🧮 de 205.1": "C"})
wb.record("B5.Q2b", Q2B, decimals=2, mistakes={"c'est la médiane : la MSE est minimisée par une autre grandeur (encadré 🧮 de 205.1)": 0.0,
          "c'est le milieu entre la plus petite et la plus grande cible : relis l'encadré 🧮 de 205.1": 0.5})
wb.record("B5.Q2c", Q2C, decimals=2, mistakes={"c'est la MSE de la prédiction 0 : prends la prédiction de b)": 5 / 3,
          "la MSE est une moyenne des carrés des écarts : divise par le nombre de cibles": 3 * Q2C,
          "élève les écarts au carré avant de faire la moyenne": float(np.mean(np.abs(np.array([-1, 0, 2]) - Q2B)))})
wb.record("B5.Q2d", True, mistakes={"relis « Ce qu'on voudrait » (§205.3) : que dit-on de ce pas quand β_t est petit ?": False})
wb.record("B5.Q2e", False, mistakes={"relis l'approximation ln ᾱ_T ≈ −Σβ (§205.2) : à somme égale, que deviennent les β quand il y a deux fois plus de pas ?": True})

**B5.Q3 — ᾱ_t : quelle part de l'image reste-t-il au pas t ?**

In [ ]:
wb.record("B5.Q3a", Q3A, decimals=2, mistakes={"c'est ᾱ_t : relis la forme fermée, le coefficient de x₀ est-il ᾱ_t ?": 0.64,
          "relis la question : on demande le coefficient de x₀, pas celui du bruit": Q3B})
wb.record("B5.Q3b", Q3B, decimals=2, mistakes={"c'est 1 − ᾱ_t : relis la forme fermée, le coefficient du bruit est-il 1 − ᾱ_t ?": 0.36,
          "relis la question : on demande le coefficient du bruit, pas celui de x₀": Q3A})
wb.record("B5.Q3c", False, mistakes={"calcule la somme de tes réponses a) et b) ; relis la dernière phrase de « La forme fermée »": True})
wb.record("B5.Q3d", Q3D, decimals=2, mistakes={"le SNR met la part de l'image au numérateur": 0.36 / 0.64,
          "le SNR est un rapport de variances, pas de coefficients": Q3A / Q3B})
wb.record("B5.Q3e", "C", mistakes={"relis la définition de ᾱ_t : un produit de nombres plus petits que 1 peut-il croître ?": "A", "regarde la figure `plannings.png` (§205.2) : ᾱ_t reste-t-il constant ?": "B"})

**B5.Q4 — Prédire le bruit, l'image ou la vitesse : la même information ?**

In [ ]:
wb.record("B5.Q4a", "A", mistakes={"relis « Prédire le bruit » (§205.3) : la formule de x̂₀ est-elle une approximation quand le bruit est le vrai ?": "B", "relis « Prédire le bruit » (§205.3) : la forme fermée s'inverse-t-elle ?": "C"})
wb.record("B5.Q4b", Q4B, decimals=3, mistakes={"relis la formule de x̂₀ : on divise par la racine de ᾱ_t, et le bruit est multiplié par la racine de 1 − ᾱ_t": (0.5 - 0.36 * 0.2) / 0.64,
          "vérifie le signe : on retire le bruit de x_t": (0.5 + 0.6 * 0.2) / 0.8,
          "n'oublie pas de diviser par la racine de ᾱ_t": 0.5 - 0.6 * 0.2})
wb.record("B5.Q4c", True, mistakes={"relis le paragraphe qui suit la moyenne μ_θ (§205.3) : qu'ont trouvé Ho et coll. pour la prédiction de x₀ ?": False})
wb.record("B5.Q4d", "C", mistakes={"le réseau reçoit-il encore t quand il prédit le bruit ? Relis « Dire au réseau quel est le pas »": "A", "le bruit est-il toujours plus petit que l'image ? Relis la raison donnée en 205.3": "B"})
wb.record("B5.Q4e", False, mistakes={"relis le paragraphe qui suit l'encadré 🧮 sur Euler (§205.9) : les deux « v » sont-ils la même grandeur ?": True})

**B5.Q5 — Le pas de temps est une entrée du réseau : pourquoi, et comment l'encoder ?**

In [ ]:
wb.record("B5.Q5a", "C", mistakes={"relis « Reconstruire s'apprend » (§205.1) : combien de réseaux sert-on pour tous les pas ?": "A", "ᾱ_t se calcule-t-il dans le réseau ? Relis « Dire au réseau quel est le pas »": "B"})
wb.record("B5.Q5b", False, mistakes={"relis « Dire au réseau quel est le pas » (§205.4) : que dit la fiche d'un nombre de 0 à 999 donné tel quel ?": True})
wb.record("B5.Q5c", Q5C, mistakes={"c'est le nombre de colonnes : chaque couple en occupe deux": 128,
          "relis l'appel : d_model = 128": 32})
wb.record("B5.Q5d", "B", mistakes={"relis « Un bloc résiduel conditionné par le pas » : l'encodage est-il concaténé ?": "A", "relis la ligne `h = h + self.time(temb)[:, :, None, None]` du bloc résiduel": "C"})
wb.record("B5.Q5e", True, mistakes={"relis la fin de « Dire au réseau quel est le pas » : que dit la fiche des permutations de colonnes ?": False})

**B5.Q6 — U-Net : à quoi servent les connexions de saut ?**

In [ ]:
wb.record("B5.Q6a", "A", mistakes={"relis « Le U-Net et ses connexions de saut » : à quelle résolution relient-elles l'encodeur et le décodeur ?": "B", "c'est une connexion résiduelle : relis le piège classique « deux skip connections »": "C"})
wb.record("B5.Q6b", Q6B, mistakes={"la connexion de saut concatène : les canaux de l'encodeur s'ajoutent à ceux du décodeur": 128,
          "relis la question : 128 canaux du décodeur et 64 de l'encodeur": 256,
          "relis la question : la convolution suivante reçoit aussi les canaux du décodeur": 64})
wb.record("B5.Q6c", True, mistakes={"relis le paragraphe qui suit la figure du U-Net (§205.4) : que deviendrait la sortie sans connexions de saut ?": False})
wb.record("B5.Q6d", Q6D, mistakes={"recompte les réductions de l'énoncé : en as-tu appliqué une de moins ?": 8,
          "tu n'as appliqué qu'une réduction : relis l'énoncé": 16})
wb.record("B5.Q6e", "C", mistakes={"la group norm a-t-elle des paramètres γ et β ? Relis le compte du bloc résiduel": "A", "relis la définition de la group norm (§205.4) : que normalise-t-elle ?": "B"})

**B5.Q7 — Échantillonnage DDPM : pourquoi est-il lent et aléatoire ?**

In [ ]:
wb.record("B5.Q7a", round(Q7A), mistakes={"c'est le nombre de passes : on demande des secondes": 1000,
          "vérifie la multiplication : as-tu décalé la virgule ?": 500})
wb.record("B5.Q7b", False, mistakes={"relis « Ce que coûte un échantillon » (§205.5) : que tire-t-on à chaque pas ?": True})
wb.record("B5.Q7c", True, mistakes={"relis la phrase qui suit l'algorithme 2 (§205.5)": False})
wb.record("B5.Q7d", "A", mistakes={"le planning interdit-il les batches ? Relis « Ce que coûte un échantillon »": "B", "un réseau peut-il servir plusieurs calculs ? Relis « Ce que coûte un échantillon » : de quoi chaque passe a-t-elle besoin ?": "C"})
wb.record("B5.Q7e", "B", mistakes={"relis « Quelle variance ? » (§205.5) : le réseau prédit-il la variance ?": "A", "relis la fin de 205.3 : β̃_t ≤ β_t, et l'égalité ne vaut qu'au premier pas ?": "C"})

**B5.Q8 — DDIM : moins de pas avec le même réseau, vrai ou faux**

In [ ]:
wb.record("B5.Q8a", False, mistakes={"relis « L'idée de DDIM » (§205.6) : l'entraînement change-t-il ?": True})
wb.record("B5.Q8b", True, mistakes={"relis la formule DDIM avec η = 0 : y a-t-il un tirage ?": False})
wb.record("B5.Q8c", True, mistakes={"relis « Le paramètre η » (§205.6) : que vaut σ² quand η = 1 et t' = t − 1 ?": False})
wb.record("B5.Q8d", True, mistakes={"relis l'étape 2 de DDIM (§205.6) : t' doit-il être le pas juste avant ?": False})
wb.record("B5.Q8e", False, mistakes={"relis « Pourquoi c'est plus rapide » (§205.6) : une trajectoire se suit-elle aussi bien avec très peu de pas ?": True})

**B5.Q9 — Guidance sans classifieur : ce que règle l'échelle w**

In [ ]:
wb.record("B5.Q9a", Q9A, decimals=1, mistakes={"vérifie le signe de l'extrapolation": -Q9A,
          "(1 + w) multiplie la prédiction avec condition, w celle sans condition": 3 * 0.5 - 2 * 0.3,
          "relis la question : w = 2": 0.3})
wb.record("B5.Q9b", "B", mistakes={"relis la formule avec w = 0 : quelle prédiction reste-t-il ?": "A", "relis la formule avec w = 0 : y a-t-il une moyenne ?": "C"})
wb.record("B5.Q9c", False, mistakes={"relis le paragraphe qui suit la figure de la guidance (§205.7)": True})
wb.record("B5.Q9d", 2, mistakes={"relis le piège classique « deux conventions pour l'échelle de guidance » (§205.7)": 3,
          "guidance_scale = 1 + w : résous pour w": 4})
wb.record("B5.Q9e", "C", mistakes={"relis « La guidance sans classifieur » : combien de réseaux entraîne-t-on ?": "A", "le réseau peut-il prédire sans condition s'il n'a jamais vu ∅ ? Relis « La guidance sans classifieur »": "B"})

**B5.Q10 — Un bon FID peut-il cacher de la copie des données d'entraînement ?**

In [ ]:
wb.record("B5.Q10a", True, mistakes={"relis « La mémorisation » (§205.8) : que verrait une distance de Fréchet d'un modèle qui recopie ?": False})
wb.record("B5.Q10b", "A", mistakes={"plus d'images rend-il la distance de Fréchet sensible à la copie ? Relis « La mémorisation »": "B", "la loss dit-elle quelles images sont recopiées ? Relis « La mémorisation »": "C"})
wb.record("B5.Q10c", False, mistakes={"relis la dernière puce des résultats de Carlini et coll. (§205.8)": True})
wb.record("B5.Q10d", "B", mistakes={"relis les trois règles de « La distance de Fréchet » (§205.8)": "A", "la distance dépend-elle du nombre d'images de façon proportionnelle ? Relis les trois règles de 205.8": "C"})
wb.record("B5.Q10e", False, mistakes={"relis « Fidélité et diversité » (§205.8) : un seul nombre sépare-t-il les deux qualités ?": True})

**B5.Q11 — Diffusion latente, transformers de diffusion, flow matching : ce qui a changé depuis 2020**

In [ ]:
wb.record("B5.Q11a", Q11A, mistakes={"n'oublie pas les 3 canaux du latent": (256 // 4) ** 2,
          "relis l'énoncé : le latent a 3 canaux": (256 // 4) ** 2 * 4,
          "le facteur f divise la largeur et la hauteur, chacune": 256 * 256 * 3 // 4})
wb.record("B5.Q11b", "A", mistakes={"relis « Les transformers de diffusion » (§205.9) : d'où viennent les tokens d'un DiT ?": "B", "relis « Les transformers de diffusion » : combien de tokens pour un latent de 32 × 32 × 4 ?": "C"})
wb.record("B5.Q11c", "C", mistakes={"dérive x_s par rapport à s : vérifie le signe": "A", "dérive x_s par rapport à s : le coefficient de x₀ est 1 − s": "B"})
wb.record("B5.Q11d", True, mistakes={"relis l'encadré 🧮 « la méthode d'Euler » (§205.9) : d'où vient l'erreur d'Euler ?": False})
wb.record("B5.Q11e", False, mistakes={"relis « Les transformers de diffusion » et « Flow matching et rectified flow » (§205.9)": True})

**B5.R1 — L'encodage sinusoïdal des positions**

In [ ]:
wb.record("B5.R1a", 1, mistakes={"la colonne 1 est impaire : sinus ou cosinus ?": 0})
wb.record("B5.R1b", R1B, decimals=2, mistakes={"une période de sinus est un tour complet : relis la formule de la colonne 0": math.pi,
          "relis la formule : à la colonne 0, la fréquence vaut 1": 1.0})
wb.record("B5.R1c", False, mistakes={"relis l'encodage sinusoïdal en B3 : quelles colonnes changent vite d'une position à la suivante ?": True})
wb.record("B5.R1d", True, mistakes={"calcule la période 2π · 10 000^(126/128) de la dernière paire et compare-la à 1 000": False})
wb.record("B5.R1e", "B", mistakes={"relis la formule de l'énoncé : PE[p, 2i] est-il un cosinus ?": "A", "relis la formule de l'énoncé": "C"})

**B5.R2 — Ce que mesure la distance de Fréchet et l'effondrement de mode**

In [ ]:
wb.record("B5.R2a", round(R2A), mistakes={"il manque un des deux termes de la formule : la moyenne ou la covariance": 1,
          "n'oublie pas le terme −2(Σ₁Σ₂)^(1/2)": 6},
          fractional="relis le calcul : avec ces nombres, le résultat est entier")
wb.record("B5.R2b", "A", mistakes={"la distance de Fréchet juge-t-elle chaque image à part ? Relis la formule": "B", "une distance nulle demanderait la même moyenne et la même covariance : est-ce le cas ?": "C"})
wb.record("B5.R2c", True, mistakes={"chaque terme de la formule est-il nul quand les moyennes et les covariances sont égales, et seulement alors ?": False})
wb.record("B5.R2d", False, mistakes={"relis le ch. 27 : sur quoi le FID calcule-t-il ses moyennes et covariances ?": True})
wb.record("B5.R2e", round(R2E), mistakes={"un générateur qui rend toujours la même feature a-t-il la covariance des vraies images ?": 0},
          fractional="relis le calcul : avec ces nombres, le résultat est entier")

**B5.R3 — L'astuce de reparamétrisation et le terme KL du VAE**

In [ ]:
wb.record("B5.R3a", R3A, decimals=2, mistakes={"ln σ² = ln 0,25 : σ est la racine de 0,25, pas 0,25": 0.5 + 0.25 * (-2),
          "z = μ + σ ε, avec σ = exp(0,5 ln σ²), pas ln σ²": 0.5 + math.log(0.25) * (-2),
          "vérifie le signe : ε = −2": 0.5 + 0.5 * 2})
wb.record("B5.R3b", R3B, decimals=4, mistakes={"n'oublie pas le facteur 1/2 de la formule": 2 * R3B,
          "il manque le terme − ln σ²": 0.5 * (0.25 + 0.25 - 1),
          "σ² = 0,25 : relis le terme de variance de la formule": 0.5 * (0.25 + 0.5 - 1 - math.log(0.25))})
wb.record("B5.R3c", "A", mistakes={"la reparamétrisation change-t-elle la taille du latent ? Relis le ch. 25": "B", "z reste-t-il aléatoire avec la reparamétrisation ? Relis le ch. 25": "C"})
wb.record("B5.R3d", True, mistakes={"relis l'encadré 🔗 « Lien avec la reparamétrisation (ch. 25) » (§205.3)": False})
wb.record("B5.R3e", False, mistakes={"relis « Le lien avec l'ELBO » (§205.3) : ce terme dépend-il des paramètres du réseau ?": True})

**Ex B5.1 — Guidance sans classifieur : combiner les deux prédictions pour w = 0, 1 et 3**

In [ ]:
wb.record("B5.1a", W0_B51, decimals=2, mistakes={"avec w = 0, quelle prédiction garde-t-on ? Relis la formule": [0.5, 0.1]})
wb.record("B5.1b", W1_B51, decimals=2, mistakes={"la guidance n'est pas une moyenne des deux prédictions : relis la formule": [0.65, -0.2],
          "(1 + w) multiplie la prédiction avec condition, w celle sans condition": cfg_b51(1, EU_B51, EC_B51),
          "relis la question : w = 1": W0_B51})
wb.record("B5.1c", W3_B51, decimals=2, mistakes={"(1 + w) multiplie la prédiction avec condition, w celle sans condition": cfg_b51(3, EU_B51, EC_B51),
          "c'est guidance_scale = 3, la convention de diffusers : relis le piège classique de 205.7": cfg_b51(2),
          "relis la question : w = 3": W1_B51})
wb.record("B5.1d", 4, mistakes={"relis le piège classique « deux conventions pour l'échelle de guidance » (§205.7)": 3,
          "guidance_scale = 1 + w : vérifie le sens": 2})
wb.record("B5.1e", GAP_B51, decimals=2, mistakes={"multiplie l'écart des deux prédictions par w": GAP_B51 / 3,
          "la norme euclidienne est la racine de la somme des carrés": 3 * (0.3 + 0.6),
          "relis la question : on demande l'écart à ε_c, pas la norme de ε̃": float(np.linalg.norm(W3_B51))})
wb.record("B5.1f", True, mistakes={"remplace w par −1 dans la formule": False})
wb.record("B5.1g", False, mistakes={"compare ta réponse c) aux deux prédictions, composante par composante": True})
wb.record("B5.1h", EVALS_B51, mistakes={"la guidance demande deux prédictions à chaque pas": 50,
          "relis la question : 50 pas DDIM, pas 1 000": 2 * 1000})

**Ex B5.2 — Planning linéaire : β_t, α_t et ᾱ_t pour T = 4, puis ᾱ_T pour T = 1 000**

In [ ]:
wb.record("B5.2a", BETA2_B52, decimals=2, mistakes={"4 valeurs de 0,05 à 0,35 font 3 intervalles, pas 4": 0.05 + 0.3 / 4,
          "relis la question : β_2 est la deuxième valeur, à l'indice 1": 0.25})
wb.record("B5.2b", AB2_B52, decimals=4, mistakes={"c'est α_2 : ᾱ_2 est le produit des α jusqu'au pas 2": float(ALPHAS_B52[1]),
          "ᾱ est un produit des 1 − β, pas 1 moins la somme des β": 1 - 0.05 - 0.15,
          "relis la question : on demande ᾱ_2, au pas 2": float(AB_B52[0])})
wb.record("B5.2c", AB4_B52, decimals=4, mistakes={"c'est α_4 : ᾱ_4 est le produit des α jusqu'au pas 4": float(ALPHAS_B52[3]),
          "ᾱ est un produit des 1 − β, pas 1 moins la somme des β": 1 - float(BETAS_B52.sum()),
          "on demande ᾱ_4, pas sa racine": KEEP4_B52,
          "relis la question : le produit va jusqu'au pas 4": float(AB_B52[2])})
wb.record("B5.2d", KEEP4_B52, decimals=4, mistakes={"c'est ᾱ_4 : relis la forme fermée, le coefficient de x₀ est sa racine": AB4_B52,
          "relis la question : on demande le coefficient de x₀, pas celui du bruit": math.sqrt(1 - AB4_B52)})
wb.record("B5.2e", APPROX4_B52, decimals=4, mistakes={"recompte la somme des quatre β": math.exp(-1.0),
          "relis la question : on demande l'approximation e^(−Σβ), pas la valeur exacte": AB4_B52})
wb.record("B5.2f", False, mistakes={"compare tes réponses c) et e) : calcule l'écart relatif": True})
wb.record("B5.2g", SUM1000_B52, decimals=2, mistakes={"la somme d'une suite arithmétique est T × (premier + dernier) / 2 : n'oublie pas le / 2": 2 * SUM1000_B52,
          "relis la question : on demande la somme des 1 000 β, pas leur moyenne": SUM1000_B52 / 1000,
          "relis l'énoncé : le dernier β vaut 0,02": 1000 * (1e-4 + 0.2) / 2})
wb.record("B5.2h", AB1000_E5_B52, decimals=2, mistakes={"garde la somme des β de g) sans l'arrondir à 10": math.exp(-10) * 1e5,
          "relis la question : multiplie l'approximation par 10⁵": math.exp(-SUM1000_B52) * 1e6,
          "tu as arrondi e^(−Σβ) à cinq décimales : pour un si petit nombre, garde cinq chiffres significatifs": 4.0,
          "c'est le produit exact des 1 − β : on demande l'approximation e^(−Σβ)": float(np.prod(1 - np.linspace(1e-4, 0.02, 1000))) * 1e5})
wb.record("B5.2i", KEEP1000_PCT_B52, decimals=2, mistakes={"h) est multiplié par 10⁵ : reviens à ᾱ_T avant de prendre la racine": math.sqrt(AB1000_E5_B52),
          "relis la question : en pourcentage, donc multiplié par 100": KEEP1000_PCT_B52 / 100,
          "c'est la racine du produit exact : l'énoncé demande celle de ta valeur de h)": 100 * math.sqrt(float(np.prod(1 - np.linspace(1e-4, 0.02, 1000))))})
wb.record("B5.2j", SUM400_B52, decimals=2, mistakes={"relis la question : les mêmes bornes, mais 400 β seulement": SUM1000_B52,
          "n'oublie pas le / 2 de la somme d'une suite arithmétique": 2 * SUM400_B52})
wb.record("B5.2k", SUM400_SCALED_B52, decimals=2, mistakes={"relis la question : les deux bornes sont multipliées par 1 000/400": SUM400_B52,
          "n'oublie pas le / 2 de la somme d'une suite arithmétique": 2 * SUM400_SCALED_B52})
wb.record("B5.2l", True, mistakes={"compare tes réponses g) et k)": False})

**Ex B5.3 — Rapport signal sur bruit : à quel pas un chiffre devient-il illisible, planning linéaire ou cosinus ?**

In [ ]:
wb.record("B5.3a", S500_B53, decimals=3, mistakes={"relis la formule de S(t) : t(t − 1), pas t²": lin_sum_b53(500, square=True),
          "S(500) somme les 500 premiers β, pas 500 fois la moyenne de tous les β": 500 * (1e-4 + 0.02) / 2})
wb.record("B5.3b", LIN500_B53, decimals=2, mistakes={"le SNR met la part de l'image au numérateur, celle du bruit au dénominateur": 1 / LIN500_B53,
          "c'est ᾱ_t : on demande le SNR": math.exp(-S500_B53)})
wb.record("B5.3c", COS500_B53, decimals=2, mistakes={"n'oublie pas le décalage s = 0,008": 1.0,
          "c'est ᾱ_t : on demande le SNR": cos_ab_b53(0.5)})
wb.record("B5.3d", LIN1_B53, decimals=2, mistakes={"SNR = 1 veut dire ᾱ = 1/2 : relis le lien entre ᾱ_t et S(t)": lin_cross_b53(1.0),
          "résous l'équation du second degré en t, puis divise t par T": lin_cross_b53(math.log(2)) / 2})
wb.record("B5.3e", COS1_B53, decimals=2, mistakes={"SNR = 1 veut dire ᾱ = 1/2 : c'est le carré du cosinus qui vaut 1/2, pas le cosinus": cos_cross_b53(0.25)})
wb.record("B5.3f", LIN01_B53, decimals=2, mistakes={"SNR = 0,1 ne veut pas dire ᾱ = 0,1 : écris ᾱ en fonction du SNR": lin_cross_b53(math.log(10)),
          "relis la question : le seuil de lisibilité est 0,1, pas 1": LIN1_B53})
wb.record("B5.3g", COS01_B53, decimals=2, mistakes={"n'oublie pas le décalage s = 0,008": cos_cross_b53(1 / 11, s=0.0),
          "SNR = 0,1 ne veut pas dire ᾱ = 0,1 : écris ᾱ en fonction du SNR": cos_cross_b53(0.1)})
wb.record("B5.3h", True, mistakes={"compare ta réponse f) à 0,5": False})
wb.record("B5.3i", False, mistakes={"relis ta réponse g) : quelle part des pas reste lisible ?": True})
wb.record("B5.3j", "B", mistakes={"compare f) − d) et g) − e)": "A", "calcule f) − d) et g) − e) : sont-elles égales ?": "C"})

**Ex B5.4 — Un pas DDPM à la main, en dimension 1, avec un bruit prédit donné**

In [ ]:
wb.record("B5.4a", COEF_B54, decimals=4, mistakes={"relis l'algorithme 2 : β_t est divisé par la racine de 1 − ᾱ_t": float(BETAS_B52[2] / (1 - AB_B52[2])),
          "décalage d'un cran : le pas 3 est l'indice 2, avec β_3 = 0,25": float(BETAS_B52[3] / math.sqrt(1 - AB_B52[3])),
          "relis l'algorithme 2 : c'est 1 − ᾱ_t, pas 1 − α_t": float(BETAS_B52[2] / math.sqrt(BETAS_B52[2]))})
wb.record("B5.4b", MEAN_B54, decimals=4, mistakes={"vérifie le signe : on retire le bruit prédit": ddpm_mean_b54(X3_B54, EPS_B54, sign=1.0),
          "relis l'algorithme 2 : β_t est divisé par la racine de 1 − ᾱ_t": ddpm_mean_b54(X3_B54, EPS_B54, root=False),
          "on divise par la racine de α_t, pas de ᾱ_t": float((X3_B54 - COEF_B54 * EPS_B54) / math.sqrt(AB_B52[2])),
          "décalage d'un cran : le pas 3 est l'indice 2": ddpm_mean_b54(X3_B54, EPS_B54, i=3)})
wb.record("B5.4c", BTILDE_B54, decimals=4, mistakes={"c'est β_3 : la variance du posterior a un facteur en plus": float(BETAS_B52[2]),
          "décalage d'un cran : le pas 3 est l'indice 2": posterior_b54(3)[2],
          "relis la formule de β̃ : (1 − ᾱ_{t−1}) au numérateur, (1 − ᾱ_t) au dénominateur": float(BETAS_B52[2] * (1 - AB_B52[2]) / (1 - AB_B52[1]))})
wb.record("B5.4d", X2_B54, decimals=4, mistakes={"z multiplie l'écart-type σ_t, pas la variance": MEAN_B54 + BTILDE_B54 * Z_B54,
          "vérifie le signe de z : z = −1": MEAN_B54 - math.sqrt(BTILDE_B54) * Z_B54,
          "relis la question : on demande x₂, avec le tirage z": MEAN_B54})
wb.record("B5.4e", X2_BETA_B54, decimals=4, mistakes={"z multiplie l'écart-type, la racine de β_3": MEAN_B54 + float(BETAS_B52[2]) * Z_B54,
          "relis la question : la variance est β_3, pas β̃_3": X2_B54})
wb.record("B5.4f", CLIP_MEAN_B54, decimals=4, mistakes={"tu n'as pas écrêté x̂₀ dans [−1, 1]": NOCLIP_MEAN_B54,
          "relis la question : on demande la moyenne, pas x̂₀ écrêté": 1.0,
          "as-tu arrondi les coefficients du posterior à quatre décimales ? Garde-en cinq : ce chemin tombe à la limite d'un arrondi": 1.2036})
wb.record("B5.4g", LAST_B54, decimals=4, mistakes={"vérifie le signe : on retire le bruit prédit": ddpm_mean_b54(0.5, 0.2, i=0, sign=1.0),
          "relis la question : au pas 1, on prend les constantes de l'indice 0": ddpm_mean_b54(0.5, 0.2, i=1)})
wb.record("B5.4h", False, mistakes={"relis la docstring de ddpm_step (« no noise at all at t = 0 ») et la phrase qui suit l'algorithme 2": True})
wb.record("B5.4i", True, mistakes={"compare tes calculs de d) et e) : quelle partie change ?": False})

**Ex B5.5 — Un pas DDIM à la main et l'image estimée x̂_0**

In [ ]:
wb.record("B5.5a", X0HAT_B55, decimals=4, mistakes={"vérifie le signe : on retire le bruit de x_t": x0hat_b55(eps=-EPS_B55),
          "relis la formule de x̂₀ : des racines de ᾱ_t et de 1 − ᾱ_t": float((X4_B55 - (1 - AB_B52[3]) * EPS_B55) / AB_B52[3]),
          "décalage d'un cran : le pas 4 est l'indice 3": x0hat_b55(i=2)})
wb.record("B5.5b", X1_B55, decimals=4, mistakes={"le pas DDIM réutilise aussi le bruit prédit, multiplié par la racine de 1 − ᾱ_{t'}": ddim_b55(3, 0, noise_term=False)[0],
          "relis la question : on s'arrête au pas 1 (indice 0), pas à l'image propre": X0HAT_B55,
          "relis la question : du pas 4 au pas 1": X2_B55})
wb.record("B5.5c", "B", mistakes={"ta librairie compte les pas en indices : le pas t est l'indice t − 1": "A", "l'indice −1 désigne l'image propre : relis la question, on va au pas 1": "C"})
wb.record("B5.5d", X2_B55, decimals=4, mistakes={"le pas DDIM réutilise aussi le bruit prédit": ddim_b55(3, 1, noise_term=False)[0],
          "décalage d'un cran : le pas 2 est l'indice 1": ddim_b55(3, 2)[0]})
wb.record("B5.5e", True, mistakes={"recalcule x̂₀ à partir de ton x₂ et du même ε_θ : que trouves-tu ?": False})
wb.record("B5.5f", SIGMA2_43_B55, decimals=4, mistakes={"c'est β_4 : relis la formule de σ dans « Le paramètre η »": float(BETAS_B52[3]),
          "on demande σ², pas σ": math.sqrt(SIGMA2_43_B55),
          "relis la formule de σ : le premier rapport est (1 − ᾱ_{t'}) / (1 − ᾱ_t)": float((1 - AB_B52[3]) / (1 - AB_B52[2]) * (1 - AB_B52[3] / AB_B52[2]))})
wb.record("B5.5g", SIGMA_41_B55, decimals=4, mistakes={"on demande σ, pas σ²": SIGMA_41_B55 ** 2,
          "σ tombe près d'une limite d'arrondi : as-tu arrondi σ² à cinq décimales avant la racine ? Garde-en six": math.sqrt(round(SIGMA_41_B55 ** 2, 5)),
          "relis la question : du pas 4 au pas 1, soit de l'indice 3 à l'indice 0": ddim_b55(3, 1, eta=1.0)[1]})
wb.record("B5.5h", DIR_41_B55, decimals=4, mistakes={"n'oublie pas de retirer σ² sous la racine": math.sqrt(1 - AB_B52[0]),
          "prends la racine de 1 − ᾱ₁ − σ²": float(1 - AB_B52[0] - SIGMA_41_B55 ** 2),
          "1 − ᾱ₁ et σ² sont presque égaux : leur différence a perdu des chiffres ; garde six décimales de σ²": math.sqrt(1 - AB_B52[0] - round(SIGMA_41_B55 ** 2, 5)),
          "tu as repris σ arrondi de g) : 1 − ᾱ₁ − σ² est une petite différence, calcule σ² sans l'arrondir": math.sqrt(1 - AB_B52[0] - round(SIGMA_41_B55, 4) ** 2)})
wb.record("B5.5i", False, mistakes={"compare tes réponses g) et h)": True})

## Partie A · Les mathématiques de ta librairie et leurs contrôles

Tu écris dans `mylearn/diffusion.py` tout ce qui, dans un modèle de diffusion, ne dépend pas du réseau : les plannings de bruit et leurs constantes (B5.12), le bruitage en forme fermée (B5.14), le posterior et l'estimation de $\mathbf{x}_0$ (B5.16), puis un pas DDPM et un pas DDIM (B5.17). Entre deux, tu prédis ce qu'un classifieur voit encore d'un chiffre bruité (B5.13) et tu lis les plannings sur une figure (B5.15). Pour finir, tu enchaînes tes pas sur des points 2D avec le vrai bruit, le contrôle qui attrape les erreurs d'indice et de racine avant tout entraînement (B5.18).

**Contrat commun de `diffusion.py`** (les docstrings, plus ces précisions que vérifient les tests) :
- les pas sont des **indices** de 0 à $T - 1$ : le pas $t$ de l'article, celui des formules de la fiche, est l'indice $t - 1$ ; un indice $-1$ ou $T$ lève `ValueError`, jamais `IndexError` (−1 n'est pas « le dernier pas ») ;
- tes fonctions ne modifient jamais leurs entrées (pas de `-=` ni de `*=` sur un tableau reçu) ; les entrées peuvent être des listes ;
- une valeur hors de son intervalle lève `ValueError`, **`nan` compris** : `nan` ne vérifie aucune condition « 0 < β < 1 », il doit donc être refusé ;
- tolérances des tests : $10^{-9}$ en relatif pour les plannings (des float64, par contrat), $10^{-6}$ en relatif et en absolu ailleurs (un calcul en float32 passe) ;
- si la bibliothèque diffusers est installée (elle l'est sur Colab), 25 tests comparent aussi tes fonctions à ses `DDPMScheduler` et `DDIMScheduler` ; sans elle, ils sont ignorés et les formes fermées restent testées.

La cellule suivante charge les outils des parties A et B : `TEST_DIFF` (le fichier des tests), `T_STEPS` (le nombre de pas du notebook : 400 en mode rapide, 1 000 en mode complet), `BETAS` (le planning linéaire du notebook, aux bornes multipliées par $1\,000/T$ : avec les bornes par défaut et $T = 400$, il resterait trop d'image au dernier pas, le piège « SNR final non nul » de la fiche), `DIGITS` (1 000 chiffres de test de MNIST, ramenés dans $[-1, 1]$ par `wbd.to_signed`, toujours les mêmes), `toy_schedule` (les neuf constantes de `noise_schedule`, écrites sans ton code, pour les vérifications) et `show_images`. Le module `wbd` (`wb.diffusion`) contient l'infrastructure du chapitre : données, plannings, jouets 2D et modèles de référence ; le notebook ne s'en sert que pour vérifier ton code ou quand il manque.

In [ ]:
# Tools of parts A and B: your NumPy library diffusion.py, then the 2-D toys
import copy
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from wb import diffusion as wbd
from wb.nbtools import (StopCheck, checked, filled, fr, pair, refuses, returned, run_pytest_group, same, shape_text,
                        verdict, written)

TEST_DIFF = "tests/test_chb5_diffusion.py"          # the tests of your library diffusion.py
T_STEPS = wbd.n_steps(FAST_MODE)                     # 400 steps in fast mode, 1 000 in full mode
BETAS = wbd.schedule_betas("linear", T_STEPS)        # linear, bounds x 1000 / T (computed here, without your code)
DIGITS = wbd.image_subset("mnist", 1000, split="test")   # 1 000 MNIST test digits in [-1, 1], always the same


def fr_e(x):
    """A small number in French scientific notation: 3,6e-05."""
    return f"{x:.1e}".replace(".", ",")


def toy_schedule(betas):
    """The nine constants of noise_schedule (B5.12), written once without your code, for the checks."""
    b = np.asarray(betas, dtype=float)
    a = 1 - b
    ab = np.cumprod(a)
    ab_prev = np.concatenate([[1.0], ab[:-1]])
    return {"betas": b, "alphas": a, "alpha_bars": ab, "alpha_bars_prev": ab_prev, "sqrt_alpha_bars": np.sqrt(ab),
            "sqrt_one_minus_alpha_bars": np.sqrt(1 - ab), "posterior_variance": b * (1 - ab_prev) / (1 - ab),
            "posterior_mean_coef1": b * np.sqrt(ab_prev) / (1 - ab),
            "posterior_mean_coef2": (1 - ab_prev) * np.sqrt(a) / (1 - ab)}


def show_images(images, titles=None, ncol=None, size=1.3, suptitle=None):
    """Images in [-1, 1] (a tensor or an array (N, 1, 28, 28)) in a grid of ncol columns."""
    arr = images.detach().cpu().numpy() if torch.is_tensor(images) else np.asarray(images, dtype=float)
    arr = wbd.to_unit(arr.reshape(-1, 28, 28))
    ncol = ncol or len(arr)
    nrow = math.ceil(len(arr) / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(size * ncol, size * nrow + (0.4 if suptitle else 0.1)), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for i, img in enumerate(arr):
        axes.ravel()[i].imshow(img, cmap="gray", vmin=0, vmax=1)
        if titles is not None:
            axes.ravel()[i].set_title(titles[i], fontsize=8)
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()


print(f"T = {T_STEPS} pas en {'mode rapide' if FAST_MODE else 'mode complet'} ; planning linéaire de bornes "
      f"{fr_e(BETAS[0])} et {fr_e(BETAS[-1])} : ᾱ au dernier indice = {fr_e(wbd.alpha_bars(BETAS)[-1])}. "
      f"{len(DIGITS)} chiffres de test de MNIST dans [−1, 1].")

### Ex B5.12 — linear_beta_schedule, cosine_beta_schedule et noise_schedule 🔨 ★★ ⏱️ 20 min
**Objectif :** écrire les deux plannings de bruit et toutes les constantes qui en dérivent.  
**Prérequis :** ✏️ B5.2 · fiche §205.2 (« Le planning linéaire de Ho et coll. », « Le planning cosinus de Nichol et Dhariwal », « Bruiter un batch : `noise_schedule` et `q_sample` ») · **mylearn :** `diffusion.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/diffusion.py` (créé par `python tools/start_chapter.py B5`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb5_diffusion.py -q` les lance tous dans un terminal.

**Contrat**
- `linear_beta_schedule(n_steps, beta_start=1e-4, beta_end=0.02)` : un tableau float64 de forme `(T,)`, de `beta_start` (indice 0) à `beta_end` (indice $T - 1$), régulièrement espacés ; pour `n_steps = 1`, `[beta_start]`, comme `np.linspace` (aucune division par $T - 1$) ; `ValueError` si `n_steps < 1` ou si $0 < \text{beta\_start} \le \text{beta\_end} < 1$ n'est pas vérifié (`nan` compris).
- `cosine_beta_schedule(n_steps, s=0.008, max_beta=0.999)` : la fonction $f$ de la docstring évaluée aux $T + 1$ points $t/T$, $t = 0 \dots T$ (ni $T$ points, ni des points de $[0, 1)$), puis $\beta_t = \min(1 - f(t+1)/f(t),\ \text{max\_beta})$ pour $t = 0 \dots T - 1$, en float64 ; l'écrêtage à `max_beta` est atteint au dernier indice ; `ValueError` si `n_steps < 1`, `s < 0` ou `max_beta` hors de $(0, 1)$ (`nan` compris).
- `noise_schedule(betas)` : un dictionnaire aux neuf clés de la docstring, chacune un tableau float64 de forme `(T,)`, **même pour des betas en float32** ; `alpha_bars_prev[0]` vaut 1, donc `posterior_variance[0]` vaut 0 **exactement** ; `ValueError` si `betas` n'est pas 1-D ou si un $\beta$ est hors de $(0, 1)$, `nan` compris.

La première cellule de vérification essaie tes trois fonctions sur les exemples des docstrings, contre `np.linspace` et la forme fermée du cosinus, sur le cas $T = 1$, l'écrêtage et six refus, puis affiche $\bar\alpha$ au dernier indice du planning linéaire de $T$ = `T_STEPS` pas, avec les bornes par défaut puis multipliées par $1\,000/T$ (rien à taper). La seconde lance les 60 tests de l'exercice (`-k "test_linear_beta_schedule_ or test_cosine_beta_schedule_ or test_noise_schedule_"`).

Dans tes notes : pourquoi `posterior_variance[0]` vaut-il 0, et qu'en conclure pour le dernier pas de l'échantillonnage ? Avec $T = 400$, que vaut $\bar\alpha$ au dernier indice avec les bornes par défaut, quelle part de l'image reste-t-il (son coefficient $\sqrt{\bar\alpha}$), et pourquoi multiplier les bornes par $1\,000/T$ règle-t-il le problème (✏️ B5.2) ? Pourquoi un test « aucun $\beta$ n'est $\le 0$ ni $\ge 1$ » laisse-t-il passer un `nan` ?

In [ ]:
with checked("B5.12"):
    M = mylearn.diffusion
    lin_12 = M.linear_beta_schedule(5)
    if returned("B5.12", "linear_beta_schedule", lin_12):
        verdict("B5.12", np.asarray(lin_12).dtype == np.float64, "linear_beta_schedule rend des float64.",
                f"linear_beta_schedule doit rendre des float64, reçu {np.asarray(lin_12).dtype}.")
        same("B5.12", "linear_beta_schedule(5) (exemple de la docstring)", lin_12, np.linspace(1e-4, 0.02, 5))
    same("B5.12", f"linear_beta_schedule({T_STEPS}, …) aux bornes × 1000/T, contre np.linspace",
         M.linear_beta_schedule(T_STEPS, 1e-4 * 1000 / T_STEPS, 0.02 * 1000 / T_STEPS), BETAS)
    same("B5.12", "linear_beta_schedule(1, 0.3, 0.5) : un seul pas, beta_start comme np.linspace",
         M.linear_beta_schedule(1, 0.3, 0.5), [0.3])
    refuses("B5.12", "linear_beta_schedule(10, beta_start=nan)", lambda: M.linear_beta_schedule(10, float("nan")))
    refuses("B5.12", "linear_beta_schedule(10, 0.02, 0.01) (beta_start > beta_end)",
            lambda: M.linear_beta_schedule(10, 0.02, 0.01))
    refuses("B5.12", "linear_beta_schedule(0)", lambda: M.linear_beta_schedule(0))

    f_12 = np.cos((np.arange(6) / 5 + 0.008) / 1.008 * np.pi / 2) ** 2          # f at the T + 1 = 6 points t / T
    same("B5.12", "cosine_beta_schedule(5) contre la forme fermée", M.cosine_beta_schedule(5),
         np.minimum(1 - f_12[1:] / f_12[:-1], 0.999))
    same("B5.12", "cosine_beta_schedule(1000) contre la forme fermée", M.cosine_beta_schedule(1000),
         wbd.schedule_betas("cosine", 1000))
    same("B5.12", "cosine_beta_schedule(8, max_beta=0.3) : l'écrêtage atteint au dernier indice",
         np.asarray(M.cosine_beta_schedule(8, max_beta=0.3))[-1], 0.3)
    refuses("B5.12", "cosine_beta_schedule(10, max_beta=1.0)", lambda: M.cosine_beta_schedule(10, max_beta=1.0))
    refuses("B5.12", "cosine_beta_schedule(10, s=-0.1)", lambda: M.cosine_beta_schedule(10, s=-0.1))

    betas_12 = np.array([0.1, 0.2, 0.5])
    sched_12 = M.noise_schedule(betas_12)
    if returned("B5.12", "noise_schedule", sched_12):
        want_12 = toy_schedule(betas_12)
        keys_12 = set(sched_12) if isinstance(sched_12, dict) else set()
        verdict("B5.12", keys_12 == set(want_12), "noise_schedule rend les neuf clés de la docstring.",
                f"noise_schedule doit rendre un dictionnaire aux clés {sorted(want_12)} ; manquent "
                f"{sorted(set(want_12) - keys_12)}, en trop {sorted(keys_12 - set(want_12))}.")
        for name_12 in sorted(set(want_12) & keys_12):
            same("B5.12", f"noise_schedule([0.1, 0.2, 0.5])[{name_12!r}]", sched_12[name_12], want_12[name_12])
        if "posterior_variance" in keys_12:
            verdict("B5.12", np.asarray(sched_12["posterior_variance"])[0] == 0,
                    "posterior_variance[0] vaut 0 exactement.",
                    "posterior_variance[0] doit valoir 0 exactement (alpha_bars_prev[0] = 1).")
    sched32_12 = M.noise_schedule(np.array([0.1, 0.2], dtype=np.float32))
    verdict("B5.12", all(np.asarray(v).dtype == np.float64 for v in sched32_12.values()),
            "des betas en float32 donnent des constantes en float64.",
            "noise_schedule doit rendre des float64, même pour des betas en float32.")
    verdict("B5.12", np.array_equal(betas_12, [0.1, 0.2, 0.5]), "les betas reçus n'ont pas changé.",
            "noise_schedule a modifié les betas reçus.")
    refuses("B5.12", "noise_schedule([0.1, nan])", lambda: M.noise_schedule([0.1, float("nan")]))
    refuses("B5.12", "noise_schedule([0.1, 1.0])", lambda: M.noise_schedule([0.1, 1.0]))
    refuses("B5.12", "noise_schedule([[0.1, 0.2]]) (2-D)", lambda: M.noise_schedule([[0.1, 0.2]]))

    end_default_12 = float(M.noise_schedule(M.linear_beta_schedule(T_STEPS))["alpha_bars"][-1])
    end_scaled_12 = float(M.noise_schedule(BETAS)["alpha_bars"][-1])
    print(f"ᾱ au dernier indice, T = {T_STEPS} : bornes par défaut {fr_e(end_default_12)}, bornes × 1000/T "
          f"{fr_e(end_scaled_12)} (avec ta librairie).")

In [ ]:
if written(lambda: mylearn.diffusion.linear_beta_schedule(2)):
    run_pytest_group(TEST_DIFF, impl="ref", keyword="test_linear_beta_schedule_ or test_cosine_beta_schedule_ or test_noise_schedule_")   # 60 tests
else:
    print("⏳ Ex B5.12 : les tests attendent tes fonctions.")

### Ex B5.13 — Prédire l'allure d'un chiffre bruité aux pas 100, 300 et 600, planning linéaire puis cosinus 🔮 ★ ⏱️ 10 min
**Objectif :** prédire jusqu'à quel niveau de bruit un classifieur entraîné sur des chiffres propres reconnaît encore un chiffre bruité, et comparer à la convention de lisibilité de ✏️ B5.3.  
**Prérequis :** Ex B5.12 · ✏️ B5.3 · fiche §205.2 (« Le rapport signal sur bruit ») · ch. 27 (le classifieur de la partie B) · **Fil rouge :** MNIST · **Parcours :** R, C

En ✏️ B5.3, tu as adopté une **convention** : un chiffre reste lisible tant que son rapport signal sur bruit vaut au moins 0,1. Qu'en pense un vrai lecteur ? La cellule suivante prépare `CLF`, le petit CNN de la partie B du ch. 27 (`wbd.classifier`), entraîné une fois sur 10 000 chiffres **propres** de MNIST ; il en reconnaît environ 96 % sur les 1 000 chiffres de test. Elle donne aussi `noisy_13(planning, pas)`, qui bruite ces 1 000 chiffres par la forme fermée, au **pas de l'article** 100, 300 ou 600 (les indices 99, 299 et 599), du planning linéaire ou cosinus de $T = 1\,000$, toujours avec le même bruit.

L'expérience montre quatre chiffres à ces trois pas pour chaque planning, puis, pour chacun des six cas, le SNR et l'**accuracy** du classifieur sur les chiffres bruités. Tu peux calculer les six SNR (✏️ B5.3), pas l'accuracy. **Avant d'exécuter**, écris tes deux prédictions :
- a) `strong_13` : dans combien des six cas (3 pas × 2 plannings) le classifieur reconnaît-il **au moins 70 %** des chiffres ? Un entier de 0 à 6 ;
- b) `cos600_13` : au pas 600 du planning cosinus, où le SNR dépasse largement le seuil 0,1 de ta convention, le classifieur reconnaît-il **plus de la moitié** des chiffres ? `True` ou `False`.

La cellule d'expérience n'affiche rien tant que tes deux prédictions ne sont pas écrites ; elle les vérifie à la fin. Rien n'y dépend de ton code.

In [ ]:
CLF = wbd.classifier("mnist")                      # the CNN of ch. 27, trained once on clean digits, then reloaded
STEPS_13 = (100, 300, 600)                          # steps of the paper: indices 99, 299, 599
AB_13 = {"linéaire": wbd.alpha_bars(wbd.schedule_betas("linear", 1000)),
         "cosinus": wbd.alpha_bars(wbd.schedule_betas("cosine", 1000))}
EPS_13 = np.random.default_rng(13).standard_normal(tuple(DIGITS.images.shape))   # one noise per digit, for every case
X0_13 = DIGITS.images.double().numpy()


def noisy_13(kind, step):
    """The 1 000 test digits at the step `step` of the paper (index step - 1) of the schedule `kind` (T = 1 000),
    all with the same noise EPS_13 (the closed form, written here without your q_sample)."""
    a = AB_13[kind][step - 1]
    return math.sqrt(a) * X0_13 + math.sqrt(1 - a) * EPS_13


print(f"Le classifieur reconnaît {fr(100 * np.mean(wbd.predict(CLF, X0_13) == DIGITS.labels.numpy()), 1)} % des "
      "1 000 chiffres propres.")

In [ ]:
strong_13, cos600_13 = 3, False

In [ ]:
wb.record("B5.13a", 3, mistakes={"relis le tableau : au pas 100, le classifieur reconnaît-il encore les chiffres ?": 0, "relis le tableau : compte le pas 100 des deux plannings": 1, "relis le tableau : le cosinus au pas 300 atteint-il 70 % ?": 2, "relis le tableau : compte un à un les cas à 70 % ou plus": 4, "relis le tableau : au pas 600, le cosinus atteint-il 70 % ?": 5, "relis le tableau : que vaut l'accuracy au pas 600 du linéaire ?": 6}, fractional="on demande un nombre de cas, un entier de 0 à 6", choices=[0, 1, 2, 3, 4, 5, 6], prediction=True)
wb.record("B5.13b", False, mistakes={"relis le tableau : au pas 600 du cosinus, l'accuracy dépasse-t-elle 50 % ?": True}, prediction=True)

In [ ]:
if any(answer is ... or answer is None for answer in [strong_13, cos600_13]):
    print("⏳ Ex B5.13 : écris d'abord tes deux prédictions dans la cellule précédente.")
else:
    # 1. four digits, clean then at the steps 100, 300 and 600 of each schedule (the same noise everywhere)
    for kind_13 in AB_13:
        row_13 = [X0_13[:4]] + [noisy_13(kind_13, s)[:4] for s in STEPS_13]
        show_images(np.stack(row_13, axis=1).reshape(-1, 1, 28, 28), ncol=16,
                    titles=[f"{s}" for _ in range(4) for s in ("propre",) + STEPS_13], size=0.9,
                    suptitle=f"planning {kind_13}, T = 1 000 : chaque chiffre propre, puis aux pas 100, 300, 600")

    # 2. the accuracy of the classifier, and the SNR, in each of the 6 cases
    acc_13 = {}
    for kind_13 in AB_13:
        for step_13 in STEPS_13:
            a_13 = AB_13[kind_13][step_13 - 1]
            pred_13 = wbd.predict(CLF, noisy_13(kind_13, step_13))
            acc_13[kind_13, step_13] = float(np.mean(pred_13 == DIGITS.labels.numpy()))
            top_13 = np.bincount(pred_13, minlength=10)
            print(f"{kind_13:>9} pas {step_13} : SNR {fr(a_13 / (1 - a_13), 3):>7} · accuracy {fr(100 * acc_13[kind_13, step_13], 1):>5} % "
                  f"· classe la plus prédite : {top_13.argmax()} ({fr(100 * top_13.max() / len(pred_13), 0)} % des chiffres)")
    wb.check("B5.13a", strong_13)
    wb.check("B5.13b", cos600_13)

Dans tes notes : pour chaque prédiction ratée, qu'est-ce qui t'a trompé ? À partir de quel SNR, à peu près, le classifieur décroche-t-il, et ton œil, sur les images ? Quelle classe le classifieur prédit-il quand il ne sait plus, et pourquoi un réseau entraîné sur des images propres se trompe-t-il ainsi sur du bruit qu'il n'a jamais vu ? Qu'est-ce que cela dit du débruiteur d'un modèle de diffusion, qui, lui, voit des images de tous les niveaux de bruit pendant son entraînement ?

### Ex B5.14 — q_sample : bruiter un batch d'images à des pas différents 🔨 ★★ ⏱️ 20 min
**Objectif :** écrire le bruitage en forme fermée d'un batch où chaque exemple a son propre pas.  
**Prérequis :** Ex B5.12 · ∂ B5.6 · fiche §205.2 (« La forme fermée », « Bruiter un batch » et le piège « `alpha_bars[t]` sans remise en forme ») · **Fil rouge :** MNIST · **mylearn :** `diffusion.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/diffusion.py` (créé par `python tools/start_chapter.py B5`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb5_diffusion.py -q` les lance tous dans un terminal.

**Contrat** de `q_sample(x0, t, alpha_bars, noise=None, rng=None)`, qui rend le couple `(x_t, noise)` :
- $\mathbf{x}_t = \sqrt{\bar\alpha_{t}}\,\mathbf{x}_0 + \sqrt{1 - \bar\alpha_{t}}\,\boldsymbol{\varepsilon}$ pour chaque exemple, avec **son** indice `t[n]` ; le coefficient est remis en forme `(N, 1, …, 1)`, quel que soit le nombre d'axes de `x0` (1-D, 2-D, `(N, C, H, W)`) ;
- `t` a **exactement un indice entier par exemple**, de forme `(N,)` : un `t` de longueur 1 pour un batch de plusieurs exemples lève `ValueError`, **même si NumPy saurait le diffuser** ; un indice hors de $[0, T)$ aussi ;
- `noise`, s'il est donné, a **exactement** la forme de `x0` (un bruit de forme `(2,)` pour `x0` de forme `(3, 2)` est refusé, même s'il se diffuse) ; sinon, **un seul** appel `rng.standard_normal(x0.shape)`, et c'est ce bruit qui est rendu (la cible de l'entraînement) ; `rng=None` : `np.random.default_rng()`, sans graine.

La première cellule de vérification essaie ta fonction sur l'exemple de la docstring, sur 6 chiffres de MNIST `(6, 1, 28, 28)` à 6 indices différents, sur un `x0` en 1-D, sur le bruit tiré par `rng`, et quatre refus ; elle montre les 6 chiffres bruités. La seconde lance les 19 tests de l'exercice (`-k "test_q_sample_"`).

Dans tes notes : que se passe-t-il, sans erreur, si tu multiplies `x0` de forme `(3, 1, 4, 3)` par `np.sqrt(alpha_bars[t])` de forme `(3,)` sans remise en forme (fiche, piège « `alpha_bars[t]` sans remise en forme ») ? Pourquoi rendre le bruit tiré, et pas seulement $\mathbf{x}_t$ ? Pourquoi refuser un `t` de longueur 1 que NumPy diffuserait volontiers ?

In [ ]:
xt_14 = None
with checked("B5.14"):
    M = mylearn.diffusion
    ab_14 = np.array([0.9, 0.72, 0.36])
    x0_14 = np.array([[1.0, -1.0], [0.5, 0.0]])
    out_14, noise_14 = pair("B5.14", "q_sample", M.q_sample(x0_14.copy(), [0, 2], ab_14, noise=np.ones((2, 2))),
                            names=("x_t", "noise"))
    same("B5.14", "q_sample de l'exemple de la docstring", out_14,
         np.sqrt(ab_14[[0, 2]])[:, None] * x0_14 + np.sqrt(1 - ab_14[[0, 2]])[:, None], rtol=1e-6, atol=1e-6)

    ab_nb = wbd.alpha_bars(BETAS)
    imgs_14 = DIGITS.images[:6].double().numpy()
    t_14 = np.array([0, 1, 50, 150, 300, T_STEPS - 1])
    eps_14 = np.random.default_rng(14).standard_normal(imgs_14.shape)
    copies_14 = (imgs_14.copy(), t_14.copy(), eps_14.copy(), ab_nb.copy())
    xt_14, n_14 = pair("B5.14", "q_sample", M.q_sample(imgs_14, t_14, ab_nb, noise=eps_14), names=("x_t", "noise"))
    want_14 = np.sqrt(ab_nb[t_14])[:, None, None, None] * imgs_14 + np.sqrt(1 - ab_nb[t_14])[:, None, None, None] * eps_14
    if not same("B5.14", "6 chiffres (6, 1, 28, 28), un pas par image", xt_14, want_14, rtol=1e-6, atol=1e-6):
        xt_14 = None
    same("B5.14", "le bruit rendu est celui reçu", n_14, eps_14, rtol=1e-6, atol=1e-6)
    verdict("B5.14", all(np.array_equal(a, b) for a, b in zip((imgs_14, t_14, eps_14, ab_nb), copies_14)),
            "les entrées n'ont pas changé.", "q_sample a modifié une de ses entrées (x0, t, noise ou alpha_bars).")
    same("B5.14", "x0 en 1-D (3 exemples d'une valeur)", M.q_sample(np.array([0.5, -0.5, 1.0]), [0, 1, 2], ab_14,
                                                                    noise=np.array([1.0, 2.0, 3.0]))[0],
         np.sqrt(ab_14) * [0.5, -0.5, 1.0] + np.sqrt(1 - ab_14) * [1.0, 2.0, 3.0], rtol=1e-6, atol=1e-6)
    x_r, n_r = pair("B5.14", "q_sample", M.q_sample(x0_14, [1, 1], ab_14, rng=np.random.default_rng(5)),
                    names=("x_t", "noise"))
    same("B5.14", "sans noise : le bruit de rng.standard_normal(x0.shape), rendu", n_r,
         np.random.default_rng(5).standard_normal((2, 2)), rtol=1e-6, atol=1e-6)
    refuses("B5.14", "t de longueur 1 pour 2 exemples", lambda: M.q_sample(x0_14, [1], ab_14, noise=np.ones((2, 2))))
    refuses("B5.14", "un bruit de forme (2,) pour x0 de forme (2, 2)",
            lambda: M.q_sample(x0_14, [0, 1], ab_14, noise=np.ones(2)))
    refuses("B5.14", "t = -1", lambda: M.q_sample(x0_14, [-1, 0], ab_14, noise=np.ones((2, 2))))
    refuses("B5.14", "t = T", lambda: M.q_sample(x0_14, [0, 3], ab_14, noise=np.ones((2, 2))))
if xt_14 is not None:
    show_images(xt_14, titles=[f"indice {t}" for t in t_14], size=1.5,
                suptitle=f"ta q_sample : 6 chiffres, un pas chacun (T = {T_STEPS})")

In [ ]:
if written(lambda: mylearn.diffusion.q_sample([0.0], [0], [0.5], noise=[0.0])):
    run_pytest_group(TEST_DIFF, impl="ref", keyword="test_q_sample_")   # 19 tests
else:
    print("⏳ Ex B5.14 : les tests attendent ta fonction q_sample.")

### Ex B5.15 — Lire ᾱ_t, le rapport signal sur bruit et la grille de bruitage des deux plannings 📈 ★ ⏱️ 15 min
**Objectif :** lire sur une figure ce qui distingue les plannings, et voir un même chiffre se dissoudre dans le bruit.  
**Prérequis :** Ex B5.14 · ✏️ B5.3 · fiche §205.2 (« Le planning linéaire… », « Le planning cosinus… », « Le rapport signal sur bruit » et le piège « un SNR final qui n'est pas nul ») · **Fil rouge :** MNIST · **Parcours :** C

La figure trace $\bar\alpha$ (à gauche) et $\ln \operatorname{SNR}$ (à droite) de quatre plannings, en fonction de la **fraction de la chaîne** parcourue, (indice + 1)/$T$, pour les comparer malgré leurs $T$ différents : **L**, le linéaire de Ho et coll. ($T = 1\,000$) ; **C**, le cosinus ($T = 1\,000$) ; **D**, le linéaire raccourci à $T = 400$ avec les bornes par défaut ; **S**, le linéaire à $T = 400$ avec les bornes multipliées par $1\,000/T$, celui du notebook en mode rapide. Dessous, le même chiffre et le même bruit aux indices 0, 99, 199, …, 999 de L, puis de C. Rien ne dépend de ton code. Réponds en lisant la figure :
- a) `same_curve_15` : laquelle des deux courbes à $T = 400$ suit celle de L : `"D"` ou `"S"` ?
- b) `largest_end_15` : quel planning garde **le plus** de l'image à son dernier indice : `"L"`, `"C"`, `"D"` ou `"S"` ?
- c) `cos_above_15` : sur quelle part de la chaîne le $\ln \operatorname{SNR}$ de C est-il au-dessus de celui de L ? `"A"` moins de la moitié ; `"B"` entre la moitié et 90 % ; `"C"` plus de 95 %.

Dans tes notes : avec L, à partir de quelle colonne de la grille ne distingues-tu plus le chiffre, et avec C ? Que gagne le planning cosinus, en nombre de pas « utiles » (où il reste quelque chose à débruiter), d'après la figure et ✏️ B5.3 ? Pourquoi D est-il un piège, et que verrait un modèle entraîné avec lui, puis échantillonné à partir d'un bruit pur ?

In [ ]:
# The four schedules of the figure, against the fraction t / T of the chain (computed without your code)
SCHEDULES_15 = {"L": ("linéaire, T = 1 000", wbd.schedule_betas("linear", 1000)),
                "C": ("cosinus, T = 1 000", wbd.schedule_betas("cosine", 1000)),
                "D": ("linéaire, T = 400, bornes par défaut", np.linspace(1e-4, 0.02, 400)),
                "S": ("linéaire, T = 400, bornes × 1000/T", wbd.schedule_betas("linear", 400))}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for (letter, (label, betas)), style in zip(SCHEDULES_15.items(), ("-", "-", "--", ":")):
    ab = wbd.alpha_bars(betas)
    frac = np.arange(1, len(ab) + 1) / len(ab)
    ax1.plot(frac, ab, style, lw=2, label=f"{letter} : {label}")
    ax2.plot(frac, np.log(ab / (1 - ab)), style, lw=2, label=letter)
ax1.set_xlabel("fraction de la chaîne, (indice + 1) / T")
ax1.set_ylabel("ᾱ")
ax1.legend(fontsize=8)
ax2.set_xlabel("fraction de la chaîne, (indice + 1) / T")
ax2.set_ylabel("ln SNR")
ax2.legend()
plt.show()

# One digit at the indices 0, 99, ..., 999 of the two schedules of T = 1 000, with the same noise
idx_15 = [0] + list(range(99, 1000, 100))
x0_15 = DIGITS.images[0].double().numpy()
eps_15 = np.random.default_rng(15).standard_normal(x0_15.shape)
grid_15 = [math.sqrt(a) * x0_15 + math.sqrt(1 - a) * eps_15
           for key in ("L", "C") for a in wbd.alpha_bars(SCHEDULES_15[key][1])[idx_15]]
show_images(np.stack(grid_15), titles=[f"{k} {i}" for k in ("L", "C") for i in idx_15], ncol=len(idx_15), size=1.35,
            suptitle="le même chiffre et le même bruit, aux indices 0 à 999 : linéaire (L), puis cosinus (C)")

In [ ]:
same_curve_15, largest_end_15, cos_above_15 = "S", "D", "C"

with wb.attempt("B5.15"):
    wb.check("B5.15a", same_curve_15)
with wb.attempt("B5.15"):
    wb.check("B5.15b", largest_end_15)
with wb.attempt("B5.15"):
    wb.check("B5.15c", cos_above_15)

In [ ]:
wb.record("B5.15a", "S", mistakes={"compare les deux courbes à T = 400 à la courbe L du graphique de gauche : laquelle se superpose ?": "D"})
wb.record("B5.15b", "D", mistakes={"lis la fin des courbes du graphique de droite : la courbe L finit-elle le plus haut ?": "L", "lis la fin des courbes du graphique de droite : la courbe C finit-elle le plus haut ?": "C", "lis la fin des courbes du graphique de droite : la courbe S finit-elle le plus haut ?": "S"})
wb.record("B5.15c", "C", mistakes={"suis les courbes C et L du graphique de droite : où se croisent-elles ?": "A", "suis les courbes C et L du graphique de droite jusqu'au dernier indice : où se croisent-elles ?": "B"})

### Ex B5.16 — q_posterior_mean_variance et predict_x0_from_eps 🔨 ★★ ⏱️ 30 min
**Objectif :** écrire la moyenne et la variance du posterior d'un pas, et l'estimation de $\mathbf{x}_0$ à partir d'un bruit.  
**Prérequis :** Ex B5.14 · ∂ B5.7 · fiche §205.3 (« Le posterior : un pas en arrière quand on connaît $\mathbf{x}_0$ », « Prédire le bruit ») · **mylearn :** `diffusion.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/diffusion.py` (créé par `python tools/start_chapter.py B5`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb5_diffusion.py -q` les lance tous dans un terminal.

**Contrat**
- `q_posterior_mean_variance(x0, x_t, t, schedule)` rend le couple `(mean, variance)` :
  - `schedule` est le dictionnaire de `noise_schedule` (B5.12) ; tu y lis `posterior_mean_coef1`, `posterior_mean_coef2` et `posterior_variance` à l'indice de chaque exemple ;
  - `mean` a la forme de `x0` ; `variance` a la forme `(N, 1, …, 1)`, avec **autant d'axes que `x0`** (`(N,)` si `x0` est 1-D), et vaut 0 à l'indice 0 ;
  - mêmes règles que `q_sample` pour `t` (un indice entier par exemple, dans $[0, T)$) ; `x_t` a **exactement** la forme de `x0`, sans diffusion.
- `predict_x0_from_eps(x_t, t, eps, alpha_bars)` : $\hat{\mathbf{x}}_0 = \big(\mathbf{x}_t - \sqrt{1 - \bar\alpha_t}\,\boldsymbol{\varepsilon}\big) / \sqrt{\bar\alpha_t}$, de la forme de `x_t` ; `eps` a exactement la forme de `x_t` ; mêmes règles pour `t`. L'aller-retour `predict_x0_from_eps(q_sample(x0, t, ab, noise)[0], t, noise, ab)` redonne `x0`.

La première cellule de vérification essaie tes deux fonctions sur les exemples des docstrings, les formes de la variance en 4-D et en 1-D, un **Bayes numérique** en dimension 1 (le produit des deux densités gaussiennes de ∂ B5.7, normalisé sur une grille de 600 001 points, dont on lit la moyenne et la variance), l'aller-retour avec le vrai bruit et quatre refus. La seconde lance les 29 tests de l'exercice (`-k "test_q_posterior_mean_variance_ or test_predict_x0_from_eps_"`).

Dans tes notes : à l'indice 0, que valent la moyenne et la variance du posterior, et pourquoi ? À combien près la moyenne du Bayes numérique et ta formule s'accordent-elles (la ligne sous les vérifications), et pourquoi une grille suffit-elle ici ? Que devient l'erreur de $\hat{\mathbf{x}}_0$ quand le bruit prédit se trompe de 0,1, au pas 10 puis au dernier pas du planning du notebook ?

In [ ]:
with checked("B5.16"):
    M = mylearn.diffusion
    s_16 = toy_schedule([0.1, 0.2, 0.5])
    x0_16 = np.array([[1.0, -1.0], [0.5, 0.0]])
    xt_16 = np.array([[1.2649, -0.6325], [1.1, 0.8]])
    copies_16 = (x0_16.copy(), xt_16.copy())
    mean_16, var_16 = pair("B5.16", "q_posterior_mean_variance",
                           M.q_posterior_mean_variance(x0_16, xt_16, [0, 2], s_16), names=("mean", "variance"))
    t_16 = np.array([0, 2])
    same("B5.16", "moyenne du posterior (exemple de la docstring)", mean_16,
         s_16["posterior_mean_coef1"][t_16][:, None] * x0_16 + s_16["posterior_mean_coef2"][t_16][:, None] * xt_16,
         rtol=1e-6, atol=1e-6)
    same("B5.16", "variance du posterior, forme (2, 1)", var_16, s_16["posterior_variance"][t_16][:, None],
         rtol=1e-6, atol=1e-6)
    verdict("B5.16", np.asarray(var_16).ravel()[0] == 0, "à l'indice 0, la variance vaut 0 exactement.",
            "à l'indice 0, la variance du posterior doit valoir 0 exactement.")
    verdict("B5.16", all(np.array_equal(a, b) for a, b in zip((x0_16, xt_16), copies_16)),
            "les entrées n'ont pas changé.", "q_posterior_mean_variance a modifié x0 ou x_t.")
    _, var4_16 = M.q_posterior_mean_variance(np.zeros((3, 1, 2, 2)), np.ones((3, 1, 2, 2)), [0, 1, 2], s_16)
    verdict("B5.16", np.shape(var4_16) == (3, 1, 1, 1), "variance de forme (3, 1, 1, 1) pour x0 de forme (3, 1, 2, 2).",
            f"pour x0 de forme (3, 1, 2, 2), la variance doit avoir la forme (3, 1, 1, 1), reçu {np.shape(var4_16)}.")
    _, var1_16 = M.q_posterior_mean_variance(np.zeros(3), np.ones(3), [0, 1, 2], s_16)
    verdict("B5.16", np.shape(var1_16) == (3,), "variance de forme (3,) pour x0 en 1-D.",
            f"pour x0 de forme (3,), la variance doit avoir la forme (3,), reçu {np.shape(var1_16)}.")

    # Bayes on a fine grid, in 1-D: prior q(x_{t-1} | x0) times the likelihood q(x_t | x_{t-1}), at index 2
    x0_b, xt_b, t_b = -0.4, 0.7, 2
    grid = np.linspace(-15, 15, 600_001)
    ab, a = s_16["alpha_bars"], s_16["alphas"]
    log_post = (-(grid - math.sqrt(ab[t_b - 1]) * x0_b) ** 2 / (2 * (1 - ab[t_b - 1]))
                - (xt_b - math.sqrt(a[t_b]) * grid) ** 2 / (2 * (1 - a[t_b])))
    w_b = np.exp(log_post - log_post.max())
    m_b = float((grid * w_b).sum() / w_b.sum())
    mean_b, var_b = M.q_posterior_mean_variance(np.array([x0_b]), np.array([xt_b]), [t_b], s_16)
    same("B5.16", "moyenne contre Bayes sur une grille (x0 = −0,4, x_t = 0,7, indice 2)", mean_b, [m_b], rtol=1e-6, atol=1e-6)
    same("B5.16", "variance contre Bayes sur une grille", var_b, [float(((grid - m_b) ** 2 * w_b).sum() / w_b.sum())],
         rtol=1e-6, atol=1e-6)
    try:
        mine_b = float(np.ravel(np.asarray(mean_b, dtype=float))[0])
        print(f"   Bayes sur la grille : moyenne {m_b:.10f} ; ta formule : {mine_b:.10f} ; écart {fr_e(abs(m_b - mine_b))}")
    except (TypeError, ValueError, IndexError):
        pass
    refuses("B5.16", "x_t de forme (2, 3) pour x0 de forme (2, 2)",
            lambda: M.q_posterior_mean_variance(x0_16, np.ones((2, 3)), [0, 1], s_16))
    refuses("B5.16", "t = 3 avec T = 3", lambda: M.q_posterior_mean_variance(x0_16, xt_16, [0, 3], s_16))

    eps_16 = np.array([[0.3, -2.0], [1.0, 0.5]])
    ab_16 = s_16["alpha_bars"]
    noisy_16 = np.sqrt(ab_16[[1, 2]])[:, None] * x0_16 + np.sqrt(1 - ab_16[[1, 2]])[:, None] * eps_16
    same("B5.16", "predict_x0_from_eps : l'aller-retour avec le vrai bruit redonne x0",
         M.predict_x0_from_eps(noisy_16, [1, 2], eps_16, ab_16), x0_16, rtol=1e-6, atol=1e-6)
    refuses("B5.16", "predict_x0_from_eps avec eps de forme (2,)",
            lambda: M.predict_x0_from_eps(noisy_16, [1, 2], np.ones(2), ab_16))
    refuses("B5.16", "predict_x0_from_eps avec t = -1", lambda: M.predict_x0_from_eps(noisy_16, [-1, 2], eps_16, ab_16))

In [ ]:
if written(lambda: mylearn.diffusion.predict_x0_from_eps([0.0], [0], [0.0], [0.5])):
    run_pytest_group(TEST_DIFF, impl="ref", keyword="test_q_posterior_mean_variance_ or test_predict_x0_from_eps_")   # 29 tests
else:
    print("⏳ Ex B5.16 : les tests attendent tes fonctions.")

### Ex B5.17 — ddpm_step et ddim_step 🔨 ★★★ ⏱️ 40 min
**Objectif :** écrire un pas d'échantillonnage DDPM et un pas DDIM, avec leurs cas particuliers.  
**Prérequis :** Ex B5.16 · ✏️ B5.4, B5.5 · fiche §205.5 (« L'algorithme », « Un pas à la main », le piège « l'échelle et le dernier pas »), §205.6 (« L'idée de DDIM », « Le paramètre η ») · **mylearn :** `diffusion.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/diffusion.py` (créé par `python tools/start_chapter.py B5`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb5_diffusion.py -q` les lance tous dans un terminal.

**Contrat**
- `ddpm_step(x_t, t, eps_pred, schedule, rng=None, variance="posterior", clip_x0=None)` rend $\mathbf{x}_{t-1}$, de la forme de `x_t` :
  - `t` est un **entier Python** de $[0, T)$, le même pour tout le batch ; −1 est refusé ;
  - `variance` vaut exactement `"posterior"` ($\tilde\beta_t$) ou `"beta"` ($\beta_t$), et se vérifie **même à $t = 0$** : la clause *Raises* de la docstring est inconditionnelle ;
  - sans `clip_x0`, la moyenne de l'algorithme 2 de Ho et coll. ; avec `clip_x0`, $\hat{\mathbf{x}}_0$ écrêté à `[-clip_x0, clip_x0]` (la borne reçue, pas 1), puis la moyenne du posterior ;
  - à $t > 0$, on ajoute `sqrt(var) * z` : **l'écart-type**, pas la variance ; $\mathbf{z}$ est tiré de `rng`, de la forme de `x_t`, sans dépendre des valeurs de `x_t` ni de `eps_pred` (deux appels avec la même graine tirent le même $\mathbf{z}$ : c'est ainsi que les tests isolent la moyenne) ; **aucun bruit à $t = 0$**.
- `ddim_step(x_t, t, t_prev, eps_pred, alpha_bars, eta=0.0, rng=None)` rend $\mathbf{x}_{t_{\text{prev}}}$ :
  - `t_prev = -1` veut dire « l'image propre » : $\bar\alpha_{\text{prev}} = 1$ ; **ne lis pas `alpha_bars[-1]`** ;
  - `ValueError` si $-1 \le t_{\text{prev}} < t < T$ n'est pas vérifié (`t_prev >= t` refusé), ou si `eta` est hors de $[0, 1]$, `nan` compris ;
  - avec `eta = 0`, **aucun nombre n'est tiré de `rng`** (les tests vérifient l'état du générateur).

La première cellule de vérification essaie `ddpm_step` à $t = 0$ (l'exemple de la docstring), sa moyenne à l'indice 2 (deux appels à la même graine, l'un avec `x_t` et `eps_pred` nuls, dont la différence est la moyenne), le rapport des bruits des deux variances, l'écrêtage à 0,5 et trois refus ; puis `ddim_step` avec le vrai bruit (vers −1, puis vers l'indice 0), l'état de `rng` avec `eta = 0` et trois refus. La seconde lance les 101 tests de l'exercice (`-k "test_ddpm_step_ or test_ddim_step_"`).

Dans tes notes : pourquoi n'ajoute-t-on pas de bruit au dernier pas, et que verrais-tu sur des images si tu en ajoutais (fiche, piège « l'échelle et le dernier pas ») ? Pourquoi `alpha_bars[-1]` serait-il une erreur silencieuse pour `t_prev = -1` ? Avec `eta = 1` et `t_prev = t - 1`, que vaut $\sigma^2$, et qu'en conclus-tu sur le lien entre DDIM et DDPM ?

In [ ]:
with checked("B5.17"):
    M = mylearn.diffusion
    s_17 = toy_schedule([0.1, 0.2, 0.5])
    b, a, ab = s_17["betas"], s_17["alphas"], s_17["alpha_bars"]

    def algo2_17(x, t, eps):        # the mean of Algorithm 2 (Ho et al.)
        return (np.asarray(x) - b[t] / math.sqrt(1 - ab[t]) * np.asarray(eps)) / math.sqrt(a[t])

    x_17, e_17 = np.array([[0.5, -0.5]]), np.array([[1.0, 1.0]])
    same("B5.17", "ddpm_step à t = 0 (exemple de la docstring : pas de bruit)", M.ddpm_step(x_17, 0, e_17, s_17),
         algo2_17(x_17, 0, e_17), rtol=1e-6, atol=1e-6)
    xb_17 = np.random.default_rng(1).standard_normal((4, 3))
    eb_17 = np.random.default_rng(2).standard_normal((4, 3))
    copies_17 = (xb_17.copy(), eb_17.copy())
    full_17 = M.ddpm_step(xb_17, 2, eb_17, s_17, rng=np.random.default_rng(7))
    again_17 = M.ddpm_step(xb_17, 2, eb_17, s_17, rng=np.random.default_rng(7))
    verdict("B5.17", np.array_equal(np.asarray(full_17), np.asarray(again_17)), "même graine, même résultat.",
            "deux appels avec la même graine doivent rendre le même x_{t-1} : z est-il tiré de rng ?")
    noise_17 = np.asarray(M.ddpm_step(np.zeros((4, 3)), 2, np.zeros((4, 3)), s_17, rng=np.random.default_rng(7)))
    same("B5.17", "moyenne à l'indice 2 (même graine : la différence de deux appels)", np.asarray(full_17) - noise_17,
         algo2_17(xb_17, 2, eb_17), rtol=1e-6, atol=1e-6)
    beta_17 = np.asarray(M.ddpm_step(np.zeros((4, 3)), 2, np.zeros((4, 3)), s_17, rng=np.random.default_rng(7),
                                     variance="beta"))
    same("B5.17", "bruit avec variance='beta' contre 'posterior' : le rapport des écarts-types",
         beta_17, noise_17 * math.sqrt(b[2] / s_17["posterior_variance"][2]), rtol=1e-6, atol=1e-6)
    verdict("B5.17", all(np.array_equal(u, v) for u, v in zip((xb_17, eb_17), copies_17)), "les entrées n'ont pas changé.",
            "ddpm_step a modifié x_t ou eps_pred.")
    big_17 = np.array([[-3.0, 2.0, 0.1]])                                     # x0_hat far outside [-0.5, 0.5]
    xc_17 = np.array([[0.2, -0.1, 0.3]])
    x0hat_17 = (xc_17 - math.sqrt(1 - ab[1]) * big_17) / math.sqrt(ab[1])
    clipped_17 = (np.asarray(M.ddpm_step(xc_17, 1, big_17, s_17, rng=np.random.default_rng(3), clip_x0=0.5))
                  - np.asarray(M.ddpm_step(np.zeros((1, 3)), 1, np.zeros((1, 3)), s_17, rng=np.random.default_rng(3),
                                           clip_x0=0.5)))
    same("B5.17", "clip_x0 = 0.5 : moyenne du posterior de x̂₀ écrêté à [−0,5 ; 0,5]", clipped_17,
         s_17["posterior_mean_coef1"][1] * np.clip(x0hat_17, -0.5, 0.5) + s_17["posterior_mean_coef2"][1] * xc_17,
         rtol=1e-6, atol=1e-6)
    refuses("B5.17", "variance='betas' (inconnue), même à t = 0", lambda: M.ddpm_step(x_17, 0, e_17, s_17, variance="betas"))
    refuses("B5.17", "ddpm_step à t = 3 avec T = 3", lambda: M.ddpm_step(x_17, 3, e_17, s_17))
    refuses("B5.17", "ddpm_step à t = -1", lambda: M.ddpm_step(x_17, -1, e_17, s_17))

    x0_17 = np.array([[0.5, -1.0, 0.25]])
    eps_17 = np.array([[1.0, 0.0, -2.0]])
    xt_17 = math.sqrt(ab[2]) * x0_17 + math.sqrt(1 - ab[2]) * eps_17
    same("B5.17", "ddim_step de l'indice 2 à -1 avec le vrai bruit : x0", M.ddim_step(xt_17, 2, -1, eps_17, ab), x0_17,
         rtol=1e-6, atol=1e-6)
    same("B5.17", "ddim_step de l'indice 2 à 0 avec le vrai bruit : le x au pas 0 du même bruit",
         M.ddim_step(xt_17, 2, 0, eps_17, ab), math.sqrt(ab[0]) * x0_17 + math.sqrt(1 - ab[0]) * eps_17,
         rtol=1e-6, atol=1e-6)
    rng_17 = np.random.default_rng(11)
    state_17 = copy.deepcopy(rng_17.bit_generator.state)
    M.ddim_step(xt_17, 2, 1, eps_17, ab, eta=0.0, rng=rng_17)
    verdict("B5.17", rng_17.bit_generator.state == state_17, "avec eta = 0, rng n'a pas servi.",
            "avec eta = 0, ddim_step ne doit tirer aucun nombre de rng (son état a changé).")
    refuses("B5.17", "ddim_step avec t_prev = t", lambda: M.ddim_step(xt_17, 2, 2, eps_17, ab))
    refuses("B5.17", "ddim_step avec eta = 1.5", lambda: M.ddim_step(xt_17, 2, 1, eps_17, ab, eta=1.5))
    refuses("B5.17", "ddim_step avec eta = nan", lambda: M.ddim_step(xt_17, 2, 1, eps_17, ab, eta=float("nan")))

In [ ]:
if written(lambda: mylearn.diffusion.ddim_step([0.0], 1, 0, [0.0], [0.9, 0.8])):
    run_pytest_group(TEST_DIFF, impl="ref", keyword="test_ddpm_step_ or test_ddim_step_")   # 101 tests
else:
    print("⏳ Ex B5.17 : les tests attendent tes fonctions.")

### Ex B5.18 — Avec le vrai bruit, DDPM et DDIM retrouvent-ils x_0 ? Contrôle sur des données 2D 🔬 ★★ ⏱️ 25 min
**Objectif :** enchaîner tes pas DDPM et DDIM avec le vrai bruit, pour savoir lesquels doivent retrouver $\mathbf{x}_0$ exactement : le contrôle de ta librairie avant tout entraînement.  
**Prérequis :** Ex B5.17 · fiche §205.5 (« L'algorithme »), §205.6 (« L'idée de DDIM », « Le paramètre η ») · **Fil rouge :** synthétique · **Parcours :** M, C

Un réseau parfait prédirait le **vrai** bruit. Sans réseau, on peut donc éprouver une chaîne d'échantillonnage : partir d'une donnée $\mathbf{x}_0$ connue, la bruiter jusqu'au dernier indice, puis remonter en donnant aux pas « le vrai bruit » à la place de $\boldsymbol{\varepsilon}_\theta$. La fiche dit qu'**un** pas DDIM vers l'indice −1 rend alors $\mathbf{x}_0$. Mais quel est « le vrai bruit » au milieu d'une chaîne, et qu'en est-il des chaînes entières ?

**Contrat**
- `true_noise_18(x_t, t, x0, alpha_bars)` : le bruit $\boldsymbol{\varepsilon}$ tel que $\mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1 - \bar\alpha_t}\,\boldsymbol{\varepsilon}$, pour un indice `t` entier (le même pour tout le batch) ; un nouveau tableau, de la forme de `x_t` ;
- `ddpm_chain_18(x0, eps, schedule, rng, oracle=False)` : $\mathbf{x}$ à l'indice $T - 1$ fait de `x0` et `eps` par la forme fermée, puis ta `ddpm_step` de l'indice $T - 1$ à 0 (avec `rng`, sans écrêtage, la variance par défaut), avec pour `eps_pred` le bruit **initial** `eps` à chaque pas (`oracle=False`), ou le vrai bruit du $\mathbf{x}$ **courant**, `true_noise_18(x, t, x0, …)` (`oracle=True`) ; rend le $\mathbf{x}$ final ;
- `ddim_chain_18(x0, eps, alpha_bars, timesteps, eta=0.0, rng=None)` : $\mathbf{x}$ à l'indice `timesteps[0]` fait de `x0` et `eps`, puis ta `ddim_step` de chaque indice de la sous-séquence décroissante `timesteps` au suivant, et du dernier à −1, toujours avec `eps_pred = eps`, le bruit initial ; rend le $\mathbf{x}$ final.

La cellule de vérification essaie tes trois fonctions sur un planning de 3 pas, contre la chaîne écrite pas à pas. Puis l'expérience prend 500 points de l'anneau de 8 gaussiennes (`wbd.ring`), le planning du notebook (ta `noise_schedule` de `BETAS`), et mesure l'écart maximal entre $\mathbf{x}_0$ et la fin de cinq chaînes : DDPM avec le bruit initial ; DDPM avec le vrai bruit du $\mathbf{x}$ courant ; DDIM ($\eta = 0$) sur tous les indices ; DDIM ($\eta = 0$) sur 10 indices (`wbd.ddim_timesteps(T_STEPS, 10)`) ; DDIM ($\eta = 1$) sur tous les indices. Réponds ensuite, d'après le tableau, par `True` ou `False` : la chaîne retrouve-t-elle $\mathbf{x}_0$ à $10^{-6}$ près ?
- a) `ddpm_fixed_18` : DDPM, bruit initial à chaque pas ;
- b) `ddpm_oracle_18` : DDPM, vrai bruit du $\mathbf{x}$ courant ;
- c) `ddim_ten_18` : DDIM, $\eta = 0$, 10 pas ;
- d) `ddim_eta1_18` : DDIM, $\eta = 1$, tous les pas.

Dans tes notes : pourquoi le bruit initial n'est-il plus « le vrai bruit » d'un $\mathbf{x}_t$ de la chaîne DDPM après le premier pas, alors qu'il le reste tout au long de la chaîne DDIM avec $\eta = 0$ ? Pourquoi la chaîne DDPM avec le vrai bruit courant finit-elle exactement sur $\mathbf{x}_0$, quelle que soit sa trajectoire (regarde le dernier pas) ? Pourquoi le nombre de pas de DDIM ne change-t-il rien ici, alors qu'il comptera avec un réseau ? Quelle erreur de ta librairie ce contrôle aurait-il attrapée (un indice décalé, $\bar\alpha$ au lieu de $\sqrt{\bar\alpha}$), et laquelle pas ?

In [ ]:
def true_noise_18(x_t, t, x0, alpha_bars):
    """The noise eps such that x_t = sqrt(alpha_bar_t) x0 + sqrt(1 - alpha_bar_t) eps, for one step index t
    (an int, the same for the whole batch); a new array with the shape of x_t."""
    ab = float(np.asarray(alpha_bars)[t])
    return (np.asarray(x_t, dtype=float) - math.sqrt(ab) * np.asarray(x0, dtype=float)) / math.sqrt(1 - ab)


def ddpm_chain_18(x0, eps, schedule, rng, oracle=False):
    """The whole DDPM chain: x at index T - 1 made from x0 and eps by the closed form, then ddpm_step (your library)
    from index T - 1 down to 0, with as eps_pred the initial eps (oracle=False) or the true noise of the current x
    (true_noise_18, oracle=True). Returns the final x."""
    ab = schedule["alpha_bars"]
    T = len(ab)
    x = math.sqrt(ab[T - 1]) * np.asarray(x0, dtype=float) + math.sqrt(1 - ab[T - 1]) * np.asarray(eps, dtype=float)
    for t in range(T - 1, -1, -1):
        eps_pred = true_noise_18(x, t, x0, ab) if oracle else eps
        x = mylearn.diffusion.ddpm_step(x, t, eps_pred, schedule, rng=rng)
    return x


def ddim_chain_18(x0, eps, alpha_bars, timesteps, eta=0.0, rng=None):
    """The DDIM chain on the decreasing sub-sequence timesteps: x at index timesteps[0] made from x0 and eps by the
    closed form, then ddim_step (your library) from each index to the next one, and from the last one to -1, always
    with eps_pred = eps, the initial noise. Returns the final x."""
    steps = [int(t) for t in timesteps] + [-1]
    a0 = float(alpha_bars[steps[0]])
    x = math.sqrt(a0) * np.asarray(x0, dtype=float) + math.sqrt(1 - a0) * np.asarray(eps, dtype=float)
    for t, t_prev in zip(steps[:-1], steps[1:]):
        x = mylearn.diffusion.ddim_step(x, t, t_prev, eps, alpha_bars, eta=eta, rng=rng)
    return x


ddpm_fixed_18, ddpm_oracle_18, ddim_ten_18, ddim_eta1_18 = False, True, True, False


chains_ok_18 = False
with checked("B5.18"):
    s_18 = toy_schedule([0.1, 0.2, 0.5])
    ab_18 = s_18["alpha_bars"]
    x0_c = np.array([[0.3, -0.7], [1.2, 0.1]])
    e_c = np.array([[0.5, 1.0], [-1.5, 0.2]])
    xt_c = math.sqrt(ab_18[2]) * x0_c + math.sqrt(1 - ab_18[2]) * e_c
    copy_c = xt_c.copy()
    ok_18 = same("B5.18", "true_noise_18 redonne le bruit de x_t", true_noise_18(xt_c, 2, x0_c, ab_18), e_c,
                 rtol=1e-9, atol=1e-12)
    verdict("B5.18", np.array_equal(xt_c, copy_c), "true_noise_18 n'a pas modifié x_t.", "true_noise_18 a modifié x_t.")
    ok_18 &= np.array_equal(xt_c, copy_c)
    # the DDPM chain with T = 3 and the initial noise, written step by step without your functions
    rng_a, rng_b = np.random.default_rng(4), np.random.default_rng(4)
    x_ref = xt_c.copy()
    for t in (2, 1, 0):
        mean = (x_ref - s_18["betas"][t] / math.sqrt(1 - ab_18[t]) * e_c) / math.sqrt(s_18["alphas"][t])
        x_ref = mean if t == 0 else mean + np.asarray(mylearn.diffusion.ddpm_step(
            np.zeros_like(x_ref), t, np.zeros_like(x_ref), s_18, rng=rng_b))
    ok_18 &= same("B5.18", "ddpm_chain_18 (T = 3, bruit initial) contre la chaîne pas à pas",
                  ddpm_chain_18(x0_c, e_c, s_18, rng_a), x_ref, rtol=1e-6, atol=1e-6)
    x_dd = math.sqrt(ab_18[2]) * x0_c + math.sqrt(1 - ab_18[2]) * e_c
    x0h = (x_dd - math.sqrt(1 - ab_18[2]) * e_c) / math.sqrt(ab_18[2])
    ok_18 &= same("B5.18", "ddim_chain_18 sur [2, 0] (η = 0) contre la formule", ddim_chain_18(x0_c, e_c, ab_18, [2, 0]),
                  x0h, rtol=1e-6, atol=1e-6)
    chains_ok_18 = bool(ok_18)
def upstream_18():
    """⏳ when a function of B5.12 or B5.14 that the experiment may go through is wrong: the small checks above run on
    a planning of 3 steps and one step for the whole batch, where these errors do not show."""
    D = mylearn.diffusion
    try:
        ab_ok = np.allclose(np.asarray(D.noise_schedule(BETAS)["alpha_bars"]), toy_schedule(BETAS)["alpha_bars"],
                            rtol=1e-9, atol=1e-12)
    except Exception:  # noqa: BLE001 - not written, or wrong: B5.12 says which
        ab_ok = False
    if not ab_ok:
        print("⏳ Ex B5.18 : l'expérience prend le planning du notebook de ta noise_schedule : elle attend une "
              "noise_schedule juste (🔨 B5.12, relance ses vérifications).")
        return False
    x0_u, t_u, ab_u = np.array([[1.0, -1.0], [0.5, 0.0], [-0.5, 2.0]]), np.array([0, 1, 2]), np.array([0.9, 0.5, 0.2])
    try:
        got_u = np.asarray(D.q_sample(x0_u, t_u, ab_u, noise=np.ones((3, 2)))[0])
    except NotImplementedError:
        return True                                   # q_sample not written: your chains do not use it
    except Exception:  # noqa: BLE001
        got_u = None
    if got_u is None or got_u.shape != (3, 2) or not np.allclose(got_u, np.sqrt(ab_u)[:, None] * x0_u
                                                                  + np.sqrt(1 - ab_u)[:, None], atol=1e-6):
        print("⏳ Ex B5.18 : si tes chaînes passent par ta q_sample, l'expérience attend une q_sample juste sur un batch "
              "(🔨 B5.14, relance ses vérifications).")
        return False
    return True


if not chains_ok_18:
    print("⏳ Ex B5.18 : l'expérience attend tes trois fonctions (et ddpm_step, ddim_step de B5.17).")
elif upstream_18():
    with checked("B5.18"):
        X0_18 = wbd.ring(500, seed=18)                                   # 500 points of the ring of 8 Gaussians
        EPS_18 = np.random.default_rng(180).standard_normal(X0_18.shape)
        SCHED_18 = mylearn.diffusion.noise_schedule(BETAS)
        AB_18 = SCHED_18["alpha_bars"]
        runs_18 = {"DDPM, bruit initial à chaque pas": ddpm_chain_18(X0_18, EPS_18, SCHED_18, np.random.default_rng(1)),
                   "DDPM, vrai bruit du x courant": ddpm_chain_18(X0_18, EPS_18, SCHED_18, np.random.default_rng(1), oracle=True),
                   f"DDIM η = 0, les {T_STEPS} pas": ddim_chain_18(X0_18, EPS_18, AB_18, np.arange(T_STEPS - 1, -1, -1)),
                   "DDIM η = 0, 10 pas": ddim_chain_18(X0_18, EPS_18, AB_18, wbd.ddim_timesteps(T_STEPS, 10)),
                   f"DDIM η = 1, les {T_STEPS} pas": ddim_chain_18(X0_18, EPS_18, AB_18, np.arange(T_STEPS - 1, -1, -1),
                                                                    eta=1.0, rng=np.random.default_rng(2))}
        ERRORS_18 = {name: float(np.abs(np.asarray(x) - X0_18).max()) for name, x in runs_18.items()}
        for name, err in ERRORS_18.items():
            print(f"{name:>32} : écart maximal à x0 = {fr_e(err)}")
        verdict("B5.18", max(ERRORS_18[k] for k in list(ERRORS_18)[1:4]) < 1e-9,
                "les trois chaînes exactes retrouvent x0 à moins de 1e-9.",
                "une chaîne qui devrait être exacte s'écarte de x0 de plus de 1e-9 : relis tes trois fonctions.")
        fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))         # each with its own scale
        for ax, name in zip(axes, [list(runs_18)[0], list(runs_18)[3], list(runs_18)[4]]):
            ax.scatter(*X0_18.T, s=4, alpha=0.4, label="x0")
            ax.scatter(*np.asarray(runs_18[name]).T, s=4, alpha=0.4, label="fin de la chaîne")
            ax.set_title(name, fontsize=9)
            ax.set_aspect("equal")
        axes[0].legend(fontsize=8)
        plt.show()
        for letter, answer in zip("abcd", [ddpm_fixed_18, ddpm_oracle_18, ddim_ten_18, ddim_eta1_18]):
            with wb.attempt("B5.18"):
                wb.check(f"B5.18{letter}", answer)

In [ ]:
wb.record("B5.18a", False, mistakes={"relis l'écart de la première ligne du tableau": True})
wb.record("B5.18b", True, mistakes={"relis l'écart de la deuxième ligne du tableau": False})
wb.record("B5.18c", True, mistakes={"relis l'écart de la quatrième ligne du tableau": False})
wb.record("B5.18d", False, mistakes={"relis l'écart de la dernière ligne du tableau": True})

## Partie B · Jouets 2D : un MLP débruiteur, puis le flow matching

Avant les images, des points du plan : l'**anneau de 8 gaussiennes** (`wbd.ring` : 8 centres sur un cercle de rayon 1,5, écart-type 0,15 autour de chacun ; sa variance par coordonnée, 1,15, est proche de celle du bruit). Tout s'y voit d'un coup d'œil et s'entraîne en une dizaine de secondes. Tu y entraînes un MLP qui prédit le bruit, conditionné par le pas comme le U-Net de la fiche, puis tu l'échantillonnes avec ta `ddpm_step` (B5.19) ; puis tu entraînes le **même** MLP à prédire une vitesse (flow matching, fiche §205.9) et tu compares les deux échantillonneurs à nombre de pas égal (B5.20).

**Mesurer la qualité.** Une distance de Fréchet en 2D ne compare que les moyennes et les covariances : un nuage gaussien de variance 1 a presque celles de l'anneau. On prend la **distance de Wasserstein en tranches** (`wbd.sliced_wasserstein`) : la moyenne, sur 64 directions du plan, de la distance de Wasserstein du ch. 27 entre les deux projections (l'écart moyen de leurs valeurs triées). La fonction `quality(points)` rend cette distance à `REAL_B` (2 000 points de l'anneau), la part des points à moins de trois écarts-types d'un centre et le nombre de modes atteints (`wbd.mode_report`) ; la cellule suivante affiche ses deux repères, deux échantillons de l'anneau et du bruit pur.

**Encoder le pas.** Le MLP reçoit une **position entière**, comme le U-Net : l'indice $t$ du pas pour la diffusion ; pour le flow matching, dont le temps $s$ est continu sur $[0, 1]$, on le prend sur une grille de 1 000 valeurs, $s = u / 1\,000$ avec $u$ entier de 1 à 1 000 (`wbd.FM_POSITIONS` = 1 001 positions). La position $p$ est encodée par la ligne $p$ de ta `positional_encoding` de B3 (`mylearn.attention`) : si tu n'as pas écrit `attention.py`, la référence la remplace (une ligne ℹ️ le dit) ; s'il existe sans cette fonction, la vérification te le signale. La référence d'`attention.py` s'appuie elle-même sur celles de `nn/activations.py` (ch. 17) et de `nn/regularization.py` (ch. 20) : si tu n'as pas ces modules, deux autres lignes ℹ️ le disent, sans effet sur la partie B.

`SETTINGS_B` donne le budget des deux entraînements : 3 000 itérations, des batches de 512 points, AdamW au learning rate 2e-3 (environ 10 s chacun sur un CPU à deux cœurs). `wbd` contient une solution de référence (`Denoiser2D`, `train_toy`, `sample_toy_ddpm`…) : la vérification s'en sert pour comparer tes fonctions ; ne la lis qu'une fois les tiennes écrites.

In [ ]:
# Part B: the ring of 8 Gaussians, the settings of the two trainings, the measures of quality
DATA_B = torch.tensor(wbd.ring(50_000, seed=1900), dtype=torch.float32)   # 50 000 training points
REAL_B = wbd.ring(2000, seed=2020)                                         # 2 000 other points, for the measures
SETTINGS_B = dict(wbd.TOY_SETTINGS)                    # 3 000 iterations, batches of 512, AdamW at 2e-3 (≈ 10 s)
SCHED_B = toy_schedule(BETAS)                          # the schedule of the notebook, T = T_STEPS (without your code)
FLOOR_B = wbd.sliced_wasserstein(wbd.ring(2000, seed=2021), REAL_B)        # two samples of the ring itself
BLOB_B = wbd.sliced_wasserstein(np.random.default_rng(3).standard_normal((2000, 2)), REAL_B)   # pure noise


def quality(points):
    """(sliced Wasserstein distance to REAL_B, share of points on a mode, modes reached) of 2 000 points."""
    report = wbd.mode_report(points)
    return wbd.sliced_wasserstein(points, REAL_B), report["on_mode"], report["modes"]


def first_losses_B(model, batch_size, seed, alpha_bars=None):
    """The first loss of the recipe of the contract on DATA_B (B5.19 with alpha_bars, B5.20, flow matching, without),
    then the first loss of the usual slips, with the same generator and the same model: [(loss, where), ...]. Before any
    update, the first loss depends only on the draws, the input of the network and its target."""
    def draws(order, low=1, high=1001):
        g, out = torch.Generator().manual_seed(seed), {}
        for what in order:
            if what == "i":
                out[what] = torch.randint(len(DATA_B), (batch_size,), generator=g)
            elif what == "t":
                out[what] = (torch.randint(0, len(alpha_bars), (batch_size,), generator=g) if alpha_bars is not None
                             else torch.randint(low, high, (batch_size,), generator=g))
            else:
                out[what] = torch.randn((batch_size, 2), generator=g)
        return DATA_B[out["i"]], out["t"], out["e"]

    def loss(x_in, t, target):
        with torch.no_grad():
            return float(F.mse_loss(model(x_in, t), target))

    rows, ab = [], None if alpha_bars is None else torch.tensor(np.asarray(alpha_bars), dtype=torch.float32)
    for k, order in enumerate(("ite", "iet", "tie", "tei", "eit", "eti")):
        x0, t, eps = draws(order)
        if ab is not None:
            a = ab[t][:, None]
            x_t = a.sqrt() * x0 + (1 - a).sqrt() * eps
            if k == 0:
                rows += [(loss(x_t, t, eps), "du contrat"),
                         (loss(x_t, t, x0), "d'une cible x₀ : relis ce que le réseau doit prédire"),
                         (loss(a * x0 + (1 - a).sqrt() * eps, t, eps), "d'un x_t fait avec ᾱ au lieu de √ᾱ : relis la "
                                                                       "forme fermée"),
                         (loss(x0, t, eps), "d'un réseau qui reçoit x₀ au lieu de x_t")]
                continue
            rows.append((loss(x_t, t, eps), "de tirages faits dans un autre ordre : relis l'ordre du contrat (les "
                                            "indices du batch, les pas, le bruit)"))
        else:
            s = (t.float() / 1000)[:, None]
            x_s = (1 - s) * x0 + s * eps
            if k == 0:
                rows += [(loss(x_s, t, eps - x0), "du contrat"),
                         (loss(x_s, t, x0 - eps), "d'une vitesse de signe opposé : relis la cible du contrat"),
                         (loss(x_s, t, eps), "d'une cible ε : en flow matching, le réseau prédit une vitesse"),
                         (loss(s * x0 + (1 - s) * eps, t, eps - x0), "d'une interpolation inversée : relis x_s (à s = 0, "
                                                                     "la donnée ; à s = 1, le bruit)"),
                         (loss(x0, t, eps - x0), "d'un réseau qui reçoit x₀ au lieu de x_s")]
                x0b, u0, epsb = draws(order, 0, 1000)
                s0 = (u0.float() / 1000)[:, None]
                rows.append((loss((1 - s0) * x0b + s0 * epsb, u0, epsb - x0b), "de positions u tirées de 0 à 999 : relis "
                                                                              "l'intervalle de u"))
                continue
            rows.append((loss(x_s, t, eps - x0), "de tirages faits dans un autre ordre : relis l'ordre du contrat (les "
                                                 "indices du batch, les positions u, le bruit)"))
    return rows


def loss_diagnosis(ex_id, got, rows):
    """After a ❌ on the first losses: which recipe gives the learner's first loss, in one line that says where to look."""
    try:
        first = float(got[0])
    except (TypeError, ValueError, IndexError):
        return
    want = rows[0][0]
    if math.isclose(first, want, rel_tol=1e-5, abs_tol=1e-7):
        print(f"   piste : ta 1ʳᵉ loss est celle du contrat, pas les suivantes : vérifie l'optimiseur (AdamW, son learning "
              "rate), zero_grad, un pas par itération, et que tu entraînes le modèle reçu.")
        return
    for value, where in rows[1:]:
        if math.isclose(first, value, rel_tol=1e-5, abs_tol=1e-7):
            print(f"   piste : ta 1ʳᵉ loss ({fr(first, 5)} au lieu de {fr(want, 5)}) est celle {where}.")
            return
    print(f"   piste : ta 1ʳᵉ loss vaut {fr(first, 5)} au lieu de {fr(want, 5)}, et n'est celle d'aucune erreur courante "
          "(cible, entrée du réseau, ordre des tirages) : compare ta boucle au contrat, ligne par ligne.")


print(f"distance de Wasserstein en tranches : {fr(FLOOR_B, 3)} entre deux échantillons de l'anneau, "
      f"{fr(BLOB_B, 3)} pour du bruit gaussien.")

### Ex B5.19 — Diffusion en 2D : un MLP débruiteur conditionné par le pas de temps, entraîné puis échantillonné 📦 ★★ ⏱️ 30 min
**Objectif :** entraîner un MLP à prédire le bruit de points 2D, puis générer de nouveaux points avec ta `ddpm_step`.  
**Prérequis :** Ex B5.17 · B3 (`positional_encoding`, B3.15) · fiche §205.3 (« La loss simplifiée »), §205.4 (« Dire au réseau quel est le pas »), §205.5 (« L'algorithme ») · **Fil rouge :** synthétique · **Parcours :** R, M, C

**Contrat**
- `Denoiser2D_19(n_positions, t_dim=32, hidden=128)`, un `nn.Module` :
  - un buffer **non persistant** `pe` (`register_buffer(..., persistent=False)` : il n'entre pas dans le `state_dict`), la table `positional_encoding(n_positions, t_dim)` de ta librairie (B3) en float32 ;
  - `net = nn.Sequential(nn.Linear(2 + t_dim, hidden), nn.SiLU(), nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 2))` ;
  - `forward(x, t)` : `x` de forme `(N, 2)` en float32, `t` de forme `(N,)` en int64 → `net` appliqué aux coordonnées **suivies** de l'encodage de la position, `(N, 2)`.
- `train_19(model, data, alpha_bars, n_iters, batch_size, lr, generator)` entraîne `model` sur place et rend la liste des `n_iters` loss (des floats Python) :
  - un `torch.optim.AdamW(model.parameters(), lr=lr)`, avec ses autres réglages par défaut ;
  - à chaque itération, **dans cet ordre**, trois tirages par `generator` : les indices du batch `torch.randint(len(data), (batch_size,))`, les pas `torch.randint(0, T, (batch_size,))` ($T$ = `len(alpha_bars)`), le bruit `torch.randn((batch_size, 2))` ; puis $\mathbf{x}_t$ par la forme fermée en float32, la loss `F.mse_loss(model(x_t, t), eps)`, les gradients et un pas de l'optimiseur.
- `sample_19(model, n, schedule, generator, rng)` : `n` points par l'échantillonnage ancestral : $\mathbf{x}$ = `torch.randn((n, 2), generator=generator)` converti en float64 NumPy, puis, pour $t$ de $T - 1$ à 0, la prédiction du réseau (sans gradient, en float32) et ta `ddpm_step(x, t, eps, schedule, rng=rng)` (variance du posterior, sans écrêtage) ; rend un tableau float64 `(n, 2)`.

La cellule de vérification contrôle ton MLP (noms et formes du `state_dict`, le buffer `pe`, la sortie aux mêmes poids que le modèle du contrat), 5 itérations de `train_19` contre la recette du contrat (mêmes tirages, mêmes loss) et `sample_19` sur 6 points. La cellule suivante entraîne ton MLP avec les réglages de `SETTINGS_B` sur `DATA_B`, génère 2 000 points, trace la loss et les points, et vérifie qu'ils couvrent les 8 modes (une distance en tranches sous 0,2, plus de 80 % des points sur un mode). Le temps affiché est une mesure, pas une réponse.

Dans tes notes : la loss descend-elle vers 0 ? Pourquoi pas, alors que le modèle génère de bons points (pense à ce que vaut l'erreur sur $\boldsymbol{\varepsilon}$ aux petits indices, où $\mathbf{x}_t$ ne dit presque rien du bruit) ? Combien de passes du réseau a coûté l'échantillonnage ? Que se passe-t-il si tu supprimes l'encodage du pas (des zéros à la place) ?

In [ ]:
class Denoiser2D_19(nn.Module):
    """MLP that predicts the noise of a 2-D point x_t at the integer position t (contract of the statement)."""

    def __init__(self, n_positions, t_dim=32, hidden=128):
        super().__init__()
        table = mylearn.attention.positional_encoding(n_positions, t_dim)        # your encoding of B3
        self.register_buffer("pe", torch.tensor(table, dtype=torch.float32), persistent=False)
        self.net = nn.Sequential(nn.Linear(2 + t_dim, hidden), nn.SiLU(), nn.Linear(hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 2))

    def forward(self, x, t):
        return self.net(torch.cat([x, self.pe[t]], dim=1))


def train_19(model, data, alpha_bars, n_iters, batch_size, lr, generator):
    """Train model to predict the noise (recipe of the contract); returns the list of the losses (floats)."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    ab = torch.tensor(np.asarray(alpha_bars), dtype=torch.float32)
    losses = []
    for _ in range(n_iters):
        x0 = data[torch.randint(len(data), (batch_size,), generator=generator)]
        t = torch.randint(0, len(ab), (batch_size,), generator=generator)
        eps = torch.randn((batch_size, 2), generator=generator)
        a = ab[t][:, None]
        loss = F.mse_loss(model(a.sqrt() * x0 + (1 - a).sqrt() * eps, t), eps)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses


def sample_19(model, n, schedule, generator, rng):
    """n points by DDPM ancestral sampling with your ddpm_step; a float64 array (n, 2)."""
    x = torch.randn((n, 2), generator=generator).double().numpy()
    with torch.no_grad():
        for t in range(len(schedule["alpha_bars"]) - 1, -1, -1):
            eps = model(torch.tensor(x, dtype=torch.float32), torch.full((n,), t, dtype=torch.long)).double().numpy()
            x = mylearn.diffusion.ddpm_step(x, t, eps, schedule, rng=rng)
    return x


model_ok_19 = False
with checked("B5.19"):
    torch.manual_seed(0)
    mine_19 = Denoiser2D_19(50)
    ref_19 = wbd.Denoiser2D(50)
    names_19 = {k: tuple(v.shape) for k, v in mine_19.state_dict().items()}
    want_19 = {k: tuple(v.shape) for k, v in ref_19.state_dict().items()}
    ok_19 = names_19 == want_19
    verdict("B5.19", ok_19, "noms et formes des paramètres : ceux du contrat (net.0 à net.6).",
            f"le state_dict doit avoir les noms et formes {want_19} ; reçu {names_19} (le buffer pe est-il non persistant ?).")
    if not ok_19:
        raise StopCheck
    pe_19 = getattr(mine_19, "pe", None)
    pe_ok_19 = (pe_19 is not None and tuple(pe_19.shape) == (50, 32)
                and np.allclose(pe_19.double().numpy(), wbd.sinusoidal_table(50, 32), atol=1e-6))
    verdict("B5.19", pe_ok_19, "le buffer pe est la table de positional_encoding(50, 32) (B3).",
            "self.pe doit être la table positional_encoding(n_positions, t_dim) de B3, en float32.")
    ok_19 &= pe_ok_19
    with torch.no_grad():
        for p_ in ref_19.parameters():
            p_.add_(0.3 * torch.randn(p_.shape))                    # weights far from the initialisation
    mine_19.load_state_dict(ref_19.state_dict())
    xq_19, tq_19 = torch.randn(7, 2), torch.tensor([0, 3, 9, 17, 25, 33, 49])
    with torch.no_grad():
        out_ok_19 = same("B5.19", "sortie de ton MLP contre celle du contrat, aux mêmes poids", mine_19(xq_19, tq_19),
                         ref_19(xq_19, tq_19), rtol=1e-5, atol=1e-6)
    if not out_ok_19:
        print("   piste : net reçoit-il les deux coordonnées, puis l'encodage self.pe[t] (dans cet ordre), et rien d'autre ?")
    ok_19 &= out_ok_19
    # 5 iterations of train_19 against the recipe of the contract (the same draws, the same model at the start)
    ab_19 = toy_schedule(np.linspace(1e-3, 0.2, 50))["alpha_bars"]
    a_19, b_19 = copy.deepcopy(ref_19), copy.deepcopy(ref_19)
    got_19 = train_19(a_19, DATA_B, ab_19, 5, 64, 1e-3, torch.Generator().manual_seed(5))
    want_l19 = wbd.train_toy(b_19, DATA_B, 5, 64, 1e-3, torch.Generator().manual_seed(5), ab_19)
    if returned("B5.19", "train_19", got_19):
        losses_ok_19 = same("B5.19", "les 5 premières loss de train_19 contre la recette du contrat (mêmes tirages, même "
                            "entrée, même cible)", got_19, want_l19, rtol=1e-5, atol=1e-6)
        if not losses_ok_19:
            loss_diagnosis("B5.19", got_19, first_losses_B(ref_19, 64, 5, ab_19))
        ok_19 &= losses_ok_19
        ok_19 &= same("B5.19", "les poids après 5 itérations", a_19.net[0].weight, b_19.net[0].weight, rtol=1e-4,
                      atol=1e-6)
    else:
        ok_19 = False
    sched_19 = toy_schedule(np.linspace(1e-3, 0.2, 50))
    b_19.eval()
    pts_19 = sample_19(b_19, 6, sched_19, torch.Generator().manual_seed(8), np.random.default_rng(9))
    if returned("B5.19", "sample_19", pts_19):
        sample_ok_19 = same("B5.19", "sample_19 (6 points, T = 50) contre l'échantillonnage DDPM du contrat", pts_19,
                            wbd.sample_toy_ddpm(b_19, 6, sched_19["alpha_bars"], torch.Generator().manual_seed(8),
                                                np.random.default_rng(9)), rtol=1e-5, atol=1e-5)
        if not sample_ok_19:
            print("   piste : le départ torch.randn((n, 2), generator=generator), puis tous les indices de T − 1 à 0 (0 "
                  "compris), chacun avec la prédiction du réseau et ta ddpm_step(x, t, eps, schedule, rng=rng).")
        ok_19 &= sample_ok_19
        model_ok_19 = bool(ok_19)

In [ ]:
# Train on the ring, then 2 000 points by DDPM (≈ 10 s of training)
MODEL_19 = None
if not model_ok_19:
    print("⏳ Ex B5.19 : l'entraînement attend ton MLP, train_19 et sample_19 (cellule précédente sans ❌).")
else:
    torch.manual_seed(19)
    MODEL_19 = Denoiser2D_19(T_STEPS)
    start_19 = time.perf_counter()
    LOSSES_19 = train_19(MODEL_19, DATA_B, SCHED_B["alpha_bars"], SETTINGS_B["n_iters"], SETTINGS_B["batch_size"],
                         SETTINGS_B["lr"], torch.Generator().manual_seed(190))
    MODEL_19.eval()
    POINTS_19 = sample_19(MODEL_19, 2000, SCHED_B, torch.Generator().manual_seed(191), np.random.default_rng(192))
    swd_19, on_19, modes_19 = quality(POINTS_19)
    print(f"entraînement et échantillonnage : {time.perf_counter() - start_19:.0f} s (une mesure, pas une réponse)")
    print(f"loss moyenne des 200 dernières itérations : {fr(np.mean(LOSSES_19[-200:]), 3)} ; distance en tranches "
          f"{fr(swd_19, 3)} ; {fr(100 * on_19, 1)} % des points sur un mode ; {modes_19} modes sur 8")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(np.convolve(LOSSES_19, np.ones(50) / 50, mode="valid"))
    ax1.set_xlabel("itération")
    ax1.set_ylabel("loss (moyenne glissante sur 50)")
    ax2.scatter(*REAL_B.T, s=3, alpha=0.3, label="anneau")
    ax2.scatter(*POINTS_19.T, s=3, alpha=0.3, label="DDPM")
    ax2.set_aspect("equal")
    ax2.legend()
    plt.show()
    verdict("B5.19", swd_19 < 0.2 and on_19 > 0.8 and modes_19 == 8,
            "les points générés couvrent les 8 modes, à distance de l'anneau proche de celle de deux échantillons.",
            "les points générés sont loin de l'anneau (distance ≥ 0,2, moins de 80 % sur un mode, ou un mode manqué) : "
            "relis l'entraînement (le pas t donné au modèle, la cible) et l'échantillonnage.")

### Ex B5.20 — Flow matching en 2D : même MLP, cible « vitesse » et intégration d'Euler 🔬 ★★★ ⏱️ 40 min
**Objectif :** entraîner le même MLP à prédire une vitesse, l'intégrer par la méthode d'Euler, et comparer la qualité au DDIM du modèle de B5.19 selon le nombre de pas, sans présumer du gagnant.  
**Prérequis :** Ex B5.19 · fiche §205.9 (« Flow matching et rectified flow », encadré « la méthode d'Euler »), §205.6 (« L'idée de DDIM ») · **Fil rouge :** synthétique · **Parcours :** M, C

Le flow matching relie la donnée et le bruit par la ligne droite $\mathbf{x}_s = (1 - s)\,\mathbf{x}_0 + s\,\boldsymbol{\varepsilon}$ (fiche §205.9) et fait prédire au réseau la vitesse $\mathbf{v} = \boldsymbol{\varepsilon} - \mathbf{x}_0$ ; on génère en suivant le champ des vitesses de $s = 1$ (le bruit) à $s = 0$. La question : à nombre de pas égal, ces points valent-ils mieux que ceux de DDIM avec le modèle de B5.19, qui prédit le bruit ? Même MLP, même budget d'entraînement, mêmes points de départ.

**Contrat**
- `train_fm_20(model, data, n_iters, batch_size, lr, generator)` : la recette de `train_19`, à trois différences près : le deuxième tirage est la position `u = torch.randint(1, 1001, (batch_size,))`, et $s = u / 1\,000$ ; l'entrée est $\mathbf{x}_s$ ; la cible est $\boldsymbol{\varepsilon} - \mathbf{x}_0$, et le modèle reçoit la position `u` (un `Denoiser2D_19(wbd.FM_POSITIONS)`, 1 001 positions). Rend la liste des loss.
- `euler_20(model, n, n_steps, generator)` : $\mathbf{x}$ = `torch.randn((n, 2), generator=generator)`, puis `n_steps` pas d'Euler égaux, de $s = 1$ vers $s = 0$ : au pas $k$ (de 0 à `n_steps` − 1), la position `round(1000 * (1 - k / n_steps))` et $\mathbf{x} \leftarrow \mathbf{x} - \mathbf{v}/\text{n\_steps}$ ; sans gradient ; rend un tableau float64 `(n, 2)`.
- `ddim_20(model, n, n_steps, alpha_bars, generator)` : $\mathbf{x}$ = `torch.randn((n, 2), generator=generator)` en float64 NumPy, puis ta `ddim_step` ($\eta = 0$) d'un indice au suivant de `wbd.ddim_timesteps(T, n_steps)` (de $T - 1$ à 0, `n_steps` indices), et du dernier à −1 ; rend un tableau float64 `(n, 2)`.

La cellule de vérification compare tes trois fonctions aux fonctions du contrat (5 itérations, puis 1, 3 et 8 pas d'Euler, 1, 4 et 50 pas de DDIM). La cellule suivante (environ 15 s) entraîne le modèle de flow matching avec `SETTINGS_B`, puis mesure la distance en tranches de 2 000 points de chaque échantillonneur pour chaque nombre de pas de `STEPS_20`, avec le modèle de B5.19 pour DDIM ; elle trace les deux courbes, avec, en bandes, les **cinq graines** du protocole complet (`FULL_20`, précalculé avec les mêmes cellules ; ta graine est la première), et montre les points à 5 pas. Réponds ensuite d'après `FULL_20` (une liste par graine, une distance par nombre de pas de `STEPS_20` ; le tableau imprimé n'en donne que les médianes, les bandes du graphique l'étendue), **pour toutes ses graines** :
- a) `fewest_20` : le plus petit nombre de pas de `STEPS_20` à partir duquel la distance d'Euler est au plus le triple de sa distance à 100 pas ;
- b) `better5_20` : à 5 pas, lequel a la plus petite distance : `"D"` (DDIM) ou `"E"` (Euler) ?

Dans tes notes : pourquoi DDIM à 1 ou 2 pas rend-il des points aussi loin (regarde $\hat{\mathbf{x}}_0$ au dernier indice, et la division par $\sqrt{\bar\alpha_t}$) ? Et Euler à 1 pas, où envoie-t-il tous les points (fiche, encadré 🧮 « la MSE apprend une moyenne ») ? Le gagnant change-t-il avec le nombre de pas ? À 100 pas, compare le DDIM et le DDPM de B5.19 sur le même modèle : que conclus-tu de l'aléa de l'échantillonnage ancestral ? Qu'est-ce que la comparaison ne dit pas (un seul dataset, des réglages choisis pour l'un ou l'autre, la loss de chaque modèle) ?

In [ ]:
STEPS_20 = (1, 2, 5, 10, 20, 50, 100)                       # numbers of sampling steps of the comparison
# The full protocol (tools/chapters/history_chB5.py): the same cells with 5 seeds (seed 0 is yours), the
# distance of 2 000 points to REAL_B for each number of steps; DDIM depends on T, so on the mode
FULL_20 = {
    "seeds": [0, 1, 2, 3, 4],
    "DDIM (bruit prédit, B5.19)": [[6.8056, 6.6697, 0.4236, 0.2481, 0.1714, 0.147, 0.1404], [6.1998, 6.0638, 0.2993, 0.166, 0.1385, 0.1305, 0.1267], [5.4068, 5.2936, 0.2236, 0.1235, 0.0988, 0.0944, 0.0941], [6.4142, 6.281, 0.4477, 0.2235, 0.142, 0.1142, 0.1103], [6.4573, 6.3336, 0.2506, 0.1266, 0.0929, 0.0793, 0.078]] if FAST_MODE else [[3.3798, 3.3267, 0.4504, 0.2462, 0.1769, 0.1459, 0.1325], [5.2962, 5.2241, 0.3061, 0.1471, 0.1043, 0.0893, 0.0885], [7.4443, 7.3397, 0.4981, 0.1764, 0.1343, 0.1246, 0.1249], [4.9974, 4.9374, 0.3049, 0.1222, 0.0803, 0.0845, 0.0825], [5.2451, 5.1807, 0.2499, 0.0878, 0.09, 0.0913, 0.0908]],
    "Euler (vitesse prédite)": [[0.9146, 0.3714, 0.1034, 0.0814, 0.0774, 0.0751, 0.0749], [0.8998, 0.3423, 0.0725, 0.0441, 0.0403, 0.0414, 0.0404], [0.8987, 0.3947, 0.0889, 0.0574, 0.0502, 0.0509, 0.0505], [0.8727, 0.3712, 0.1305, 0.1156, 0.109, 0.1006, 0.1009], [0.8969, 0.3872, 0.084, 0.0549, 0.0442, 0.0393, 0.0361]],
}

In [ ]:
def train_fm_20(model, data, n_iters, batch_size, lr, generator):
    """Train model to predict the velocity eps - x0 at s = u / 1000 (recipe of the contract); returns the losses."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    for _ in range(n_iters):
        x0 = data[torch.randint(len(data), (batch_size,), generator=generator)]
        u = torch.randint(1, 1001, (batch_size,), generator=generator)
        eps = torch.randn((batch_size, 2), generator=generator)
        s = (u.float() / 1000)[:, None]
        loss = F.mse_loss(model((1 - s) * x0 + s * eps, u), eps - x0)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses


def euler_20(model, n, n_steps, generator):
    """n points by Euler integration of the flow from s = 1 to s = 0 in n_steps steps; a float64 array (n, 2)."""
    x = torch.randn((n, 2), generator=generator)
    with torch.no_grad():
        for k in range(n_steps):
            u = round(1000 * (1 - k / n_steps))                     # s = 1, 1 - 1/n, ..., 1/n
            x = x - model(x, torch.full((n,), u, dtype=torch.long)) / n_steps
    return x.double().numpy()


def ddim_20(model, n, n_steps, alpha_bars, generator):
    """n points by DDIM (eta = 0, your ddim_step) on wbd.ddim_timesteps(T, n_steps), then to -1; float64 (n, 2)."""
    x = torch.randn((n, 2), generator=generator).double().numpy()
    steps = [int(t) for t in wbd.ddim_timesteps(len(alpha_bars), n_steps)] + [-1]
    with torch.no_grad():
        for t, t_prev in zip(steps[:-1], steps[1:]):
            eps = model(torch.tensor(x, dtype=torch.float32), torch.full((n,), t, dtype=torch.long)).double().numpy()
            x = mylearn.diffusion.ddim_step(x, t, t_prev, eps, alpha_bars)
    return x


fm_ok_20 = False
with checked("B5.20"):
    torch.manual_seed(1)
    ref_20 = wbd.Denoiser2D(wbd.FM_POSITIONS)
    a_20, b_20 = copy.deepcopy(ref_20), copy.deepcopy(ref_20)
    got_20 = train_fm_20(a_20, DATA_B, 5, 64, 1e-3, torch.Generator().manual_seed(6))
    ok_20 = returned("B5.20", "train_fm_20", got_20)
    if ok_20:
        ok_20 = same("B5.20", "les 5 premières loss de train_fm_20 contre la recette du contrat (mêmes tirages, même "
                     "entrée, même cible)", got_20,
                     wbd.train_toy(b_20, DATA_B, 5, 64, 1e-3, torch.Generator().manual_seed(6), flow=True),
                     rtol=1e-5, atol=1e-6)
        if not ok_20:
            loss_diagnosis("B5.20", got_20, first_losses_B(ref_20, 64, 6))
    b_20.eval()
    euler_ok_20 = True
    for steps_c in (1, 3, 8):
        euler_ok_20 &= same("B5.20", f"euler_20 en {steps_c} pas contre l'intégration du contrat",
                            euler_20(b_20, 5, steps_c, torch.Generator().manual_seed(7)),
                            wbd.sample_toy_euler(b_20, 5, steps_c, torch.Generator().manual_seed(7)), rtol=1e-5, atol=1e-5)
    if not euler_ok_20:
        print("   piste : au pas k, la position round(1000 * (1 - k / n_steps)) (le premier pas part du bruit pur, s = 1), "
              "et le sens du pas : on recule de s = 1 vers s = 0.")
    ok_20 &= euler_ok_20
    torch.manual_seed(2)
    eps_net_20 = wbd.Denoiser2D(50).eval()
    ab_20 = toy_schedule(np.linspace(1e-3, 0.2, 50))["alpha_bars"]
    ddim_ok_20 = True
    for steps_c in (1, 4, 50):
        ddim_ok_20 &= same("B5.20", f"ddim_20 en {steps_c} pas contre le DDIM du contrat",
                           ddim_20(eps_net_20, 5, steps_c, ab_20, torch.Generator().manual_seed(8)),
                           wbd.sample_toy_ddim(eps_net_20, 5, steps_c, ab_20, torch.Generator().manual_seed(8)),
                           rtol=1e-5, atol=1e-5)
    if not ddim_ok_20:
        print("   piste : la sous-séquence wbd.ddim_timesteps(T, n_steps), de T − 1 à 0, puis un dernier pas de 0 vers −1, "
              "chacun avec ta ddim_step (η = 0).")
    ok_20 &= ddim_ok_20
    fm_ok_20 = bool(ok_20)

In [ ]:
# The flow matching model, the same MLP and the same budget as in B5.19, then both samplers from 1 to 100 steps
MEASURED_20 = None
if not (fm_ok_20 and MODEL_19 is not None):
    print("⏳ Ex B5.20 : l'expérience attend tes trois fonctions (cellule précédente sans ❌) et le modèle de B5.19.")
else:
    torch.manual_seed(20)
    MODEL_FM_20 = Denoiser2D_19(wbd.FM_POSITIONS)
    start_20 = time.perf_counter()
    LOSSES_FM_20 = train_fm_20(MODEL_FM_20, DATA_B, SETTINGS_B["n_iters"], SETTINGS_B["batch_size"], SETTINGS_B["lr"],
                               torch.Generator().manual_seed(200))
    MODEL_FM_20.eval()
    print(f"entraînement du flow matching : {time.perf_counter() - start_20:.0f} s")
    MEASURED_20 = {"DDIM (bruit prédit, B5.19)": [], "Euler (vitesse prédite)": []}
    SAMPLES_20 = {}
    for n_20 in STEPS_20:
        pts_d = ddim_20(MODEL_19, 2000, n_20, SCHED_B["alpha_bars"], torch.Generator().manual_seed(201))
        pts_e = euler_20(MODEL_FM_20, 2000, n_20, torch.Generator().manual_seed(201))
        SAMPLES_20[n_20] = (pts_d, pts_e)
        MEASURED_20["DDIM (bruit prédit, B5.19)"].append(quality(pts_d)[0])
        MEASURED_20["Euler (vitesse prédite)"].append(quality(pts_e)[0])
    print("pas :", "  ".join(f"{n:>6}" for n in STEPS_20))
    for name, values in MEASURED_20.items():
        print(f"{name} :", "  ".join(f"{fr(v, 3):>6}" for v in values))
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
    for (name, values), color in zip(MEASURED_20.items(), ("C0", "C1")):
        ax1.plot(STEPS_20, values, "o-", color=color, label=f"{name}, ton protocole")
        rows = np.array(FULL_20[name])                         # (seeds, steps) of the full protocol
        ax1.fill_between(STEPS_20, rows.min(axis=0), rows.max(axis=0), color=color, alpha=0.15)
    ax1.axhline(FLOOR_B, color="gray", lw=0.8, ls=":", label="deux échantillons de l'anneau")
    ax1.set_xscale("log")
    ax1.set_yscale("log")
    ax1.set_xlabel("nombre de pas d'échantillonnage")
    ax1.set_ylabel("distance de Wasserstein en tranches")
    ax1.legend(fontsize=7)
    for ax, (pts, title) in zip((ax2, ax3), ((SAMPLES_20[5][0], "DDIM, 5 pas"), (SAMPLES_20[5][1], "Euler, 5 pas"))):
        ax.scatter(*REAL_B.T, s=3, alpha=0.2, color="gray")
        ax.scatter(*np.clip(pts, -4, 4).T, s=3, alpha=0.4)
        ax.set_xlim(-4, 4)
        ax.set_ylim(-4, 4)
        ax.set_aspect("equal")
        ax.set_title(title)
    plt.show()
    print(f"protocole complet ({len(FULL_20['seeds'])} graines, bandes du graphique) : distance médiane par nombre de pas")
    for name in MEASURED_20:
        print(f"{name} :", "  ".join(f"{fr(v, 3):>6}" for v in np.median(np.array(FULL_20[name]), axis=0)))

In [ ]:
fewest_20, better5_20 = 5, "E"

with wb.attempt("B5.20"):
    wb.check("B5.20a", fewest_20)
with wb.attempt("B5.20"):
    wb.check("B5.20b", better5_20)

In [ ]:
wb.record("B5.20a", 5, mistakes={"relis les distances d'Euler de FULL_20 : à 1 pas, la distance est-elle au plus le triple de celle à 100 pas pour toutes les graines ?": 1, "relis les distances d'Euler de FULL_20 : à 2 pas, la distance est-elle au plus le triple de celle à 100 pas pour toutes les graines ?": 2, "relis les distances d'Euler de FULL_20 : à 10 pas, la distance est-elle au plus le triple de celle à 100 pas pour toutes les graines ?": 10, "relis les distances d'Euler de FULL_20 : à 20 pas, la distance est-elle au plus le triple de celle à 100 pas pour toutes les graines ?": 20, "relis les distances d'Euler de FULL_20 : à 50 pas, la distance est-elle au plus le triple de celle à 100 pas pour toutes les graines ?": 50, "relis les distances d'Euler de FULL_20 : à 100 pas, la distance est-elle au plus le triple de celle à 100 pas pour toutes les graines ?": 100}, fractional="on demande un nombre de pas de STEPS_20", choices=[1, 2, 5, 10, 20, 50, 100])
wb.record("B5.20b", "E", mistakes={"relis la colonne des 5 pas de FULL_20, graine par graine": "D"})

## Partie C · Le U-Net sur MNIST et Fashion-MNIST

Place aux images. Tu écris d'abord le **U-Net minuscule** de la fiche (B5.21), que la suite de la partie entraîne sur MNIST, échantillonne, conditionne par la classe et évalue. Les images sont ramenées dans $[-1, 1]$ par une seule fonction, `wbd.to_signed` ($x / 127{,}5 - 1$ pour des octets) : le réseau, la forme fermée et l'écrêtage de $\hat{\mathbf{x}}_0$ supposent tous cette échelle, et `wbd.to_unit` les ramène dans $[0, 1]$ pour l'affichage. Les données d'entraînement sont un sous-ensemble **figé** (`wbd.image_subset`) : 10 000 images en mode rapide, les 60 000 en mode complet, toujours dans le même ordre, de sorte que les petits sous-ensembles sont inclus dans les grands.

### Ex B5.21 — Un U-Net minuscule en PyTorch : blocs convolutifs, connexions de saut, encodage du pas de temps 📦 ★★★ ⏱️ 50 min
**Objectif :** écrire le U-Net de la fiche, avec son encodage du pas et une classe facultative, et le valider étage par étage contre le modèle du contrat.  
**Prérequis :** Ex B5.19 · B3 (`positional_encoding`) · ch. 21 (convolutions de stride 2, convolution transposée) · fiche §205.4 (« Le U-Net et ses connexions de saut », « Un bloc résiduel conditionné par le pas ») · §205.7 (« Donner la classe au réseau ») · **Fil rouge :** MNIST · **Parcours :** R, C

**Contrat.** Les noms des modules comptent : la suite du chapitre sauvegarde ton modèle par son `state_dict` et le recharge.
- `ResBlock_21(c_in, c_out, t_dim)` : le bloc de la fiche (§205.4), avec ses six sous-modules `norm1` (`nn.GroupNorm(8, c_in)`), `conv1` (3×3, `padding=1`), `time` (`nn.Linear(t_dim, c_out)`), `norm2`, `conv2` et `skip` (une convolution 1×1 si `c_in != c_out`, sinon `nn.Identity()`) ; `forward(x, temb)` : $\mathbf{h}$ = `conv1(silu(norm1(x)))`, plus `time(temb)` en un biais par canal, puis `conv2(silu(norm2(h)))`, plus `skip(x)`.
- `TinyUNet_21(n_steps, channels=(32, 64, 128), t_dim=128, n_classes=None)`, avec `c1, c2, c3 = channels` :
  - `self.config`, le dictionnaire `{"n_steps", "channels" (une liste), "t_dim", "n_classes"}` de ses arguments : il suffit à reconstruire le modèle (`TinyUNet_21(**config)`) ;
  - un buffer **non persistant** `pe` : la table `positional_encoding(n_steps, t_dim)` de B3, en float32 ; `t_mlp = nn.Sequential(nn.Linear(t_dim, t_dim), nn.SiLU(), nn.Linear(t_dim, t_dim))` ;
  - si `n_classes` est un entier : `class_emb = nn.Embedding(n_classes + 1, t_dim)`, dont la **dernière** ligne, d'indice `n_classes`, est la classe vide $\varnothing$ de la guidance sans classifieur ;
  - `inp` (1 → `c1`, 3×3) ; l'encodeur `d1 = ResBlock_21(c1, c1)`, `down1` (convolution 3×3 de stride 2, `padding=1`, `c1` → `c1` : 28 → 14), `d2 = ResBlock_21(c1, c2)`, `down2` (la même convolution 3×3 de stride 2, `padding=1`, `c2` → `c2` : 14 → 7) ; `mid = ResBlock_21(c2, c3)` ; le décodeur `up2` (`nn.ConvTranspose2d(c3, c2, 4, stride=2, padding=1)` : 7 → 14), `u2 = ResBlock_21(2 * c2, c2)`, `up1` (la même convolution transposée, `nn.ConvTranspose2d(c2, c1, 4, stride=2, padding=1)` : 14 → 28), `u1 = ResBlock_21(2 * c1, c1)` ; `out` (`c1` → 1, 3×3) ; tous les blocs reçoivent `t_dim` ;
  - `forward(x, t, y=None)` : `x` de forme `(N, 1, 28, 28)`, `t` des indices int64 `(N,)`, `y` des classes int64 `(N,)` de 0 à `n_classes` (la classe vide si `y` est `None`) ; `temb = t_mlp(pe[t])`, plus `class_emb(y)` si le modèle a des classes ; puis `h1 = d1(inp(x))`, `h2 = d2(down1(h1))`, `h = mid(down2(h2))`, `h = u2(cat([up2(h), h2]))`, `h = u1(cat([up1(h), h1]))` (la concaténation sur l'axe des canaux, **la sortie du décodeur d'abord**), et `out(h)` : la prédiction du bruit, `(N, 1, 28, 28)`.

Puis, dans la même cellule, avec ton modèle :
- a) `n_small_21` : le nombre de paramètres de `TinyUNet_21(400, channels=(16, 32, 64), t_dim=64)` ;
- b) `u1_in_21` : la forme de l'entrée de `u1` (après la concaténation) pour ce modèle et un batch de 8 images, un tuple de 4 entiers.

La cellule de vérification compare ton bloc à celui du contrat aux mêmes poids, puis, pour trois réglages (celui de la fiche, `(16, 32, 64)` avec `t_dim=64`, et 10 classes), les noms et les formes du `state_dict`, le buffer `pe`, et la sortie de ton U-Net à celle du U-Net du contrat (`wbd.TinyUNet`) aux mêmes poids, éloignés de l'initialisation ; elle compare enfin la forme de chaque étage (par des *hooks*, `wbd.stage_shapes`) et vérifie tes deux réponses. La cellule suivante choisit le modèle de la suite du chapitre, `UNET_CLASS` : le tien s'il a tout passé, sinon celui du contrat, et le dit ; elle mesure une passe sur un batch de 64 images (une mesure, pas une réponse).

Dans tes notes : combien de canaux reçoit `u2`, et pourquoi la convolution qui suit doit-elle les attendre (fiche, piège « deux *skip connections* ») ? Que perdrait le U-Net sans ses connexions de saut ? Pourquoi une group norm plutôt qu'une batchnorm ici ? Pourquoi `pe` est-il un buffer non persistant, et que se passerait-il en rechargeant un `state_dict` avec un autre `n_steps` ? D'après ta mesure, combien de temps coûteraient 400 pas de DDPM pour 64 images, et 50 pas de DDIM ?

In [ ]:
class ResBlock_21(nn.Module):
    """Residual block conditioned by the step (contract of the statement)."""

    def __init__(self, c_in, c_out, t_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, c_in)
        self.conv1 = nn.Conv2d(c_in, c_out, 3, padding=1)
        self.time = nn.Linear(t_dim, c_out)
        self.norm2 = nn.GroupNorm(8, c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1)
        self.skip = nn.Conv2d(c_in, c_out, 1) if c_in != c_out else nn.Identity()

    def forward(self, x, temb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time(temb)[:, :, None, None]              # one bias per channel, from the step
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)                                 # residual connection


class TinyUNet_21(nn.Module):
    """A tiny U-Net that predicts the noise of x_t (contract of the statement)."""

    def __init__(self, n_steps, channels=(32, 64, 128), t_dim=128, n_classes=None):
        super().__init__()
        c1, c2, c3 = channels
        self.config = {"n_steps": n_steps, "channels": [c1, c2, c3], "t_dim": t_dim, "n_classes": n_classes}
        self.n_classes = n_classes
        table = mylearn.attention.positional_encoding(n_steps, t_dim)            # your encoding of B3
        self.register_buffer("pe", torch.tensor(table, dtype=torch.float32), persistent=False)
        self.t_mlp = nn.Sequential(nn.Linear(t_dim, t_dim), nn.SiLU(), nn.Linear(t_dim, t_dim))
        if n_classes is not None:
            self.class_emb = nn.Embedding(n_classes + 1, t_dim)                 # the last row is the empty class
        self.inp = nn.Conv2d(1, c1, 3, padding=1)
        self.d1 = ResBlock_21(c1, c1, t_dim)
        self.down1 = nn.Conv2d(c1, c1, 3, stride=2, padding=1)                  # 28 -> 14
        self.d2 = ResBlock_21(c1, c2, t_dim)
        self.down2 = nn.Conv2d(c2, c2, 3, stride=2, padding=1)                  # 14 -> 7
        self.mid = ResBlock_21(c2, c3, t_dim)
        self.up2 = nn.ConvTranspose2d(c3, c2, 4, stride=2, padding=1)           # 7 -> 14
        self.u2 = ResBlock_21(2 * c2, c2, t_dim)
        self.up1 = nn.ConvTranspose2d(c2, c1, 4, stride=2, padding=1)           # 14 -> 28
        self.u1 = ResBlock_21(2 * c1, c1, t_dim)
        self.out = nn.Conv2d(c1, 1, 3, padding=1)

    def forward(self, x, t, y=None):
        temb = self.t_mlp(self.pe[t])
        if self.n_classes is not None:
            temb = temb + self.class_emb(torch.full_like(t, self.n_classes) if y is None else y)
        h1 = self.d1(self.inp(x), temb)
        h2 = self.d2(self.down1(h1), temb)
        h = self.mid(self.down2(h2), temb)
        h = self.u2(torch.cat([self.up2(h), h2], dim=1), temb)
        h = self.u1(torch.cat([self.up1(h), h1], dim=1), temb)
        return self.out(h)


n_small_21 = sum(p.numel() for p in TinyUNet_21(400, channels=(16, 32, 64), t_dim=64).parameters())
u1_in_21 = (8, 32, 28, 28)


UNET_OK_21 = False
with checked("B5.21"):
    torch.manual_seed(0)
    ok_21 = True
    blk_21, ref_blk = ResBlock_21(16, 32, 24), wbd.ResBlock(16, 32, 24)
    with torch.no_grad():
        for p_ in ref_blk.parameters():
            p_.add_(0.2 * torch.randn(p_.shape))
    if {k: tuple(v.shape) for k, v in blk_21.state_dict().items()} == \
            {k: tuple(v.shape) for k, v in ref_blk.state_dict().items()}:
        blk_21.load_state_dict(ref_blk.state_dict())
        xb_21, tb_21 = torch.randn(2, 16, 6, 6), torch.randn(2, 24)
        with torch.no_grad():
            blk_ok_21 = same("B5.21", "ResBlock_21(16, 32, 24) contre le bloc du contrat, aux mêmes poids",
                             blk_21(xb_21, tb_21), ref_blk(xb_21, tb_21), rtol=1e-5, atol=1e-5)
        if not blk_ok_21:
            print("   piste : l'ordre norm1, SiLU, conv1 ; le biais du pas, time(temb), ajouté par canal entre les deux "
                  "convolutions ; puis norm2, SiLU, conv2 ; et le chemin résiduel skip(x) ajouté à la fin.")
        ok_21 &= blk_ok_21
    else:
        ok_21 = False
        print(f"❌ Ex B5.21 : ResBlock_21(16, 32, 24) doit avoir les paramètres "
              f"{sorted(ref_blk.state_dict())} (noms et formes du contrat) ; reçu {sorted(blk_21.state_dict())}.")
    for kw_21 in ({}, {"channels": (16, 32, 64), "t_dim": 64}, {"n_classes": 10}):
        mine_21, ref_21 = TinyUNet_21(T_STEPS, **kw_21), wbd.TinyUNet(T_STEPS, **kw_21)
        shapes_m = {k: tuple(v.shape) for k, v in mine_21.state_dict().items()}
        shapes_r = {k: tuple(v.shape) for k, v in ref_21.state_dict().items()}
        if shapes_m != shapes_r:
            ok_21 = False
            missing_21 = sorted(set(shapes_r.items()) - set(shapes_m.items()))[:3]
            extra_21 = sorted(set(shapes_m.items()) - set(shapes_r.items()))[:3]
            print(f"❌ Ex B5.21 : TinyUNet_21({T_STEPS}, {kw_21}) : le state_dict n'a pas les noms et les formes du "
                  f"contrat : attendu {missing_21 or 'rien de plus'}, reçu {extra_21 or 'rien de plus'} (les premières "
                  "différences, nom et forme).")
            continue
        pe_21 = getattr(mine_21, "pe", None)
        ok_21 &= same("B5.21", f"le buffer pe ({kw_21 or 'réglage par défaut'}) : la table de positional_encoding",
                      pe_21 if pe_21 is not None else torch.zeros(0), torch.tensor(wbd.sinusoidal_table(
                          T_STEPS, kw_21.get("t_dim", 128)), dtype=torch.float32), rtol=1e-5, atol=1e-6)
        with torch.no_grad():
            for p_ in ref_21.parameters():
                p_.add_(0.05 * torch.randn(p_.shape))                # weights far from the initialisation
        mine_21.load_state_dict(ref_21.state_dict())
        x_21, t_21 = torch.randn(3, 1, 28, 28), torch.tensor([0, T_STEPS // 2, T_STEPS - 1])
        with torch.no_grad():
            if "n_classes" in kw_21:
                y_21 = torch.tensor([3, 10, 7])
                out_ok_21 = same("B5.21", "U-Net conditionné (y donné, dont la classe vide 10) contre le contrat",
                                 mine_21(x_21, t_21, y_21), ref_21(x_21, t_21, y_21), rtol=1e-4, atol=1e-5)
                if not out_ok_21 and ok_21:                  # the unconditional U-Nets above were right
                    print("   piste : class_emb(y) s'ajoute-t-il à t_mlp(pe[t]), avant tous les blocs ?")
                none_ok_21 = same("B5.21", "U-Net conditionné, y = None : la classe vide", mine_21(x_21, t_21),
                                  ref_21(x_21, t_21, torch.full((3,), 10)), rtol=1e-4, atol=1e-5)
                if out_ok_21 and not none_ok_21:
                    print("   piste : sans y, chaque image reçoit la classe vide, la ligne n_classes de class_emb.")
                ok_21 &= out_ok_21 and none_ok_21
            else:
                out_ok_21 = same("B5.21", f"U-Net {kw_21 or '(32, 64, 128)'} contre le contrat, aux mêmes poids",
                                 mine_21(x_21, t_21), ref_21(x_21, t_21), rtol=1e-4, atol=1e-5)
                if not out_ok_21 and ok_21:                  # said once
                    print("   piste : l'encodage du pas temb = t_mlp(pe[t]) donné à chaque bloc, et l'ordre des "
                          "concaténations (la sortie du décodeur d'abord, la carte de l'encodeur ensuite).")
                ok_21 &= out_ok_21
        if not kw_21:
            got_s, want_s = wbd.stage_shapes(mine_21, x_21, t_21), wbd.stage_shapes(ref_21, x_21, t_21)
            ok_21 &= got_s == want_s
            verdict("B5.21", got_s == want_s, "les formes de chaque étage sont celles du contrat.",
                    f"formes des étages : attendu {want_s}, reçu {got_s}.")
    UNET_OK_21 = bool(ok_21)
with wb.attempt("B5.21"):
    wb.check("B5.21a", n_small_21)
with wb.attempt("B5.21"):
    wb.check("B5.21b", shape_text(u1_in_21))

In [ ]:
wb.record("B5.21a", 186113, mistakes={"compte les canaux reçus par u2 et u1 : la sortie du décodeur et celle de l'encodeur": 171889, "relis le contrat de t_mlp : deux couches denses": 181953}, fractional="un nombre de paramètres est un entier")
wb.record("B5.21b", "(8, 32, 28, 28)", mistakes={"c'est la sortie de up1 seule : ajoute les canaux de la connexion de saut": "(8, 16, 28, 28)", "relis la question : on demande l'entrée de u1, à 28 × 28": "(8, 64, 14, 14)", "un batch de 8 images : le premier axe compte les images": "(1, 32, 28, 28)"})

In [ ]:
# The model of part C: your U-Net if it passed its checks, otherwise the U-Net of the contract (wb.diffusion)
UNET_CLASS = TinyUNet_21 if UNET_OK_21 else wbd.TinyUNet
if not UNET_OK_21:
    print("ℹ️ Ex B5.21 : ton U-Net n'est pas encore écrit, ou n'a pas passé ses contrôles ; la suite du chapitre prend "
          "celui du contrat (wbd.TinyUNet). Relance cette cellule une fois le tien prêt.")
net_21 = UNET_CLASS(T_STEPS).eval()
x_21, t_21 = torch.randn(64, 1, 28, 28), torch.randint(0, T_STEPS, (64,))
with torch.no_grad():
    net_21(x_21, t_21)
    times_21 = []
    for _ in range(3):
        start_21 = time.perf_counter()
        net_21(x_21, t_21)
        times_21.append(time.perf_counter() - start_21)
SECONDS_21 = min(times_21)
print(f"{'ton' if UNET_OK_21 else 'le'} U-Net : une passe sur un batch de 64 images en {fr(SECONDS_21, 3)} s (le "
      f"meilleur de trois) ; {T_STEPS} pas de DDPM pour 64 images coûteraient ≈ {T_STEPS * SECONDS_21:.0f} s.")

### Ex B5.22 — Le DDPM qui ne produit que du gris : quatre bugs d'indices, de racines et d'échelle 🐛 ★★ ⏱️ 25 min
**Objectif :** trouver par des diagnostics enchaînés les quatre bugs d'un pipeline DDPM qui ne plante jamais.  
**Prérequis :** Ex B5.18, B5.21 · fiche §205.2 (« La forme fermée »), §205.5 (« L'algorithme », piège « l'échelle et le dernier pas ») · tableau des pièges · **Fil rouge :** MNIST · **Parcours :** R, C

Un collègue a écrit le pipeline d'un DDPM : `prepare_22` prépare les images, `noisy_22` les bruite par la forme fermée (c'est ce que verrait le réseau pendant l'entraînement), `sample_22` échantillonne par l'algorithme 2 de Ho et coll. (variance $\beta_t$). Rien ne plante, mais ses échantillons ne sont que du gris. Quatre bugs s'y cachent, parmi les familles du tableau des pièges de la fiche : un décalage d'indice, $\bar\alpha$ là où il faut $\sqrt{\bar\alpha}$, l'échelle des images, le dernier pas.

Pour voir l'effet du pipeline sans entraîner de réseau, la cellule de l'expérience utilise un **débruiteur parfait** (`wbd.ideal_denoiser`) : pour un dataset fini (ici 500 chiffres de MNIST, `RAW_22`, tels qu'ils sont stockés, des octets de 0 à 255), le bruit qu'un réseau entraîné sans fin sur ces images apprendrait, calculé exactement pour le processus direct que **ton** `noisy_22` définit. Elle montre 8 échantillons de ton pipeline, tel quel puis réparé.

**Ce que tu fais** : répare les trois fonctions **sur place** (mêmes noms, mêmes signatures ; garde la variance $\beta_t$ de l'algorithme 2), puis relance la cellule de vérification. Elle enchaîne quatre diagnostics, dans l'ordre où les bugs se masquent les uns les autres : le premier qui échoue arrête la chaîne et dit ce qu'il a mesuré, jamais ce qu'il faut écrire.
1. **échelle** : les pixels préparés de 64 chiffres vont-ils de −1 à 1 ?
2. **indices** : à chaque indice $t$, la part du bruit dans `noisy_22` est-elle $\sqrt{1 - \bar\alpha_t}$, celle de la forme fermée ?
3. **coefficients** : à chaque indice, les deux coefficients de `noisy_22` préservent-ils la variance ($c_1^2 + c_2^2 = 1$) ?
4. **aller-retour** : partie de `noisy_22` au dernier indice, la chaîne de `sample_22` à qui l'on donne à chaque pas le **vrai** bruit du $\mathbf{x}$ courant retrouve-t-elle les images (🔬 B5.18) ?

Dans tes notes : pour chaque bug, la ligne fautive et ce que tu as changé. Après la seule réparation du point 1, les échantillons du débruiteur parfait ont-ils l'air justes ? Pourquoi un débruiteur parfait cache-t-il les points 2 et 3, et pourquoi un réseau entraîné avec ce pipeline ne les cacherait-il pas aussi bien ? Les 8 images réparées, les reconnais-tu (compare-les au dataset) : qu'est-ce que cela annonce pour 🔬 B5.28 ?

In [ ]:
# 500 MNIST training digits as they are stored (uint8 in [0, 255]), the schedule of the notebook, and the tools of
# the exercise: the four chained diagnoses and a PERFECT denoiser for these 500 images (no network to train)
RAW_22, _ = wbd.raw_subset("mnist", 500)
AB_22 = wbd.alpha_bars(BETAS)


def coefficients_22(noisy_fn):
    """The two coefficients of a forward process x_t = c1[t] x0 + c2[t] eps, read from noisy_fn at every index."""
    T = len(AB_22)
    t = torch.arange(T)
    ones, zeros = torch.ones(T, 1, 28, 28), torch.zeros(T, 1, 28, 28)
    c1 = torch.as_tensor(noisy_fn(ones, t, zeros, AB_22)).reshape(T, -1)[:, 0].double().numpy()
    c2 = torch.as_tensor(noisy_fn(zeros, t, ones, AB_22)).reshape(T, -1)[:, 0].double().numpy()
    return c1, c2


def diagnose_22(prepare_fn, noisy_fn, sample_fn):
    """Four diagnoses of the pipeline, in a row: the first one that fails stops the chain (one bug can hide another).
    Returns True when the four pass."""

    def point1():
        x = torch.as_tensor(prepare_fn(RAW_22[:64]))
        if tuple(x.shape) != (64, 1, 28, 28) or not x.is_floating_point():
            return False, f"prepare_22 rend {tuple(x.shape)} en {x.dtype}", "attendu un tenseur de flottants (64, 1, 28, 28)"
        lo, hi = float(x.min()), float(x.max())
        if abs(lo + 1) < 1e-6 and abs(hi - 1) < 1e-6:
            return True, "les pixels préparés vont de −1 (noir) à 1 (blanc)", ""
        return False, (f"les pixels préparés vont de {fr(lo, 3)} à {fr(hi, 3)} (un pixel noir de MNIST vaut 0, un blanc "
                       "255)"), "sur quel intervalle le bruit et l'écrêtage de x̂₀ supposent-ils les images ?"

    def point2():
        _, c2 = coefficients_22(noisy_fn)
        want = np.sqrt(1 - AB_22)
        bad = np.flatnonzero(np.abs(c2 - want) > 1e-5)
        if not len(bad):
            return True, "à chaque indice, la part du bruit dans x_t est celle de la forme fermée", ""
        i = int(bad[0])
        return False, (f"à l'indice {i}, la part du bruit dans x_t vaut {fr(c2[i], 4)} (forme fermée à cet indice : "
                       f"{fr(want[i], 4)})"), "quel ᾱ ta fonction lit-elle pour un exemple d'indice t ?"

    def point3():
        c1, c2 = coefficients_22(noisy_fn)
        total = c1 ** 2 + c2 ** 2
        bad = np.flatnonzero(np.abs(total - 1) > 1e-5)
        if not len(bad):
            return True, "à chaque indice, c1² + c2² = 1 : la variance d'un pixel de variance 1 est préservée", ""
        i = int(bad[len(bad) // 2])
        return False, (f"à l'indice {i}, x_t = {fr(c1[i], 4)} x₀ + {fr(c2[i], 4)} ε : un pixel de variance 1 et un "
                       f"bruit indépendant de variance 1 donnent une variance {fr(total[i], 4)}, pas 1"), \
            "relis la forme fermée (fiche §205.2) : que vaut le coefficient de x₀ ?"

    def point4():
        x0 = torch.as_tensor(prepare_fn(RAW_22[:8])).float()
        eps = torch.randn(x0.shape, generator=torch.Generator().manual_seed(22))
        last = torch.full((8,), len(AB_22) - 1)
        x_T = torch.as_tensor(noisy_fn(x0, last, eps, AB_22)).float()

        def true_noise(x, t):                     # the noise of the current x, for the image x0 (a perfect network)
            a = float(AB_22[int(t)])
            return (x - math.sqrt(a) * x0) / math.sqrt(1 - a)

        out = torch.as_tensor(sample_fn(true_noise, x_T, AB_22, torch.Generator().manual_seed(23)))
        err = float((out.double() - x0.double()).abs().max())
        if err < 1e-3:
            return True, f"avec le vrai bruit du x courant, la chaîne finit à {fr_e(err)} de x₀", ""
        return False, (f"avec le vrai bruit du x courant à chaque pas, la chaîne finit à {fr(err, 4)} de x₀ (B5.18 : "
                       "elle devrait le retrouver)"), "qu'ajoute ta boucle à chaque pas, et à quels indices ?"

    points = [("échelle des images", point1), ("indices du bruitage", point2), ("coefficients du bruitage", point3),
              ("aller-retour avec le vrai bruit", point4)]
    for k, (name, run) in enumerate(points, start=1):
        try:
            ok, measured, why = run()
        except Exception as error:  # noqa: BLE001 - the learner's code: one ❌ line, then the chain stops
            ok, measured, why = False, f"une erreur {type(error).__name__} : {error}", "relis la fonction en cause"
        if ok:
            print(f"✅ Ex B5.22 : point {k}, {name} ({measured}).")
        else:
            print(f"❌ Ex B5.22 : point {k}, {name} : {measured} ; {why}")
            for later in range(k + 1, len(points) + 1):
                print(f"⏳ Ex B5.22 : point {later} : corrige d'abord le point {k}.")
            return False
    print("✅ Ex B5.22 : les quatre diagnostics passent.")
    return True


def show_pipeline_22(prepare_fn, noisy_fn, sample_fn, title):
    """Eight samples of the pipeline with the perfect denoiser of its own forward process for the 500 digits."""
    try:
        data = torch.as_tensor(prepare_fn(RAW_22)).float()
        c1, c2 = coefficients_22(noisy_fn)
        eps_fn = wbd.ideal_denoiser(data, c1, c2)
        x_T = torch.randn((8, 1, 28, 28), generator=torch.Generator().manual_seed(220))
        out = torch.as_tensor(sample_fn(eps_fn, x_T, AB_22, torch.Generator().manual_seed(221))).float()
    except Exception as error:  # noqa: BLE001 - the learner's code
        print(f"⏳ Ex B5.22 : ton pipeline lève {type(error).__name__} ({error}) : rien à montrer.")
        return None
    show_images(out, ncol=8, size=1.2, suptitle=title)
    print(f"pixels générés : de {fr(float(out.min()), 2)} à {fr(float(out.max()), 2)}, moyenne {fr(float(out.mean()), 2)} "
          f"(images de MNIST dans [−1, 1] : moyenne {fr(float(wbd.to_signed(RAW_22).mean()), 2)})")
    return out

In [ ]:
def prepare_22(raw):
    """uint8 images (N, 28, 28) in [0, 255] -> a float32 tensor (N, 1, 28, 28) in the range of the noise."""
    return torch.as_tensor(np.asarray(raw), dtype=torch.float32)[:, None] / 127.5 - 1      # 1. [-1, 1], like the noise


def noisy_22(x0, t, eps, alpha_bars):
    """x_t of the closed form, for a batch x0 (N, 1, 28, 28), the step index t of each image ((N,) int64, from 0
    to T - 1) and the noise eps (N, 1, 28, 28)."""
    ab = torch.as_tensor(np.asarray(alpha_bars), dtype=torch.float32)[t].view(-1, 1, 1, 1)   # 2. t is already an index
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps                                            # 3. sqrt(alpha_bar)


def sample_22(eps_fn, x, alpha_bars, generator):
    """Ancestral DDPM sampling (Algorithm 2, variance beta_t) from x, the noise at index T - 1, down to the image;
    eps_fn(x, t) is the noise predicted at the index t; the draws z come from generator."""
    ab = np.asarray(alpha_bars, dtype=np.float64)
    betas = 1 - ab / np.concatenate([[1.0], ab[:-1]])
    for t in range(len(ab) - 1, -1, -1):
        eps = eps_fn(x, t)
        x = (x - betas[t] / math.sqrt(1 - ab[t]) * eps) / math.sqrt(1 - betas[t])
        if t > 0:                                                                            # 4. no noise at index 0
            x = x + math.sqrt(betas[t]) * torch.randn(x.shape, generator=generator)
    return x


repaired_22 = False
with checked("B5.22"):
    repaired_22 = diagnose_22(prepare_22, noisy_22, sample_22)

In [ ]:
# Your pipeline, as it is now, with a perfect denoiser for the 500 digits (it shows the symptom before the repair)
samples_22 = show_pipeline_22(prepare_22, noisy_22, sample_22,
                              "ton pipeline " + ("réparé" if repaired_22 else "tel quel") + ", avec un débruiteur parfait")

### Ex B5.23 — Entraîner un DDPM sur MNIST : boucle, moyenne exponentielle des poids (EMA), courbe de loss 📦 ★★★ ⏱️ 60 min 🚀
**Objectif :** écrire la boucle d'entraînement de la loss simplifiée avec sa copie EMA, choisir $\rho$, entraîner ton U-Net et le juger sur des échantillons.  
**Prérequis :** Ex B5.21, B5.22 · ch. 19 (moyenne mobile exponentielle) · fiche §205.3 (« La loss simplifiée », « En pratique : la moyenne exponentielle des poids », piège « évaluer le mauvais réseau ») · **Fil rouge :** MNIST · **Parcours :** R, C

La cellule suivante donne la recette (`SETTINGS_23`, sans le $\rho$ de l'EMA, que tu choisis), la configuration du U-Net (`CONFIG_23`), le sous-ensemble figé de MNIST (`DATA_23`, dans $[-1, 1]$), `AB` ($\bar\alpha$ du planning du notebook), `N_UPDATES_23` (le nombre de pas d'optimisation de la recette), 2 000 chiffres de test et leurs features (`REAL_MNIST`, `REAL_FEATS`), `frechet_mnist(images)` (la distance de Fréchet de 256 images ou plus à ces chiffres, avec ta `frechet_distance_from_features` du ch. 27) et `X_FIXED`, 16 bruits de départ fixés. En mode rapide, le U-Net a les canaux (16, 32, 64) : celui de la fiche ferait quatre fois moins de pas dans le même temps. 🚀 En mode complet, le U-Net de la fiche, 20 epochs sur les 60 000 images, $T = 1\,000$ : plusieurs heures sur CPU (une estimation : ce mode n'a pas été exécuté pour ce workbook), un GPU de Colab est conseillé.

**Contrat**
- `RHO_23` : le $\rho$ de ta copie EMA, $\theta_{\text{EMA}} \leftarrow \rho\,\theta_{\text{EMA}} + (1 - \rho)\,\theta$ après chaque pas. La fiche cite $0{,}9999$ pour 800 000 pas ; choisis-le pour les `N_UPDATES_23` pas de la recette. La vérification demande qu'à la fin il reste au plus 1 % des poids de départ dans la copie ($\rho^{N} \le 0{,}01$) et qu'elle moyenne au moins 20 pas ($1/(1 - \rho) \ge 20$).
- `ema_update_23(ema, model, rho)` : la formule, pour chaque paramètre de `ema`, **sur place** (les mêmes objets `Parameter`) et sans graphe ; les buffers de `model` sont recopiés dans ceux de `ema`.
- `train_23(model, ema, images, settings, generator, alpha_bars)` entraîne `model` et sa copie `ema` sur place et rend `{"loss": [...]}`, la loss de chaque pas (des floats Python) :
  - `torch.optim.AdamW(model.parameters(), lr=settings["lr"])`, ses autres réglages par défaut ;
  - pour chacune des `settings["epochs"]` epochs, l'ordre `torch.randperm(N, generator=generator)`, découpé en batches de `settings["batch_size"]` (le dernier peut être plus petit) ;
  - pour chaque batch, **dans cet ordre**, deux tirages par `generator` : les indices des pas `torch.randint(0, T, (b,))`, puis le bruit `torch.randn(x0.shape)` ; $\mathbf{x}_t$ par la forme fermée en float32 ; la loss `F.mse_loss(model(x_t, t), eps)` ; les gradients ; un pas de l'optimiseur ; puis `ema_update_23(ema, model, settings["ema"])` ;
  - à la fin, les deux modèles en mode eval.

La cellule de vérification compare `ema_update_23` à la formule, puis 4 pas de `train_23` (2 epochs de 6 images en batches de 4) à la recette du contrat, sur un petit U-Net du contrat : les loss, les poids, la copie EMA ; et elle contrôle ton $\rho$. La cellule suivante entraîne ton U-Net (`UNET_CLASS`, environ 2 min sur un CPU à deux cœurs en mode rapide) et le **sauvegarde** une fois, avec sa copie EMA et sa loss, dans `wbd.checkpoints_dir()` (hors du suivi de git ; sur Colab, `/content/wb_cache/diffusion/`, effacé avec la session), sous une clé qui dit tout de l'entraînement (la configuration, la recette, ton $\rho$, le mode, les images) ; un nouvel entraînement efface d'abord l'ancienne sauvegarde. La dernière **juge** l'entraînement comme la fiche le demande, sur des échantillons et non sur la loss : la courbe de loss, 16 échantillons du même bruit `X_FIXED` avec la copie EMA puis avec les poids bruts (DDIM, 50 pas, $\hat{\mathbf{x}}_0$ écrêté), et la distance de Fréchet de 256 échantillons de la copie EMA, qui doit passer sous `GOAL_23` (la pire graine de la référence plus 30 %). Le temps affiché est une mesure, pas une réponse.

Dans tes notes : pourquoi ce $\rho$ ? Que vaudrait $0{,}9999^{N}$ ici, et que généreraient ces poids ? La loss descend-elle encore à la fin, et pourquoi ne suffit-elle pas à juger le modèle (fiche, piège « juger un entraînement à sa loss ») ? Lequel des deux réseaux génère le mieux, et pourquoi ? Les 16 échantillons ressemblent-ils à des chiffres ? Combien d'entre eux sont lisibles ?

In [ ]:
# The recipe of the training (wb.diffusion.unet_settings), without its EMA decay: you choose yours (RHO_23)
SETTINGS_23 = {k: v for k, v in wbd.unet_settings(FAST_MODE).items() if k != "ema"}
CONFIG_23 = {"n_steps": T_STEPS, "channels": list(SETTINGS_23["channels"]), "t_dim": SETTINGS_23["t_dim"],
             "n_classes": None}                        # the configuration of the U-Net (UNET_CLASS(**CONFIG_23))
DATA_23 = wbd.image_subset("mnist", SETTINGS_23["n_train"])   # the frozen training subset, in [-1, 1]
AB = wbd.alpha_bars(BETAS)                             # alpha_bar of the schedule of the notebook, indices 0 .. T - 1
N_UPDATES_23 = wbd.n_updates(len(DATA_23), SETTINGS_23)
REAL_MNIST = wbd.image_subset("mnist", 2000, split="test")    # 2 000 test digits: the reference of the distances
REAL_FEATS = wbd.features(CLF, REAL_MNIST.images)
X_FIXED = torch.randn((16, 1, 28, 28), generator=torch.Generator().manual_seed(2323))   # the fixed starting noise


def key_23(rho, source="learner"):
    """The key of the checkpoint of B5.23: the configuration, the recipe with your rho, the mode, who trained it."""
    return wbd.checkpoint_key(CONFIG_23, SETTINGS_23 | {"ema": float(rho)}, fast=FAST_MODE, source=source,
                              data=DATA_23)


def frechet_mnist(images):
    """The Fréchet distance between the features (CLF, 64 values) of images in [-1, 1] and those of REAL_FEATS, with
    your frechet_distance_from_features of ch. 27 (the reference when your gen_metrics.py does not exist)."""
    feats = wbd.features(CLF, torch.as_tensor(images).clamp(-1, 1))
    return float(mylearn.gen_metrics.frechet_distance_from_features(REAL_FEATS, feats))


print(f"recette : {SETTINGS_23['epochs']} epochs sur {len(DATA_23)} images, batches de {SETTINGS_23['batch_size']}, "
      f"AdamW au learning rate {SETTINGS_23['lr']} : {N_UPDATES_23} pas d'optimisation ; U-Net de canaux "
      f"{tuple(SETTINGS_23['channels'])} ({wbd.count_params(wbd.TinyUNet(**CONFIG_23)):_} paramètres), T = {T_STEPS}."
      .replace("_", " "))
FLOOR_23 = 11.4    # two sets of 256 real digits (history_chB5.py)
REF_23 = (78.5, 101.9)   # the reference (rho = 0.99): best and worst of 3 seeds
GOAL_23 = 135.0     # the distance below which the check accepts the EMA copy

In [ ]:
CKPT_SOURCE_23 = "solution"   # the solutions notebook saves its models apart: it never overwrites, nor reloads, the learner's checkpoints

RHO_23 = 0.99 if FAST_MODE else 0.999   # 1 252 steps: 0.99 ** 1252 ≈ 3e-6 of the initial weights left, a memory of
                                         # about 100 steps (8 % of the training); full mode, 9 375 steps: 0.999


def ema_update_23(ema, model, rho):
    """theta_ema <- rho * theta_ema + (1 - rho) * theta for each parameter of ema, in place and without a graph; the
    buffers of model are copied into those of ema."""
    with torch.no_grad():
        for p_ema, p in zip(ema.parameters(), model.parameters()):
            p_ema.mul_(rho).add_(p, alpha=1 - rho)
        for b_ema, b in zip(ema.buffers(), model.buffers()):
            b_ema.copy_(b)


def train_23(model, ema, images, settings, generator, alpha_bars):
    """Train model, and its EMA copy ema, in place on images (float32 (N, 1, 28, 28) in [-1, 1]) with the recipe of
    the contract; returns {"loss": the list of the losses of every step (Python floats)}."""
    ab = torch.tensor(np.asarray(alpha_bars), dtype=torch.float32)
    opt = torch.optim.AdamW(model.parameters(), lr=settings["lr"])
    losses, bs = [], settings["batch_size"]
    model.train()
    for _ in range(settings["epochs"]):
        order = torch.randperm(len(images), generator=generator)
        for i in range(0, len(images), bs):
            x0 = images[order[i:i + bs]]
            t = torch.randint(0, len(ab), (len(x0),), generator=generator)
            eps = torch.randn(x0.shape, generator=generator)
            a = ab[t].view(-1, 1, 1, 1)
            loss = F.mse_loss(model(a.sqrt() * x0 + (1 - a).sqrt() * eps, t), eps)
            opt.zero_grad()
            loss.backward()
            opt.step()
            ema_update_23(ema, model, settings["ema"])
            losses.append(loss.item())
    model.eval()
    ema.eval()
    return {"loss": losses}


probes_ok_23 = False
with checked("B5.23"):
    torch.manual_seed(0)
    small_23 = wbd.TinyUNet(20, channels=(8, 8, 16), t_dim=16)            # a small U-Net of the contract
    with torch.no_grad():
        for p_ in small_23.parameters():
            p_.add_(0.1 * torch.randn(p_.shape))                            # weights far from the initialisation
    # 1. ema_update_23 against the formula, on two copies
    ema_a, target_23 = copy.deepcopy(small_23), copy.deepcopy(small_23)
    with torch.no_grad():
        for p_ in target_23.parameters():
            p_.add_(torch.randn(p_.shape))
    ids_23 = [id(p_) for p_ in ema_a.parameters()]
    ema_b = copy.deepcopy(ema_a)
    ema_update_23(ema_a, target_23, 0.75)
    wbd.ema_update(ema_b, target_23, 0.75)
    ema_ok_23 = all(torch.allclose(a, b, rtol=1e-6, atol=1e-7) for a, b in zip(ema_a.parameters(), ema_b.parameters()))
    inplace_23 = [id(p_) for p_ in ema_a.parameters()] == ids_23 and all(p_.grad is None for p_ in ema_a.parameters())
    verdict("B5.23", ema_ok_23 and inplace_23, "ema_update_23 : 0,75 θ_EMA + 0,25 θ, sur place et sans gradient.",
            "ema_update_23(ema, model, 0.75) doit rendre chaque paramètre de ema égal à 0,75 × lui-même + 0,25 × celui de "
            "model, sur place (les mêmes objets Parameter) et sans gradient : " +
            ("les valeurs diffèrent." if not ema_ok_23 else "les paramètres ont été remplacés ou ont un gradient."))
    # 2. train_23 against the recipe of the contract: 6 digits in batches of 4 (a last batch of 2), 2 epochs
    probe_23 = {"epochs": 2, "batch_size": 4, "lr": 1e-3, "ema": 0.5}
    ab20_23 = wbd.alpha_bars(np.linspace(1e-3, 0.2, 20))   # a short schedule
    m1, e1, m2, e2 = (copy.deepcopy(small_23) for _ in range(4))
    hist_23 = train_23(m1, e1, DATA_23.images[:6].clone(), probe_23, torch.Generator().manual_seed(5), ab20_23)
    want_23 = wbd.train_unet(m2, e2, DATA_23.images[:6].clone(), probe_23, torch.Generator().manual_seed(5), ab20_23)
    if not returned("B5.23", "train_23", hist_23):
        raise StopCheck
    losses_23 = hist_23.get("loss") if isinstance(hist_23, dict) else None
    if not (isinstance(losses_23, list) and len(losses_23) == 4 and all(isinstance(v, float) for v in losses_23)):
        print(f"❌ Ex B5.23 : train_23 doit rendre {{'loss': [...]}}, une loss (float Python) par pas : 2 epochs de 6 "
              f"images en batches de 4 font 4 pas ; reçu {hist_23!r:.120}.")
        raise StopCheck
    loss_ok_23 = np.allclose(losses_23, want_23["loss"], rtol=1e-4, atol=1e-6)
    weights_ok_23 = all(torch.allclose(a, b, rtol=1e-4, atol=1e-6) for a, b in zip(m1.parameters(), m2.parameters()))
    emaw_ok_23 = all(torch.allclose(a, b, rtol=1e-4, atol=1e-6) for a, b in zip(e1.parameters(), e2.parameters()))
    if loss_ok_23 and weights_ok_23 and emaw_ok_23:
        print("✅ Ex B5.23 : 4 pas de train_23 (2 epochs, un dernier batch de 2 images) : les loss, les poids et la copie "
              "EMA de la recette du contrat.")
    elif not math.isclose(losses_23[0], want_23["loss"][0], rel_tol=1e-4):
        print(f"❌ Ex B5.23 : la loss du 1ᵉʳ pas, aux mêmes poids, vaut {fr(losses_23[0], 5)} au lieu de "
              f"{fr(want_23['loss'][0], 5)} : vérifie l'ordre des tirages (l'ordre des images, les pas, le bruit), la "
              "forme fermée et la cible.")
    elif not loss_ok_23:
        print("❌ Ex B5.23 : la 1ʳᵉ loss est juste, pas les suivantes : vérifie l'optimiseur (AdamW, son learning rate), "
              "les batches d'une epoch (le dernier fait 2 images) et l'ordre des images à chaque epoch.")
    elif not weights_ok_23:
        print("❌ Ex B5.23 : les loss sont justes, mais pas les poids après le dernier pas : un pas d'optimisation manque-t-il ?")
    else:
        print("❌ Ex B5.23 : les poids entraînés sont justes, pas la copie EMA : est-elle mise à jour après chaque pas, avec "
              "settings['ema'] ?")
    # 3. your decay rho
    rho_ok_23 = False
    if not filled(RHO_23):
        print("⏳ Ex B5.23 : choisis RHO_23.")
    elif not (isinstance(RHO_23, (int, float)) and 0 < RHO_23 < 1):
        print(f"❌ Ex B5.23 : RHO_23 est un nombre strictement entre 0 et 1 ; reçu {RHO_23!r}.")
    else:
        left_23, memory_23 = RHO_23 ** N_UPDATES_23, 1 / (1 - RHO_23)
        rho_ok_23 = left_23 <= 0.01 and memory_23 >= 20
        verdict("B5.23", rho_ok_23,
                f"ρ = {RHO_23} : après {N_UPDATES_23} pas, il reste {fr(100 * left_23, 4)} % des poids de départ dans la "
                f"copie EMA, qui moyenne environ {memory_23:.0f} pas.",
                f"ρ = {RHO_23} : après {N_UPDATES_23} pas, il resterait {fr(100 * left_23, 2)} % des poids de départ dans la "
                f"copie EMA (au plus 1 %), et elle moyennerait environ {memory_23:.0f} pas (au moins 20).")
    probes_ok_23 = ema_ok_23 and inplace_23 and loss_ok_23 and weights_ok_23 and emaw_ok_23 and rho_ok_23

In [ ]:
# Train your U-Net (UNET_CLASS) on DATA_23 and save it ONCE, with its EMA copy and its history (about 2 min on CPU in
# fast mode; 🚀 in full mode)
model_23 = ema_23 = history_23 = None
if not globals().get("probes_ok_23"):
    print("⏳ Ex B5.23 : l'entraînement attend les contrôles de la cellule précédente (et RHO_23).")
else:
    with checked("B5.23"):
        KEY_23 = key_23(RHO_23, globals().get("CKPT_SOURCE_23", "learner"))
        if wbd.delete_checkpoint(KEY_23):                    # a new training: the old model goes, even if this run fails
            print("ℹ️ l'ancienne sauvegarde de ton U-Net est effacée : la suite du chapitre relira celui-ci.")
        torch.manual_seed(SETTINGS_23["seed"])
        net_23 = UNET_CLASS(**CONFIG_23)
        ema_net_23 = copy.deepcopy(net_23)
        started_23 = time.perf_counter()
        hist_23 = train_23(net_23, ema_net_23, DATA_23.images, SETTINGS_23 | {"ema": float(RHO_23)},
                           torch.Generator().manual_seed(SETTINGS_23["seed"] + 1), AB)
        seconds_23 = time.perf_counter() - started_23
        if returned("B5.23", "train_23", hist_23):
            model_23, ema_23, history_23 = net_23, ema_net_23, hist_23
            path_23 = wbd.save_checkpoint(KEY_23, model_23, ema=ema_23, history=history_23,
                                          extra={"seconds": seconds_23})
            print(f"{len(history_23['loss'])} pas en {seconds_23:.0f} s ; sauvegardé : {path_23.name}, dans "
                  "wbd.checkpoints_dir()")

In [ ]:
# Judge the training on samples from a FIXED noise (the same X_FIXED every time), not on the loss
if model_23 is None:
    print("⏳ Ex B5.23 : le jugement attend l'entraînement de la cellule précédente.")
else:
    with checked("B5.23"):
        ckpt_23 = wbd.load_checkpoint(KEY_23)
        verdict("B5.23", ckpt_23 is not None, f"la sauvegarde se relit (sauvegardée le {ckpt_23['saved_at'] if ckpt_23 else ''}).",
                "la sauvegarde ne se relit pas.")
        losses_23 = np.array(history_23["loss"])
        fig, ax = plt.subplots(figsize=(7, 3))
        ax.plot(losses_23, lw=0.5, alpha=0.4, label="loss de chaque pas")
        ax.plot(np.arange(99, len(losses_23)), np.convolve(losses_23, np.ones(100) / 100, mode="valid"), lw=2,
                label="moyenne glissante sur 100 pas")
        ax.set_yscale("log")
        ax.set_xlabel("pas d'optimisation")
        ax.set_ylabel("MSE sur le bruit")
        ax.legend()
        plt.show()
        print(f"loss moyenne des 100 premiers pas {fr(losses_23[:100].mean(), 3)}, des 100 derniers {fr(losses_23[-100:].mean(), 3)}")
        for name_23, net in (("copie EMA", ema_23), ("poids bruts", model_23)):
            show_images(wbd.sample_ddim(net, X_FIXED, AB, 50), ncol=16, size=0.75,
                        suptitle=f"{name_23} : 16 échantillons (DDIM, 50 pas) du même bruit X_FIXED")
        x256_23 = torch.randn((256, 1, 28, 28), generator=torch.Generator().manual_seed(2324))
        FD_23 = frechet_mnist(wbd.sample_ddim(ema_23, x256_23, AB, 50))
        print(f"distance de Fréchet de 256 échantillons de la copie EMA (DDIM, 50 pas) aux 2 000 chiffres de test : "
              f"{fr(FD_23, 1)} (deux échantillons de 256 vrais chiffres : environ {fr(FLOOR_23, 0)})")
        verdict("B5.23", FD_23 < GOAL_23,
                f"la copie EMA génère des formes de chiffres : distance {fr(FD_23, 1)}, sous {GOAL_23:.0f} (la référence : de "
                f"{fr(REF_23[0], 0)} à {fr(REF_23[1], 0)} selon la graine).",
                f"distance {fr(FD_23, 1)} : attendu moins de {GOAL_23:.0f} (référence de {fr(REF_23[0], 0)} à "
                f"{fr(REF_23[1], 0)}) ; regarde les échantillons, puis relance les contrôles de la cellule de vérification.")

### Ex B5.24 — Échantillonner : DDPM à 1 000 pas contre DDIM à 50 pas, qualité et temps 📦 ★★ ⏱️ 30 min
**Objectif :** écrire les deux échantillonneurs avec ta librairie, et comparer leurs images et leur coût sur ton modèle.  
**Prérequis :** Ex B5.17, B5.23 · fiche §205.5 (« L'algorithme », « Ce que coûte un échantillon »), §205.6 (« L'idée de DDIM », « Pourquoi c'est plus rapide ») · **Fil rouge :** MNIST · **Parcours :** R, C

La première cellule relit ton modèle : la **copie EMA** de ton U-Net de B5.23, `MODEL`, depuis sa sauvegarde (elle n'entraîne rien). S'il manque, elle le dit ; `REFERENCE_FALLBACK = True` dans cette cellule prend le U-Net de référence de `wb.diffusion` (entraîné une fois avec la recette de B5.23, environ 2 min). Elle donne aussi `need_model` et `PassCounter`, qui compte les passes d'un réseau. En mode rapide, la chaîne a $T = 400$ pas (1 000 en mode complet) : « DDPM à 1 000 pas » devient DDPM à 400 pas.

**Contrat** (le réseau est appelé `model(x, torch.full((N,), t))` en float32, sans graphe ; entre deux pas, `x` est un tableau float64 de NumPy ; les deux fonctions rendent un tenseur float32 `(N, 1, 28, 28)` et ne modifient pas `x_T`) :
- `sample_ddpm_24(model, x_T, schedule, rng, clip_x0=1.0)` : de l'indice $T - 1$ à 0, une passe du réseau puis **ta** `ddpm_step(x, t, eps, schedule, rng=rng, clip_x0=clip_x0)` (B5.17 ; `schedule` = ta `noise_schedule(BETAS)`) : $T$ passes ;
- `sample_ddim_24(model, x_T, alpha_bars, n_steps, clip_x0=1.0)` : sur la sous-séquence `wbd.ddim_timesteps(T, n_steps)` puis vers −1, à chaque pas une passe du réseau, $\hat{\mathbf{x}}_0$ par **ta** `predict_x0_from_eps`, écrêté à $[-1, 1]$ (si `clip_x0` n'est pas `None`), le bruit **recalculé** à partir de $\hat{\mathbf{x}}_0$ écrêté, $\boldsymbol{\varepsilon} = (\mathbf{x} - \sqrt{\bar\alpha_t}\,\hat{\mathbf{x}}_0)/\sqrt{1 - \bar\alpha_t}$, puis **ta** `ddim_step` avec ce bruit : `n_steps` passes.

La cellule de vérification compare tes deux fonctions aux échantillonneurs du contrat (`wbd.sample_ddim`, `wbd.sample_ddpm`) sur 4 images et le même bruit de départ, et compte les passes du réseau. La suivante échantillonne 32 images d'un même bruit de départ par les deux méthodes (DDPM est limité à 32 images : ses $T$ passes coûtent cher), les montre, et mesure le temps, la confiance moyenne du classifieur `CLF` (la probabilité de sa classe préférée) et la part d'images qu'il reconnaît avec une probabilité de plus de 90 % (environ 15 s en mode rapide). Une distance de Fréchet sur si peu d'images ne voudrait rien dire (fiche §205.8) : 🔬 B5.25 la mesure sur plus d'images. Les durées sont des mesures, pas des réponses.

Dans tes notes : combien de fois DDIM est-il plus rapide ici, et pourquoi ce rapport est-il proche du rapport des nombres de passes ? À qualité voisine, lequel choisirais-tu, et dans quel cas l'autre ? Partis du même bruit, les deux échantillonneurs donnent-ils les mêmes chiffres ? Pourquoi DDIM rend-il toujours les mêmes images pour un même bruit, et DDPM pas ? Que se passe-t-il, sur quelques images, si tu passes `clip_x0=None` à DDIM ?

In [ ]:
# The model of the rest of part C: your U-Net trained by B5.23, read from its checkpoint (never retrained here)
REFERENCE_FALLBACK = False   # True: if your model of B5.23 is missing, use the reference U-Net of wb.diffusion (trained
                             # once with the recipe of B5.23, about 2 min on CPU in fast mode, then reloaded)
MODEL = CKPT = None
if isinstance(globals().get("RHO_23"), (int, float)) and 0 < RHO_23 < 1:
    CKPT = wbd.load_checkpoint(key_23(RHO_23, globals().get("CKPT_SOURCE_23", "learner")))
if CKPT is not None:
    try:
        MODEL = wbd.build_from_checkpoint(UNET_CLASS, CKPT, which="ema")
        where_c2 = ""
    except Exception:  # noqa: BLE001 - UNET_CLASS not the contract in this session (a restarted kernel)
        MODEL = wbd.build_from_checkpoint(wbd.TinyUNet, CKPT, which="ema")
        where_c2 = " dans la classe TinyUNet de wb.diffusion (mêmes noms, mêmes poids)"
    print(f"ℹ️ la copie EMA de ton U-Net de B5.23, sauvegardée le {CKPT['saved_at']}, relue de {CKPT['path'].name}"
          f"{where_c2}.")
elif REFERENCE_FALLBACK:
    MODEL, CKPT = wbd.reference_model(FAST_MODE)
    print(f"ℹ️ ton U-Net de B5.23 manque : la copie EMA du U-Net de RÉFÉRENCE (sauvegardé le {CKPT['saved_at']}) le "
          "remplace dans la suite de la partie C.")
else:
    print("⏳ aucune sauvegarde de ton U-Net de B5.23 (" + ("mode rapide" if FAST_MODE else "mode complet") + ") : "
          "exécute les cellules de B5.23 (RHO_23 compris), ou mets REFERENCE_FALLBACK = True dans cette cellule pour "
          "prendre le U-Net de référence (2 min la première fois). Sur Colab, la sauvegarde s'efface avec la session.")


def need_model(ex_id):
    """Stop a check cell with ⏳ while no trained U-Net is loaded."""
    if MODEL is None:
        print(f"⏳ Ex {ex_id} : cet exercice attend ton U-Net entraîné (B5.23), ou REFERENCE_FALLBACK = True dans la "
              "cellule des outils.")
        raise StopCheck


class PassCounter:
    """Count the passes of a network: every call of ``model`` (the batch size of each call) while active."""

    def __init__(self, model):
        self.model, self.calls, self.images = model, 0, 0

    def __enter__(self):
        def hook(module, args):
            self.calls += 1
            self.images += int(args[0].shape[0])
        self.handle = self.model.register_forward_pre_hook(hook)
        return self

    def __exit__(self, *exc):
        self.handle.remove()
        return False

In [ ]:
def _eps_24(model, x, t):
    """The network's noise at the index t for the float64 array x, as a float64 array (float32 inside, no graph)."""
    with torch.no_grad():
        out = model(torch.as_tensor(x, dtype=torch.float32), torch.full((len(x),), t, dtype=torch.long))
    return out.double().numpy()


def sample_ddpm_24(model, x_T, schedule, rng, clip_x0=1.0):
    """Ancestral DDPM sampling with YOUR ddpm_step, from x_T (a tensor (N, 1, 28, 28), the noise at index T - 1) down
    to the images; a float32 tensor (N, 1, 28, 28)."""
    x = x_T.double().numpy()
    for t in range(len(schedule["betas"]) - 1, -1, -1):
        x = mylearn.diffusion.ddpm_step(x, t, _eps_24(model, x, t), schedule, rng=rng, clip_x0=clip_x0)
    return torch.as_tensor(x, dtype=torch.float32)


def sample_ddim_24(model, x_T, alpha_bars, n_steps, clip_x0=1.0):
    """Deterministic DDIM with YOUR ddim_step on wbd.ddim_timesteps(T, n_steps), then to -1, x̂0 clipped and the
    noise recomputed from it; a float32 tensor (N, 1, 28, 28)."""
    D = mylearn.diffusion
    x = x_T.double().numpy()
    steps = [int(t) for t in wbd.ddim_timesteps(len(alpha_bars), n_steps)] + [-1]
    for t, t_prev in zip(steps[:-1], steps[1:]):
        eps = _eps_24(model, x, t)
        if clip_x0 is not None:
            x0 = np.clip(D.predict_x0_from_eps(x, np.full(len(x), t), eps, alpha_bars), -clip_x0, clip_x0)
            eps = (x - math.sqrt(alpha_bars[t]) * x0) / math.sqrt(1 - alpha_bars[t])
        x = D.ddim_step(x, t, t_prev, eps, alpha_bars)
    return torch.as_tensor(x, dtype=torch.float32)


samplers_ok_24 = False
with checked("B5.24"):
    need_model("B5.24")
    SCHED = mylearn.diffusion.noise_schedule(BETAS)          # your noise_schedule (B5.12)
    x4_24 = torch.randn((4, 1, 28, 28), generator=torch.Generator().manual_seed(24))
    copy4_24 = x4_24.clone()
    ok_24 = True
    for label_24, call_24, want_24, passes_24 in (
            ("sample_ddim_24, 10 pas", lambda: sample_ddim_24(MODEL, x4_24, AB, 10),
             lambda: wbd.sample_ddim(MODEL, x4_24, AB, 10), 10),
            ("sample_ddim_24, 3 pas, sans écrêtage", lambda: sample_ddim_24(MODEL, x4_24, AB, 3, clip_x0=None),
             lambda: wbd.sample_ddim(MODEL, x4_24, AB, 3, clip_x0=None), 3),
            (f"sample_ddpm_24, {T_STEPS} pas", lambda: sample_ddpm_24(MODEL, x4_24, SCHED, np.random.default_rng(3)),
             lambda: wbd.sample_ddpm(MODEL, x4_24, AB, np.random.default_rng(3)), T_STEPS)):
        with PassCounter(MODEL) as count_24:
            got_24 = call_24()
        if not returned("B5.24", label_24.split(",")[0], got_24):
            ok_24 = False
            continue
        if not (torch.is_tensor(got_24) and got_24.dtype == torch.float32 and tuple(got_24.shape) == (4, 1, 28, 28)):
            print(f"❌ Ex B5.24 : {label_24} doit rendre un tenseur float32 (4, 1, 28, 28) ; reçu {type(got_24).__name__} "
                  f"{getattr(got_24, 'dtype', '')} {tuple(getattr(got_24, 'shape', ()))}.")
            ok_24 = False
            continue
        ok_24 &= same("B5.24", f"{label_24} contre l'échantillonneur du contrat (même bruit de départ)", got_24, want_24(),
                      rtol=1e-4, atol=1e-4)
        ok_24 &= count_24.calls == passes_24
        verdict("B5.24", count_24.calls == passes_24, f"{label_24} : {count_24.calls} passes du réseau.",
                f"{label_24} : {count_24.calls} passes du réseau, au lieu de {passes_24}.")
    verdict("B5.24", torch.equal(x4_24, copy4_24), "le bruit de départ x_T n'a pas changé.",
            "tes échantillonneurs ont modifié x_T, le bruit de départ.")
    samplers_ok_24 = bool(ok_24) and torch.equal(x4_24, copy4_24)

In [ ]:
# The same 32 starting noises, sampled by DDPM (all the T steps) and by DDIM (50 steps): images and time
if not samplers_ok_24:
    print("⏳ Ex B5.24 : la comparaison attend tes deux échantillonneurs (cellule précédente sans ❌).")
else:
    with checked("B5.24"):
        x32_24 = torch.randn((32, 1, 28, 28), generator=torch.Generator().manual_seed(240))
        RESULTS_24 = {}
        for name_24, run_24 in ((f"DDPM, {T_STEPS} pas", lambda: sample_ddpm_24(MODEL, x32_24, SCHED, np.random.default_rng(241))),
                                ("DDIM, 50 pas", lambda: sample_ddim_24(MODEL, x32_24, AB, 50))):
            start_24 = time.perf_counter()
            out_24 = run_24()
            seconds_24 = time.perf_counter() - start_24
            with torch.no_grad():
                proba_24 = torch.softmax(CLF(out_24.clamp(-1, 1)), dim=1).max(dim=1).values
            RESULTS_24[name_24] = {"images": out_24, "seconds": seconds_24, "confidence": float(proba_24.mean()),
                                   "sure": float((proba_24 > 0.9).float().mean())}
            show_images(out_24[:16], ncol=16, size=0.75, suptitle=f"{name_24} : les 16 premiers des 32 échantillons")
        print(f"{'échantillonneur':<16} {'secondes':>9} {'s / image':>10} {'confiance moyenne':>18} {'reconnus à 90 %':>16}")
        for name_24, r in RESULTS_24.items():
            print(f"{name_24:<16} {fr(r['seconds'], 1):>9} {fr(r['seconds'] / 32, 3):>10} {fr(r['confidence'], 3):>18} "
                  f"{fr(100 * r['sure'], 0) + ' %':>16}")
        print("(des mesures, pas des réponses : les durées dépendent de la machine ; la confiance est la probabilité que "
              "CLF donne à sa classe préférée, en moyenne sur les 32 images)")

### Ex B5.25 — Nombre de pas DDIM, planning linéaire ou cosinus : distance de Fréchet contre temps de calcul 🔬 ★★★ ⏱️ 45 min 🚀
**Objectif :** mesurer ce que coûte et ce que rapporte chaque pas de DDIM, et comparer deux plannings à nombre de pas égal.  
**Prérequis :** Ex B5.24 · ✏️ B5.3 · ch. 27 (distance de Fréchet) · fiche §205.6 (« Pourquoi c'est plus rapide », piège « une sous-séquence qui ne part pas du dernier indice »), §205.8 (« La distance de Fréchet ») · **Fil rouge :** MNIST · **Parcours :** C

Combien de pas DDIM faut-il à ton modèle, et le planning cosinus, qui garde l'image lisible plus longtemps (✏️ B5.3), en demande-t-il moins ? La cellule suivante affiche les sous-séquences de 5 et 10 pas (`wbd.ddim_timesteps`, de l'indice $T - 1$ à 0 : jamais l'espacement « leading ») et donne `STEPS_25`, `N_25` et `FULL_25`, le **protocole complet** précalculé : la recette de B5.23 avec le planning linéaire du notebook (L) et avec le planning cosinus (C), sur plusieurs graines, et la distance de Fréchet de 256 échantillons pour chaque nombre de pas de `STEPS_25`.

**Contrat**
- `fd_curve_25(model, alpha_bars, steps_list, x_T)` : pour chaque nombre de pas $n$ de `steps_list`, le couple (`frechet_mnist` des images de ta `sample_ddim_24(model, x_T, alpha_bars, n)`, la durée de cet échantillonnage en secondes) ; une liste de couples de floats.

Puis, d'après le tableau `FULL_25`, **pour toutes ses graines** :
- a) `enough_25` : le plus petit nombre de pas de `STEPS_25` à partir duquel la distance du planning linéaire reste au plus 1,2 fois sa distance à 100 pas ;
- b) `better5_25` : à 5 pas, quel planning a la plus petite distance, graine par graine, `"L"` ou `"C"` ?

La cellule de vérification compare `fd_curve_25` à DDIM du contrat sur 80 images et 2 et 4 pas, puis vérifie tes deux réponses. La suivante trace la courbe distance contre secondes de **ton** modèle (`N_25` images, environ 20 s en mode rapide), et, à côté, les bandes du protocole complet pour les deux plannings. 🚀 En mode complet, elle entraîne aussi un modèle au planning cosinus avec ta `train_23` (aussi long que B5.23) et trace sa courbe.

Dans tes notes : à partir de combien de pas la courbe de ton modèle se met-elle à plat, et combien de secondes coûte chaque point ? Pourquoi la distance remonte-t-elle parfois un peu quand on ajoute des pas ? Pourquoi les distances de ton modèle ne se comparent-elles pas directement à celles du protocole complet (nombre d'images) ? Le planning cosinus rend-il le petit nombre de pas moins coûteux en qualité ? Relie la réponse à la répartition des pas « utiles » de ✏️ B5.3 et 📈 B5.15.

In [ ]:
STEPS_25 = (5, 10, 20, 50, 100)                       # numbers of DDIM steps of the curve
N_25 = 128 if FAST_MODE else 1000                       # images per point (the same starting noise for every point)
for n_25 in (5, 10):
    print(f"sous-séquence de {n_25} pas, de l'indice {T_STEPS - 1} à 0 : {wbd.ddim_timesteps(T_STEPS, n_25).tolist()}")
# The full protocol (tools/chapters/history_chB5.py): the recipe of B5.23 with the linear schedule (L, the notebook's)
# and with the cosine schedule (C), 3 seeds each, the distance of 256 DDIM samples for each number of steps
FULL_25 = {
    "L": [[223.2, 133.0, 102.1, 92.5, 90.1], [169.6, 98.9, 84.5, 78.5, 76.7], [206.0, 124.1, 108.7, 101.9, 99.9]],
    "C": [[164.7, 116.9, 101.6, 93.8, 91.6], [146.4, 113.4, 100.6, 93.7, 94.4], [149.6, 114.9, 106.5, 104.8, 104.9]],
}

In [ ]:
def fd_curve_25(model, alpha_bars, steps_list, x_T):
    """For each number of steps n of steps_list: (frechet_mnist of sample_ddim_24(model, x_T, alpha_bars, n), the
    seconds of the sampling); a list of pairs of floats."""
    out = []
    for n in steps_list:
        start = time.perf_counter()
        images = sample_ddim_24(model, x_T, alpha_bars, n)
        seconds = time.perf_counter() - start
        out.append((frechet_mnist(images), seconds))
    return out


enough_25, better5_25 = 20, "C"


curve_ok_25 = False
with checked("B5.25"):
    need_model("B5.25")
    x8_25 = torch.randn((80, 1, 28, 28), generator=torch.Generator().manual_seed(25))
    got_25 = fd_curve_25(MODEL, AB, [2, 4], x8_25)
    if returned("B5.25", "fd_curve_25", got_25):
        try:
            pairs_25 = [(float(a), float(b)) for a, b in got_25]
        except (TypeError, ValueError):
            pairs_25 = None
        if pairs_25 is None or len(pairs_25) != 2:
            print(f"❌ Ex B5.25 : fd_curve_25 doit rendre une liste de couples (distance, secondes), un par nombre de pas ; "
                  f"reçu {got_25!r:.100}.")
        else:
            want_25 = [frechet_mnist(wbd.sample_ddim(MODEL, x8_25, AB, n)) for n in (2, 4)]
            curve_ok_25 = same("B5.25", "les distances de fd_curve_25 (80 images, 2 et 4 pas) contre DDIM du contrat",
                               [a for a, _ in pairs_25], want_25, rtol=1e-3, atol=1e-3)
            verdict("B5.25", all(b > 0 for _, b in pairs_25), "les durées sont positives.", "une durée n'est pas positive.")
with wb.attempt("B5.25"):
    wb.check("B5.25a", enough_25)
with wb.attempt("B5.25"):
    wb.check("B5.25b", better5_25)

In [ ]:
wb.record("B5.25a", 20, mistakes={"relis la ligne L du protocole complet : à 5 pas, toutes les graines restent-elles au plus à 1,2 fois leur distance à 100 pas, et ensuite aussi ?": 5, "relis la ligne L du protocole complet : à 10 pas, toutes les graines restent-elles au plus à 1,2 fois leur distance à 100 pas, et ensuite aussi ?": 10, "relis la ligne L du protocole complet : à 50 pas, toutes les graines restent-elles au plus à 1,2 fois leur distance à 100 pas, et ensuite aussi ?": 50, "relis la ligne L du protocole complet : à 100 pas, toutes les graines restent-elles au plus à 1,2 fois leur distance à 100 pas, et ensuite aussi ?": 100}, fractional="on demande un nombre de pas de STEPS_25", choices=[5, 10, 20, 50, 100])
wb.record("B5.25b", "C", mistakes={"relis la colonne des 5 pas du protocole complet, graine par graine": "L"})

In [ ]:
# Your linear model: distance against time for each number of DDIM steps, over the bands of the full protocol
CURVE_25 = None
if not curve_ok_25:
    print("⏳ Ex B5.25 : la courbe attend fd_curve_25 (cellule précédente sans ❌).")
else:
    with checked("B5.25"):
        x_25 = torch.randn((N_25, 1, 28, 28), generator=torch.Generator().manual_seed(250))
        CURVE_25 = fd_curve_25(MODEL, AB, STEPS_25, x_25)
        curves_25 = {"L, ton modèle": CURVE_25}
        if not FAST_MODE:                                   # 🚀 full mode: a second model, with the cosine schedule
            torch.manual_seed(SETTINGS_23["seed"])
            cos_net_25 = UNET_CLASS(**CONFIG_23)
            cos_ema_25 = copy.deepcopy(cos_net_25)
            ab_cos_25 = wbd.alpha_bars(wbd.schedule_betas("cosine", T_STEPS))
            train_23(cos_net_25, cos_ema_25, DATA_23.images, SETTINGS_23 | {"ema": float(RHO_23)},
                     torch.Generator().manual_seed(SETTINGS_23["seed"] + 1), ab_cos_25)
            curves_25["C, ton modèle"] = fd_curve_25(cos_ema_25, ab_cos_25, STEPS_25, x_25)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        for (name_25, curve), color in zip(curves_25.items(), ("C0", "C1")):
            ax1.plot([s for _, s in curve], [d for d, _ in curve], "o-", color=color, label=name_25)
            for (d, s), n in zip(curve, STEPS_25):
                ax1.annotate(str(n), (s, d), fontsize=7)
        for key_25, color in (("L", "C0"), ("C", "C1")):
            rows_25 = np.array(FULL_25[key_25])                 # (seeds, steps)
            ax2.fill_between(STEPS_25, rows_25.min(axis=0), rows_25.max(axis=0), color=color, alpha=0.2)
            ax2.plot(STEPS_25, np.median(rows_25, axis=0), "o-", color=color, label=f"{key_25}, protocole complet")
        if not FAST_MODE:
            ax2.plot(STEPS_25, [d for d, _ in curves_25["L, ton modèle"]], "x--", color="C0", label="L, ton modèle")
        for ax, xlabel in ((ax1, "secondes d'échantillonnage"), (ax2, "nombre de pas DDIM")):
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_xlabel(xlabel)
            ax.set_ylabel("distance de Fréchet")
            ax.legend(fontsize=8)
        plt.show()
        print("pas :", "  ".join(f"{n:>6}" for n in STEPS_25))
        print(f"ta distance ({N_25} images) :", "  ".join(f"{fr(d, 1):>6}" for d, _ in CURVE_25))
        for key_25 in ("L", "C"):
            print(f"protocole complet {key_25} (médiane, 256 images) :",
                  "  ".join(f"{fr(v, 1):>6}" for v in np.median(np.array(FULL_25[key_25]), axis=0)))

### Ex B5.26 — Prédire l'effet de w = 0, 1, 3 et 7 sur la netteté et la diversité 🔮 ★ ⏱️ 10 min
**Objectif :** prédire ce que la guidance change sur Fashion-MNIST, là où la fiche ne donne que le compromis général.  
**Prérequis :** ✏️ B5.1 · fiche §205.7 (« La guidance sans classifieur », piège « deux conventions pour l'échelle de guidance »), §205.8 (« Fidélité et diversité ») · **Fil rouge :** Fashion-MNIST · **Parcours :** C

La fiche donne le compromis général mesuré par Ho et Salimans sur ImageNet. Et sur Fashion-MNIST, avec un petit U-Net conditionné par la classe ? Le **protocole complet** (précalculé, la recette de 📦 B5.27 en mode rapide sur plusieurs graines) génère 200 images (20 par classe, du même bruit pour chaque $w$, DDIM 20 pas, $\hat{\mathbf{x}}_0$ écrêté) pour $w = 0, 1, 3$ et 7, dans la convention de Ho et Salimans (`guidance_scale` $= 1 + w$), et les mesure de quatre façons :
- la **distance de Fréchet** aux 2 000 images de test (features du CNN du ch. 27 entraîné sur Fashion-MNIST) ;
- la part d'images dont le classifieur reconnaît la **classe demandée** ;
- la **diversité** : la moyenne, sur les classes, de l'écart-type moyen des features des images de la classe ;
- la part de **pixels saturés** ($|x| > 0{,}98$, du noir ou du blanc presque purs).

**Avant d'exécuter**, écris tes trois prédictions (`True` ou `False`, pour **toutes** les graines) :
- a) `fd7_26` : à $w = 7$, la distance de Fréchet est-elle **plus petite** qu'à $w = 0$ ?
- b) `acc0_26` : à $w = 0$ (le modèle conditionné, sans guidance), le classifieur reconnaît-il la classe demandée dans **plus de 70 %** des images ?
- c) `div0_26` : à $w = 0$, la diversité des images générées est-elle **plus grande** que celle des vraies images de test ?

La cellule d'expérience n'affiche le tableau du protocole qu'une fois tes trois prédictions écrites, et les vérifie à la fin. Rien n'y dépend de ton code ; tu mesures ton propre modèle en 📦 B5.27.

In [ ]:
W_26 = (0, 1, 3, 7)                                        # the guidance scales (convention of Ho and Salimans)

In [ ]:
fd7_26, acc0_26, div0_26 = True, False, True

In [ ]:
wb.record("B5.26a", True, mistakes={"compare la colonne des distances aux lignes w = 0 et w = 7, graine par graine": False}, prediction=True)
wb.record("B5.26b", False, mistakes={"lis la ligne w = 0 de la colonne de la classe reconnue": True}, prediction=True)
wb.record("B5.26c", True, mistakes={"compare la ligne w = 0 de la colonne de la diversité à celle des vraies images, sous le tableau": False}, prediction=True)

In [ ]:
if any(answer is ... or answer is None for answer in [fd7_26, acc0_26, div0_26]):
    print("⏳ Ex B5.26 : écris d'abord tes trois prédictions dans la cellule précédente.")
else:
    TABLE_26 = wbd.precomputed("guidance_26")        # the full protocol, read here only (history_chB5.py)
    print(f"protocole : {TABLE_26['n_seeds']} graines, {TABLE_26['n_images']} images par graine (autant par classe), DDIM "
          f"{TABLE_26['steps']} pas ; médiane, et de la plus petite à la plus grande valeur des graines")
    print(f"{'w':>3} {'distance de Fréchet':>26} {'classe reconnue':>22} {'diversité':>20} {'pixels saturés':>20}")
    for w_26 in W_26:
        row_26 = TABLE_26["by_w"][str(w_26)]
        cells_26 = []
        for key_26, scale_26, digits_26 in (("fd", 1, 1), ("acc", 100, 0), ("div", 1, 3), ("sat", 100, 1)):
            v_26 = np.array(row_26[key_26]) * scale_26
            cells_26.append(f"{fr(np.median(v_26), digits_26)} ({fr(v_26.min(), digits_26)}–{fr(v_26.max(), digits_26)})")
        print(f"{w_26:>3} {cells_26[0]:>26} {cells_26[1] + ' %':>22} {cells_26[2]:>20} {cells_26[3] + ' %':>20}")
    print(f"vraies images de test : diversité {fr(TABLE_26['real']['div'], 3)}, pixels saturés "
          f"{fr(100 * TABLE_26['real']['sat'], 1)} % ; plancher de la distance (200 vraies images) {fr(TABLE_26['real']['fd'], 1)}")
    wb.check("B5.26a", fd7_26)
    wb.check("B5.26b", acc0_26)
    wb.check("B5.26c", div0_26)

Dans tes notes : pour chaque prédiction ratée, qu'est-ce qui t'a trompé ? Quel $w$ donne la plus petite distance, et pourquoi celui-là ? Que deviennent la classe reconnue et la diversité quand $w$ augmente ? La part de pixels saturés des vraies images est affichée sous le tableau : que dit l'écart avec les images générées ? Que mesure la distance de Fréchet, ici, que ne mesure pas la classe reconnue, et inversement ?

### Ex B5.27 — Diffusion conditionnelle et guidance sans classifieur sur Fashion-MNIST 📦 ★★★ ⏱️ 60 min 🚀
**Objectif :** conditionner ton U-Net par la classe, l'entraîner avec l'abandon du label, et générer avec la guidance.  
**Prérequis :** Ex B5.23, B5.24, B5.26 · fiche §205.7 (« Donner la classe au réseau », « La guidance sans classifieur ») · **Fil rouge :** Fashion-MNIST · **Parcours :** C

La cellule suivante donne `FASHION` (10 000 images de Fashion-MNIST en mode rapide, les 60 000 en mode complet), `CLF_F` (le CNN du ch. 27 entraîné une fois sur Fashion-MNIST, comme `CLF` sur MNIST), `REAL_F` et `REAL_F_FEATS` (2 000 images de test et leurs features), `SETTINGS_27` (la recette de B5.23 sur 2 epochs en mode rapide, avec un $\rho$ fixé pour son nombre de pas et `p_uncond = 0.1`), `CONFIG_27` (ton U-Net avec `n_classes = 10` : sa ligne 10 est la classe vide $\varnothing$, B5.21), `Y_27` (les classes demandées, autant par classe), `NAMES_F` et `guidance_measures(images, y)`, les quatre mesures de 🔮 B5.26.

**Contrat**
- `train_cond_27(model, ema, images, labels, settings, generator, alpha_bars)` : `train_23` avec les classes. Pour chaque batch, **après** les pas et le bruit, un troisième tirage `torch.rand(b, generator=generator) < settings["p_uncond"]` remplace le label par la classe vide `model.n_classes` ; le modèle reçoit `(x_t, t, y)`. La copie EMA par ta `ema_update_23`. Rend `{"loss": [...]}`.
- `guided_eps_27(model, x, t, y, w)` : au pas d'indice `t` (un entier), $(1 + w)\,\boldsymbol{\varepsilon}_\theta(\mathbf{x}, c) - w\,\boldsymbol{\varepsilon}_\theta(\mathbf{x}, \varnothing)$, les deux prédictions en **un seul** appel du réseau sur un batch de $2N$ images (les $N$ avec `y`, puis les $N$ avec la classe vide) ; pour `w == 0`, un seul appel sur les $N$ images avec `y`. Sans graphe.
- `sample_cond_27(model, x_T, y, w, alpha_bars, n_steps)` : ta `sample_ddim_24` ($\hat{\mathbf{x}}_0$ écrêté à $[-1, 1]$, le bruit recalculé), avec `guided_eps_27(model, x, t, y, w)` à la place du bruit du réseau.

La cellule de vérification compare tes fonctions au contrat sur un petit U-Net conditionné : 4 pas de `train_cond_27` (avec une probabilité d'abandon de 0,4, pour que l'abandon agisse), `guided_eps_27` à $w = 0$ et 2,5 avec le nombre d'appels du réseau, et 5 pas de `sample_cond_27`. La suivante entraîne ton U-Net conditionné (environ 1 min en mode rapide ; 🚀 en mode complet) et le sauvegarde, comme en B5.23. La dernière génère 200 images pour chaque $w$ de `W_26` (DDIM 20 pas), montre une image par classe et par $w$, et compare tes quatre mesures aux médianes du protocole de B5.26 (environ 25 s en mode rapide).

Dans tes notes : les images à $w = 0$ ressemblent-elles à leur classe ? Et à $w = 7$ ? Quel $w$ choisirais-tu pour un catalogue de vêtements, et pour un dataset d'entraînement ? Combien de passes du réseau coûte une image à 20 pas avec guidance ? Que se passerait-il avec `p_uncond = 0` (le réseau n'aurait jamais vu $\varnothing$) ? Avec `guidance_scale = 4` dans diffusers, quel $w$ utiliserais-tu (fiche, piège des deux conventions) ?

In [ ]:
# Fashion-MNIST: the training subset, the classifier of the measures (the CNN of ch. 27, trained on Fashion-MNIST),
# 2 000 test images for the reference, the recipe (the one of B5.23 with the classes, EMA decay fixed for its steps)
FASHION = wbd.image_subset("fashion", wbd.n_train(FAST_MODE))
CLF_F = wbd.classifier("fashion")
REAL_F = wbd.image_subset("fashion", 2000, split="test")
REAL_F_FEATS = wbd.features(CLF_F, REAL_F.images)
SETTINGS_27 = SETTINGS_23 | {"epochs": 2 if FAST_MODE else 20, "ema": 0.99 if FAST_MODE else 0.999, "p_uncond": 0.1,
                             "seed": 2700}
CONFIG_27 = CONFIG_23 | {"n_classes": 10}
PER_CLASS_27 = 20 if FAST_MODE else 100
Y_27 = torch.arange(10).repeat_interleave(PER_CLASS_27)     # the requested classes: PER_CLASS_27 images per class
NAMES_F = ["T-shirt", "pantalon", "pull", "robe", "manteau", "sandale", "chemise", "basket", "sac", "bottine"]


def key_27(source="learner"):
    """The key of the checkpoint of the conditional U-Net of B5.27."""
    return wbd.checkpoint_key(CONFIG_27, SETTINGS_27, fast=FAST_MODE, source=source, data=FASHION)


def guidance_measures(images, y):
    """The four measures of B5.26 for images in [-1, 1] generated for the classes y: Fréchet distance to REAL_F
    (features of CLF_F), share of images whose predicted class is y, diversity (the mean, over the classes, of the
    mean standard deviation of the features), share of saturated pixels (|x| > 0.98)."""
    images = torch.as_tensor(images).clamp(-1, 1)
    feats = wbd.features(CLF_F, images)
    yy = np.asarray(y)
    return {"fd": float(mylearn.gen_metrics.frechet_distance_from_features(REAL_F_FEATS, feats)),
            "acc": float(np.mean(wbd.predict(CLF_F, images) == yy)),
            "div": float(np.mean([feats[yy == c].std(axis=0).mean() for c in np.unique(yy)])),
            "sat": float((images.abs() > 0.98).float().mean())}


print(f"{len(FASHION)} images de Fashion-MNIST ; le classifieur des mesures reconnaît "
      f"{fr(100 * np.mean(wbd.predict(CLF_F, REAL_F.images) == REAL_F.labels.numpy()), 1)} % des 2 000 images de test.")

In [ ]:
def train_cond_27(model, ema, images, labels, settings, generator, alpha_bars):
    """train_23 with the classes: after the noise, the dropout torch.rand(b, generator=generator) < settings["p_uncond"]
    replaces the label by the empty class model.n_classes; the model receives (x_t, t, y). Returns {"loss": [...]}."""
    ab = torch.tensor(np.asarray(alpha_bars), dtype=torch.float32)
    opt = torch.optim.AdamW(model.parameters(), lr=settings["lr"])
    losses, bs = [], settings["batch_size"]
    model.train()
    for _ in range(settings["epochs"]):
        order = torch.randperm(len(images), generator=generator)
        for i in range(0, len(images), bs):
            idx = order[i:i + bs]
            x0, y = images[idx], labels[idx]
            t = torch.randint(0, len(ab), (len(x0),), generator=generator)
            eps = torch.randn(x0.shape, generator=generator)
            drop = torch.rand(len(x0), generator=generator) < settings["p_uncond"]
            y = torch.where(drop, torch.full_like(y, model.n_classes), y)          # the empty class
            a = ab[t].view(-1, 1, 1, 1)
            loss = F.mse_loss(model(a.sqrt() * x0 + (1 - a).sqrt() * eps, t, y), eps)
            opt.zero_grad()
            loss.backward()
            opt.step()
            ema_update_23(ema, model, settings["ema"])
            losses.append(loss.item())
    model.eval()
    ema.eval()
    return {"loss": losses}


def guided_eps_27(model, x, t, y, w):
    """The guided noise at the step index t (an int): (1 + w) eps(x, t, y) - w eps(x, t, ∅), the two predictions in
    ONE call of the model on 2 N images; for w = 0, one call on the N images. No graph."""
    tt = torch.full((len(x),), t, dtype=torch.long)
    with torch.no_grad():
        if w == 0:
            return model(x, tt, y)
        both = model(torch.cat([x, x]), torch.cat([tt, tt]), torch.cat([y, torch.full_like(y, model.n_classes)]))
    cond, uncond = both.chunk(2)
    return (1 + w) * cond - w * uncond


def sample_cond_27(model, x_T, y, w, alpha_bars, n_steps):
    """sample_ddim_24 (x̂0 clipped to [-1, 1]) with guided_eps_27(model, x, t, y, w) as the network's noise."""
    D = mylearn.diffusion
    x = x_T.double().numpy()
    steps = [int(t) for t in wbd.ddim_timesteps(len(alpha_bars), n_steps)] + [-1]
    for t, t_prev in zip(steps[:-1], steps[1:]):
        eps = guided_eps_27(model, torch.as_tensor(x, dtype=torch.float32), t, y, w).double().numpy()
        x0 = np.clip(D.predict_x0_from_eps(x, np.full(len(x), t), eps, alpha_bars), -1, 1)
        eps = (x - math.sqrt(alpha_bars[t]) * x0) / math.sqrt(1 - alpha_bars[t])
        x = D.ddim_step(x, t, t_prev, eps, alpha_bars)
    return torch.as_tensor(x, dtype=torch.float32)


cond_ok_27 = False
with checked("B5.27"):
    torch.manual_seed(1)
    small_27 = wbd.TinyUNet(20, channels=(8, 8, 16), t_dim=16, n_classes=3)
    with torch.no_grad():
        for p_ in small_27.parameters():
            p_.add_(0.1 * torch.randn(p_.shape))
    ab20_27 = wbd.alpha_bars(np.linspace(1e-3, 0.2, 20))   # a short schedule
    probe_27 = {"epochs": 2, "batch_size": 4, "lr": 1e-3, "ema": 0.5, "p_uncond": 0.4}
    imgs_27, labels_27 = FASHION.images[:6].clone(), FASHION.labels[:6] % 3
    m1, e1, m2, e2 = (copy.deepcopy(small_27) for _ in range(4))
    hist_27 = train_cond_27(m1, e1, imgs_27, labels_27.clone(), probe_27, torch.Generator().manual_seed(6), ab20_27)
    want_27 = wbd.train_unet(m2, e2, imgs_27, probe_27, torch.Generator().manual_seed(6), ab20_27, labels=labels_27)
    ok_27 = returned("B5.27", "train_cond_27", hist_27) and isinstance(hist_27, dict)
    if ok_27:
        ok_27 = same("B5.27", "les 4 loss de train_cond_27 contre la recette du contrat (mêmes tirages, abandon du label "
                     "compris)", hist_27.get("loss"), want_27["loss"], rtol=1e-4, atol=1e-6)
        ok_27 &= all(torch.allclose(a, b, rtol=1e-4, atol=1e-6) for a, b in zip(e1.parameters(), e2.parameters()))
        verdict("B5.27", ok_27, "les poids et la copie EMA après 4 pas : ceux de la recette.",
                "les poids ou la copie EMA après 4 pas ne sont pas ceux de la recette.")
    m2.eval()
    x5_27, y5_27 = torch.randn((5, 1, 28, 28), generator=torch.Generator().manual_seed(7)), torch.tensor([0, 1, 2, 0, 3])
    for w_27, calls_27, rows_27 in ((0.0, 1, 5), (2.5, 1, 10)):
        with PassCounter(m2) as count_27:
            got_27 = guided_eps_27(m2, x5_27, 13, y5_27, w_27)
        ok_27 &= same("B5.27", f"guided_eps_27 à w = {fr(w_27, 1)} (dont une classe vide) contre la formule", got_27,
                      wbd.predict_eps(m2, x5_27, 13, y5_27, w_27), rtol=1e-4, atol=1e-5)
        ok_27 &= (count_27.calls, count_27.images) == (calls_27, rows_27)
        verdict("B5.27", (count_27.calls, count_27.images) == (calls_27, rows_27),
                f"à w = {fr(w_27, 1)} : {calls_27} appel du réseau sur {rows_27} images.",
                f"à w = {fr(w_27, 1)}, attendu {calls_27} appel du réseau sur {rows_27} images ; vu {count_27.calls} "
                f"appel(s) sur {count_27.images} images.")
    ok_27 &= same("B5.27", "sample_cond_27 (5 pas, w = 2) contre DDIM guidé du contrat",
                  sample_cond_27(m2, x5_27, y5_27, 2.0, ab20_27, 5),
                  wbd.sample_ddim(m2, x5_27, ab20_27, 5, y=y5_27, w=2.0), rtol=1e-4, atol=1e-4)
    cond_ok_27 = bool(ok_27)

In [ ]:
# Train your conditional U-Net on Fashion-MNIST (about 1 min on CPU in fast mode; 🚀 in full mode), save it once
MODEL_27 = None
if not cond_ok_27:
    print("⏳ Ex B5.27 : l'entraînement attend les contrôles de la cellule précédente.")
else:
    with checked("B5.27"):
        KEY_27 = key_27(globals().get("CKPT_SOURCE_23", "learner"))
        wbd.delete_checkpoint(KEY_27)
        torch.manual_seed(SETTINGS_27["seed"])
        net_27 = UNET_CLASS(**CONFIG_27)
        ema_27 = copy.deepcopy(net_27)
        start_27 = time.perf_counter()
        hist_27 = train_cond_27(net_27, ema_27, FASHION.images, FASHION.labels, SETTINGS_27,
                                torch.Generator().manual_seed(SETTINGS_27["seed"] + 1), AB)
        print(f"{len(hist_27['loss'])} pas en {time.perf_counter() - start_27:.0f} s")
        wbd.save_checkpoint(KEY_27, net_27, ema=ema_27, history=hist_27)
        MODEL_27 = ema_27
        del net_27
        wbd.trim_memory()

In [ ]:
# Your model, for w = 0, 1, 3, 7: the same starting noise and the same classes for every w (DDIM, 20 steps)
MEASURES_27 = None
if MODEL_27 is None:
    print("⏳ Ex B5.27 : les mesures attendent ton modèle entraîné (cellule précédente).")
else:
    with checked("B5.27"):
        x_27 = torch.randn((len(Y_27), 1, 28, 28), generator=torch.Generator().manual_seed(271))
        MEASURES_27, grid_27 = {}, []
        for w_27 in W_26:
            out_27 = sample_cond_27(MODEL_27, x_27, Y_27, w_27, AB, 20)
            MEASURES_27[w_27] = guidance_measures(out_27, Y_27)
            grid_27.append(out_27[::PER_CLASS_27])                  # the first image of each class
        show_images(torch.cat(grid_27), ncol=10, size=0.9,
                    titles=[f"{NAMES_F[c]}, w={w}" for w in W_26 for c in range(10)],
                    suptitle="une image par classe (colonnes), pour w = 0, 1, 3 et 7 (lignes), du même bruit")
        table_27 = wbd.precomputed("guidance_26")["by_w"]
        print(f"{'w':>3} {'distance':>9} {'(protocole)':>12} {'reconnue':>9} {'(protocole)':>12} {'diversité':>10} "
              f"{'saturés':>8}")
        for w_27, m in MEASURES_27.items():
            print(f"{w_27:>3} {fr(m['fd'], 1):>9} {fr(np.median(table_27[str(w_27)]['fd']), 1):>12} "
                  f"{fr(100 * m['acc'], 0) + ' %':>9} {fr(100 * np.median(table_27[str(w_27)]['acc']), 0) + ' %':>12} "
                  f"{fr(m['div'], 3):>10} {fr(100 * m['sat'], 1) + ' %':>8}")
        verdict("B5.27", MEASURES_27[3]["acc"] > MEASURES_27[0]["acc"] and MEASURES_27[3]["div"] < MEASURES_27[0]["div"],
                "la guidance fait reconnaître la classe plus souvent, au prix de la diversité (w = 3 contre w = 0).",
                "à w = 3, la classe devrait être reconnue plus souvent et la diversité plus petite qu'à w = 0 : relis "
                "guided_eps_27 (le signe de l'extrapolation, la classe vide).")

### Ex B5.28 — Mémorisation : plus proches voisins des échantillons, entraînement sur 500 images contre 60 000 🔬 ★★★ ⏱️ 40 min 🚀
**Objectif :** chercher les copies parmi les échantillons, et mesurer comment la taille du dataset d'entraînement les change.  
**Prérequis :** Ex B5.22, B5.23 · ch. 13 (plus proches voisins) · fiche §205.8 (« La mémorisation ») · **Fil rouge :** MNIST · **Parcours :** C

Une distance faible à son plus proche voisin ne suffit pas à déclarer une copie : un « 1 » bien droit est proche de beaucoup d'autres « 1 ». Comme Carlini et coll. (fiche §205.8), on compare la distance d'un échantillon à son plus proche voisin $a$ dans le dataset d'entraînement à la distance de $a$ à **ses propres** voisins :
$$r = \frac{d(\text{échantillon},\ a)}{d(a,\ b)},$$
où $b$ est l'image d'entraînement la plus proche de $a$ (autre que $a$). Un échantillon de $r < 1/3$ est anormalement proche de $a$ : une copie. La cellule suivante donne `FULL_28`, le **protocole complet** précalculé (la recette de B5.23 avec le **même nombre de pas d'optimisation**, `N_UPDATES_23`, sur 500, 10 000 et 60 000 images, deux graines) : pour chaque modèle, la part de copies parmi 256 échantillons et le $r$ médian ; et `IdealNet`, le débruiteur parfait de B5.22 vu comme un réseau.

**Contrat**
- `nearest_28(samples, train)` : pour chaque image de `samples` `(N, 1, 28, 28)`, son plus proche voisin dans `train` `(M, 1, 28, 28)` en pixels : le couple (distances, indices), deux tableaux NumPy `(N,)` ; la distance euclidienne des images aplaties ; à égalité, le premier indice.
- `copy_ratio_28(samples, train)` : le $r$ de chaque échantillon, un tableau float64 `(N,)`.

Puis, d'après `FULL_28` :
- a) `copies500_28` : pour les modèles entraînés sur 500 images, la part des échantillons de $r < 1/3$, pour **toutes** les graines : `"A"` moins de 10 %, `"B"` de 10 à 50 %, `"C"` plus de 50 % ;
- b) `closer500_28` : le $r$ médian des modèles entraînés sur 500 images est-il **plus petit** que celui des modèles entraînés sur 60 000 images, graine par graine ? `True` ou `False`.

La cellule de vérification essaie tes deux fonctions sur quatre points du plan (où tu peux refaire les calculs à la main) et sur 5 chiffres à peine bruités, puis vérifie tes réponses. La suivante génère 128 échantillons (DDIM, 50 pas) de ton U-Net de B5.23 et du débruiteur parfait des 500 premières images de `DATA_23`, montre pour chacun les 8 échantillons de plus petit $r$ et leur voisin, et trace les deux distributions de $r$ (environ 15 s en mode rapide). 🚀 En mode complet, elle entraîne aussi ton U-Net sur ces 500 images avec le même nombre de pas que B5.23, et le mesure.

Dans tes notes : que vaut $r$ pour le débruiteur parfait, et pourquoi ? Ton U-Net copie-t-il ? Que change la taille du dataset, à nombre de pas égal (combien de fois chaque image est-elle vue) ? Pourquoi la distance de Fréchet ne verrait-elle rien (🧠 Q10) ? Pourquoi comparer en pixels, et que changerait une comparaison dans l'espace des features de `CLF` ? Que t'apprennent Somepalli et coll., et Kadkhodaie et coll. (2024), qui étudient la même question (cherche leurs articles) ?

In [ ]:
# The full protocol (tools/chapters/history_chB5.py): the recipe of B5.23 with the SAME number of optimizer steps
# (N_UPDATES_23) on 500, 10 000 and 60 000 images, deux seeds each; 256 DDIM samples (50 steps) per model and,
# for each sample, the ratio r of copy_ratio_28; the share of samples with r < 1/3
FULL_28 = {
    "copies": {"500": [0.0, 0.0], "10000": [0.0, 0.0], "60000": [0.0, 0.0]},
    "median_r": {"500": [1.2944, 1.3693], "10000": [1.4182, 1.4234], "60000": [1.49, 1.4954]},
}


class IdealNet(nn.Module):
    """The perfect denoiser of a finite dataset (wbd.ideal_denoiser, B5.22) as a network model(x, t)."""

    def __init__(self, images):
        super().__init__()
        self.eps_fn = wbd.ideal_denoiser(images, np.sqrt(AB), np.sqrt(1 - AB))

    def forward(self, x, t):
        return self.eps_fn(x, int(t[0]))

In [ ]:
def nearest_28(samples, train):
    """For each sample (N, 1, 28, 28), its nearest image of train (M, 1, 28, 28) in pixels: (distances, indices), two
    NumPy arrays (N,), the Euclidean distance of the flattened images and the index in train (the first one on ties)."""
    a = torch.as_tensor(samples).reshape(len(samples), -1).double()
    b = torch.as_tensor(train).reshape(len(train), -1).double()
    d, i = torch.cdist(a, b).min(dim=1)
    return d.numpy(), i.numpy()


def copy_ratio_28(samples, train):
    """For each sample, r = d(sample, a) / d(a, b), where a is its nearest training image and b the nearest image of
    train OTHER than a (the closest neighbour of a in its own dataset); a float64 NumPy array (N,)."""
    d, idx = nearest_28(samples, train)
    flat = torch.as_tensor(train).reshape(len(train), -1).double()
    to_all = torch.cdist(flat[idx], flat)                          # distances of each a to every training image
    to_all[torch.arange(len(idx)), torch.as_tensor(idx)] = float("inf")   # not a itself
    return d / to_all.min(dim=1).values.numpy()


copies500_28, closer500_28 = "A", True


ratio_ok_28 = False
with checked("B5.28"):
    train_c = torch.tensor([[0.0, 0.0], [3.0, 0.0], [3.0, 4.0], [10.0, 10.0]]).reshape(4, 1, 1, 2)
    queries_c = torch.tensor([[0.0, 0.5], [3.0, 3.0], [9.0, 9.0]]).reshape(3, 1, 1, 2)
    nd_28 = nearest_28(queries_c, train_c)
    if returned("B5.28", "nearest_28", nd_28):
        d_c, i_c = pair("B5.28", "nearest_28", nd_28, names=("distances", "indices"))
        ok_28 = same("B5.28", "nearest_28 sur 4 points du plan : les distances", d_c, [0.5, 1.0, math.sqrt(2)])
        ok_28 &= same("B5.28", "nearest_28 : les indices", i_c, [0, 2, 3])
        ok_28 &= same("B5.28", "copy_ratio_28 sur les mêmes points (0,5/3 ; 1/4 ; √2/√85)", copy_ratio_28(queries_c, train_c),
                      [0.5 / 3, 1.0 / 4, math.sqrt(2) / math.sqrt(85)])
        imgs_28 = DATA_23.images[:300]
        ok_28 &= same("B5.28", "copy_ratio_28 de 5 images d'entraînement à peine bruitées (σ = 0,01) contre la référence",
                      copy_ratio_28(imgs_28[:5] + 0.01 * torch.randn(5, 1, 28, 28, generator=torch.Generator().manual_seed(8)),
                                    imgs_28),
                      wbd.copy_ratio(imgs_28[:5] + 0.01 * torch.randn(5, 1, 28, 28, generator=torch.Generator().manual_seed(8)),
                                     imgs_28), rtol=1e-6, atol=1e-9)
        ratio_ok_28 = bool(ok_28)
with wb.attempt("B5.28"):
    wb.check("B5.28a", copies500_28)
with wb.attempt("B5.28"):
    wb.check("B5.28b", closer500_28)

In [ ]:
wb.record("B5.28a", "A", mistakes={"relis la ligne des 500 images de FULL_28 : la part de copies y est-elle de 10 à 50 % pour toutes les graines ?": "B", "relis la ligne des 500 images de FULL_28 : la part de copies y est-elle plus de 50 % pour toutes les graines ?": "C"})
wb.record("B5.28b", True, mistakes={"compare les r médians des lignes 500 et 60 000 de FULL_28, graine par graine": False})

In [ ]:
# Your model (B5.23) and the perfect denoiser of 500 images (B5.22): how close are their samples to their training images?
if not ratio_ok_28:
    print("⏳ Ex B5.28 : l'expérience attend nearest_28 et copy_ratio_28 (cellule précédente sans ❌).")
else:
    with checked("B5.28"):
        need_model("B5.28")
        x_28 = torch.randn((128, 1, 28, 28), generator=torch.Generator().manual_seed(280))
        small_set_28 = DATA_23.images[:500]                       # the first 500 images of the frozen subset
        runs_28 = {f"ton U-Net ({len(DATA_23)} images)": (MODEL, DATA_23.images),
                   "débruiteur parfait (500 images)": (IdealNet(small_set_28), small_set_28)}
        if not FAST_MODE:                     # 🚀 full mode: a U-Net trained with the same number of steps on 500 images
            torch.manual_seed(SETTINGS_23["seed"])
            net500_28, ema500_28 = UNET_CLASS(**CONFIG_23), None
            ema500_28 = copy.deepcopy(net500_28)
            epochs_28 = math.ceil(N_UPDATES_23 / math.ceil(500 / SETTINGS_23["batch_size"]))
            train_23(net500_28, ema500_28, small_set_28, SETTINGS_23 | {"ema": float(RHO_23), "epochs": epochs_28},
                     torch.Generator().manual_seed(SETTINGS_23["seed"] + 1), AB)
            runs_28["ton U-Net (500 images)"] = (ema500_28, small_set_28)
        RATIOS_28 = {}
        for name_28, (net, train) in runs_28.items():
            samples_28 = wbd.sample_ddim(net, x_28, AB, 50)
            r_28 = copy_ratio_28(samples_28, train)
            RATIOS_28[name_28] = r_28
            d_28, i_28 = nearest_28(samples_28, train)
            order_28 = np.argsort(r_28)[:8]
            show_images(torch.cat([samples_28[order_28], train[i_28[order_28]]]), ncol=8, size=0.9,
                        titles=[f"r={fr(r_28[k], 2)}" for k in order_28] + ["voisin"] * 8,
                        suptitle=f"{name_28} : les 8 échantillons de plus petit r, et leur plus proche voisin")
            print(f"{name_28} : r médian {fr(np.median(r_28), 2)} ; {fr(100 * np.mean(r_28 < 1 / 3), 1)} % des 128 "
                  "échantillons ont r < 1/3")
        fig, ax = plt.subplots(figsize=(7, 3))
        for name_28, r_28 in RATIOS_28.items():
            ax.hist(np.clip(r_28, 0, 2), bins=40, range=(0, 2), alpha=0.5, label=name_28)
        ax.axvline(1 / 3, color="k", ls=":", lw=1)
        ax.set_xlabel("r = d(échantillon, voisin) / d(voisin, son propre voisin)")
        ax.legend(fontsize=8)
        plt.show()
        print("protocole complet, part des échantillons de r < 1/3 (une valeur par graine) :")
        for size_28, shares in FULL_28["copies"].items():
            print(f"  {int(size_28):>6} images : " + " ; ".join(f"{fr(100 * s, 1)} %" for s in shares)
                  + f" (r médian {fr(float(np.median(FULL_28['median_r'][size_28])), 2)})")

### Ex B5.29 — Distance de Fréchet : situer le DDPM face au VAE (ch. 25) et au DCGAN (ch. 27) sur MNIST, avec le code d'évaluation du mini-projet MP6 📦 ★★★ ⏱️ 30 min
**Objectif :** évaluer trois générateurs avec le même protocole et le même code, et lire ce que les mesures disent ou non.  
**Prérequis :** Ex B5.24 · ch. 25 (VAE), ch. 27 (DCGAN, distance de Fréchet) · mini-projet MP6 (`genbench`) · fiche §205.8 · **Fil rouge :** MNIST · **Parcours :** C

Le mini-projet MP6 (`projets/partie_6_generatif/`) compare un VAE et un DCGAN avec son package `genbench` : des modèles, leurs entraînements et une évaluation (`genbench.evaluate` : `classifier_features`, `frechet_distance`, `class_report`, `nearest_neighbors`). La cellule suivante importe la **solution** de ce package (`projets/partie_6_generatif/solution/genbench` : elle ne dépend pas de ton avancement dans le MP6), entraîne une fois sur `DATA_23` son VAE (`ConvVAE`, 16 latents, $\beta = 0{,}5$, 3 epochs) et son DCGAN (3 epochs), puis les relit de `wbd.checkpoints_dir()` (environ 35 s la première fois). `genbench` travaille sur des images dans $[0, 1]$ : `CLF01` est `CLF` vu par `genbench` (features et logits d'images dans $[0, 1]$), `TRAIN01_29` les images de `DATA_23` dans $[0, 1]$. Le MP6 mesurait des images de Fashion-MNIST avec son propre classifieur (128 features) ; ici, sur MNIST, les features sont celles de `CLF` (64 valeurs), comme en B5.23 à B5.25.

**Contrat**
- `evaluate_29(images01, train01)` : l'évaluation du MP6 d'images générées dans $[0, 1]$, avec les fonctions de `genbench.evaluate` (`gb_eval`) et `CLF01` ; un dictionnaire :
  - `"fd"` : `frechet_distance` entre les features des 2 000 chiffres de test (`REAL_MNIST`, ramenés dans $[0, 1]$) et celles de `images01` ;
  - `"entropy"` et `"confidence"` : celles de `class_report(CLF01, images01)` ;
  - `"nn"` : la médiane des distances de `nearest_neighbors(images01, train01)`.

La cellule de vérification compare `evaluate_29` à un calcul direct sur 100 vrais chiffres. La suivante prend 256 images de chaque générateur (ton DDPM de B5.23 par DDIM à 50 pas, le VAE, le DCGAN) et 256 vrais chiffres d'entraînement tenus à l'écart, les mesure avec ta fonction (les mêmes features, le même nombre d'images, la même référence) et les montre.

Dans tes notes : classe les trois générateurs selon la distance de Fréchet ; ce classement suit-il ton jugement à l'œil ? Que disent l'entropie des classes (effondrement de mode, ch. 27) et la confiance du classifieur ? Pourquoi le VAE est-il si loin (pense au flou et aux features) ? Que mesure `nn`, et que vaut-il pour les vraies images ? À coût d'entraînement voisin, qu'est-ce qui rend la comparaison inégale (nombre de passes à la génération, taille des modèles, epochs) ?

In [ ]:
# The code of the mini-project MP6 (projets/partie_6_generatif/solution/genbench: its VAE of ch. 25, its DCGAN of
# ch. 27, their trainings and its evaluation), on MNIST; the features are those of CLF (64 values), as in B5.23
import sys
from pathlib import Path

MP6_29 = wb.repo_root() / "projets" / "partie_6_generatif" / "solution"
if str(MP6_29) not in sys.path:
    sys.path.insert(0, str(MP6_29))
from genbench import evaluate as gb_eval, models as gb_models, train as gb_train  # noqa: E402

TRAIN01_29 = (DATA_23.images + 1) / 2                    # genbench works in [0, 1]
GENBENCH_29 = {"vae": {"latent_dim": 16, "width": 32, "epochs": 3, "beta": 0.5, "seed": 0},
               "dcgan": {"latent_dim": 64, "width": 32, "epochs": 3, "seed": 0}}


class Clf01(nn.Module):
    """CLF seen by genbench: images in [0, 1] (CLF works in [-1, 1]), features() and the 10 logits."""

    def __init__(self, clf):
        super().__init__()
        self.clf = clf

    def features(self, x):
        return self.clf.features(2 * x - 1)

    def forward(self, x):
        return self.clf(2 * x - 1)


CLF01 = Clf01(CLF).eval()


def train_vae_29():
    s = GENBENCH_29["vae"]
    gb_train.set_seed(s["seed"])
    vae = gb_models.ConvVAE(latent_dim=s["latent_dim"], width=s["width"])
    gb_train.train_vae(vae, TRAIN01_29, epochs=s["epochs"], beta=s["beta"])
    return vae


def train_dcgan_29():
    import tempfile
    s = GENBENCH_29["dcgan"]
    gb_train.set_seed(s["seed"])
    G, D = gb_models.Generator(latent_dim=s["latent_dim"], width=s["width"]), gb_models.Discriminator(width=s["width"])
    G.apply(gb_models.init_dcgan_weights)
    D.apply(gb_models.init_dcgan_weights)
    gb_train.train_gan(G, D, TRAIN01_29, epochs=s["epochs"], folder=Path(tempfile.mkdtemp(prefix="wb_b5_29_")))
    return G


# trained once (≈ 35 s on CPU), then read from wbd.checkpoints_dir() (a key with the settings and the data)
VAE_29 = wbd.cached_module({"model": "genbench-vae", "settings": GENBENCH_29["vae"], "data": DATA_23.describe()},
                           lambda: gb_models.ConvVAE(latent_dim=16, width=32), train_vae_29)
GAN_29 = wbd.cached_module({"model": "genbench-dcgan", "settings": GENBENCH_29["dcgan"], "data": DATA_23.describe()},
                           lambda: gb_models.Generator(latent_dim=64, width=32), train_dcgan_29)

In [ ]:
def evaluate_29(images01, train01):
    """The evaluation of the MP6 for generated images in [0, 1] (N, 1, 28, 28), with the functions of
    genbench.evaluate and the classifier CLF01: {"fd": frechet_distance between the features of the 2 000 test digits
    (REAL_MNIST, brought to [0, 1]) and those of images01, "entropy" and "confidence": those of class_report,
    "nn": the median of the distances of nearest_neighbors(images01, train01)}."""
    real = gb_eval.classifier_features(CLF01, (REAL_MNIST.images + 1) / 2)
    fake = gb_eval.classifier_features(CLF01, images01)
    report = gb_eval.class_report(CLF01, images01)
    distances, _ = gb_eval.nearest_neighbors(images01, train01)
    return {"fd": gb_eval.frechet_distance(real, fake), "entropy": report["entropy"],
            "confidence": report["confidence"], "nn": float(np.median(distances))}


eval_ok_29 = False
with checked("B5.29"):
    probe_29 = (DATA_23.images[-100:] + 1) / 2
    got_29 = evaluate_29(probe_29, TRAIN01_29[:2000])
    if returned("B5.29", "evaluate_29", got_29):
        keys_29 = set(got_29) if isinstance(got_29, dict) else set()
        if keys_29 != {"fd", "entropy", "confidence", "nn"}:
            print(f"❌ Ex B5.29 : evaluate_29 rend un dictionnaire aux clés fd, entropy, confidence et nn ; reçu {got_29!r:.120}.")
        else:
            real_29 = wbd.features(CLF, REAL_MNIST.images)
            want_29 = {"fd": mylearn.gen_metrics.frechet_distance_from_features(real_29, wbd.features(CLF, 2 * probe_29 - 1)),
                       "nn": float(np.median(torch.cdist(probe_29.reshape(100, -1).double(),
                                                         TRAIN01_29[:2000].reshape(len(TRAIN01_29[:2000]), -1).double()).min(dim=1).values.numpy()))}
            proba_29 = torch.softmax(CLF(2 * probe_29 - 1), dim=1).detach()
            shares_29 = np.bincount(proba_29.argmax(dim=1).numpy(), minlength=10) / 100
            nz_29 = shares_29[shares_29 > 0]
            want_29["entropy"] = float(-(nz_29 * np.log(nz_29)).sum() / np.log(10))
            want_29["confidence"] = float(proba_29.max(dim=1).values.mean())
            eval_ok_29 = same("B5.29", "evaluate_29 sur 100 vrais chiffres (fd, entropy, confidence, nn) contre le calcul "
                              "direct", [float(got_29[k]) for k in ("fd", "entropy", "confidence", "nn")],
                              [want_29[k] for k in ("fd", "entropy", "confidence", "nn")], rtol=1e-4, atol=1e-6)

In [ ]:
# 256 images of each generator, measured by YOUR evaluate_29 (the same features, the same number of images)
if not eval_ok_29:
    print("⏳ Ex B5.29 : la comparaison attend evaluate_29 (cellule précédente sans ❌).")
else:
    with checked("B5.29"):
        need_model("B5.29")
        x_29 = torch.randn((256, 1, 28, 28), generator=torch.Generator().manual_seed(290))
        GENERATED_29 = {"DDPM (ton U-Net, DDIM 50 pas)": (wbd.sample_ddim(MODEL, x_29, AB, 50).clamp(-1, 1) + 1) / 2,
                        "VAE (ch. 25, genbench)": gb_eval.sample_images(VAE_29, 256, seed=291),
                        "DCGAN (ch. 27, genbench)": gb_eval.sample_images(GAN_29, 256, seed=291),
                        "vraies images (d'autres chiffres d'entraînement)": TRAIN01_29[-256:]}
        RESULTS_29 = {name: evaluate_29(images, TRAIN01_29[:-256]) for name, images in GENERATED_29.items()}
        show_images(torch.cat([2 * images[:8] - 1 for images in GENERATED_29.values()]), ncol=8, size=0.9,
                    suptitle="8 images de chaque générateur (lignes : DDPM, VAE, DCGAN, vraies images)")
        print(f"{'générateur':<50} {'distance':>9} {'entropie':>9} {'confiance':>10} {'voisin médian':>14}")
        for name, r in RESULTS_29.items():
            print(f"{name:<50} {fr(r['fd'], 1):>9} {fr(r['entropy'], 3):>9} {fr(r['confidence'], 3):>10} {fr(r['nn'], 2):>14}")
        print("(256 images par générateur, features de CLF, distance aux 2 000 chiffres de test ; « voisin médian » : la "
              "distance en pixels médiane au plus proche chiffre d'entraînement)")

### Ex B5.30 — Un échantillonneur autonome : sample.py avec --steps, --seed et --guidance, images et paramètres sauvegardés 🛠️ ★★ ⏱️ 25 min
**Objectif :** livrer un script en ligne de commande qui génère des images d'un checkpoint, reproductible et documenté.  
**Prérequis :** Ex B5.24, B5.27 · ch. 23 (livrer un modèle) · fiche §205.6 · **Fil rouge :** MNIST · **Parcours :** C

Un modèle ne sert qu'une fois sorti du notebook. Écris `sample.py`, un script qu'on lance dans un terminal : il relit un checkpoint de B5.23 ou de B5.27 (un fichier de `wbd.save_checkpoint`), génère des images par DDIM et sauvegarde les images **et** les paramètres qui permettent de les refaire. Tu écris son code dans la chaîne `SAMPLE_PY_30` ; la cellule de vérification l'enregistre dans `OUT_30 / "sample.py"` et le lance par `subprocess` (un vrai processus, sans les variables du notebook).

**Contrat** : `python sample.py CHECKPOINT [--steps 50] [--seed 0] [--guidance 0] [--label K] [--n 16] [--out samples]`
- le checkpoint se lit par `torch.load(..., weights_only=True)` : un dictionnaire avec `"config"` (de quoi reconstruire `wbd.TinyUNet(**config)`, mêmes noms que ton U-Net), `"ema_state_dict"` (les poids à utiliser) et `"key"`, la clé à provenance en JSON (`json.loads(payload["key"])["settings"]` donne `"T"` et `"schedule"`, le nom du planning pour `wbd.schedule_betas`) ;
- le bruit de départ `torch.randn((n, 1, 28, 28), generator=torch.Generator().manual_seed(seed))` ; les classes : toutes `--label` s'il est donné, sinon `torch.arange(n) % 10` pour un modèle conditionné, aucune pour un modèle sans classes ; les images par `wbd.sample_ddim(model, x_T, alpha_bars, steps, y=y, w=guidance)`, ramenées dans $[-1, 1]$ ;
- dans le dossier `--out` (créé s'il manque) : `samples.npy` (les images, float32 `(n, 1, 28, 28)`), `samples.png` (une grille) et `params.json`, avec au moins `checkpoint_sha256` (l'empreinte du fichier), `steps`, `subsequence` (les indices de `wbd.ddim_timesteps`), `seed`, `guidance` et `n` ;
- **refuse**, par un message sur la sortie d'erreur et un code de sortie non nul, sans rien générer : `--steps` hors de $[1, T]$, `--guidance` ou `--label` sur un modèle sans classes, `--label` hors des classes ; le code de sortie vaut 0 sinon.

La cellule de vérification lance ton script plusieurs fois (environ 20 s) : 4 images de ton modèle de B5.23 à 10 pas, comparées à `wbd.sample_ddim` ; `params.json` ; deux appels de même graine (les mêmes images au bit près) et une autre graine ; deux refus ; puis, si le modèle conditionné de B5.27 est sauvegardé, `--guidance 3 --label 7`.

Dans tes notes : pourquoi l'empreinte du checkpoint dans `params.json`, et pas seulement son chemin ? Qu'est-ce qui pourrait encore changer les images d'une machine à l'autre, à graine égale (versions, CPU ou GPU) ? Pourquoi un code de sortie non nul plutôt qu'un simple message ? Quelle option ajouterais-tu pour livrer ce script à un collègue ?

In [ ]:
import tempfile
from pathlib import Path

OUT_30 = Path(tempfile.mkdtemp(prefix="wb_b5_30_"))   # the solutions notebook writes nothing in the repository

SAMPLE_PY_30 = r"""# sample.py: images of a diffusion checkpoint of chapter B5 (DDIM, x̂0 clipped), with their parameters.
#
#     python sample.py CHECKPOINT [--steps 50] [--seed 0] [--guidance 0] [--label K] [--n 16] [--out samples]
import argparse
import datetime
import hashlib
import json
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch

from wb import diffusion as wbd


def parse_args(argv):
    p = argparse.ArgumentParser(description="DDIM samples of a checkpoint of chapter B5")
    p.add_argument("checkpoint", type=Path)
    p.add_argument("--steps", type=int, default=50)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--guidance", type=float, default=0.0)
    p.add_argument("--label", type=int, default=None)
    p.add_argument("--n", type=int, default=16)
    p.add_argument("--out", type=Path, default=Path("samples"))
    return p.parse_args(argv)


def fail(message):
    print(f"sample.py: {message}", file=sys.stderr)
    return 2


def main(argv=None):
    args = parse_args(argv)
    if not args.checkpoint.is_file():
        return fail(f"no checkpoint {args.checkpoint}")
    payload = torch.load(args.checkpoint, map_location="cpu", weights_only=True)
    settings, config = json.loads(payload["key"])["settings"], payload["config"]
    T, n_classes = config["n_steps"], config["n_classes"]
    if not 1 <= args.steps <= T:
        return fail(f"--steps must be in [1, {T}], got {args.steps}")
    if args.n < 1 or args.guidance < 0:
        return fail("--n must be >= 1 and --guidance >= 0")
    if n_classes is None and (args.guidance > 0 or args.label is not None):
        return fail("this checkpoint has no classes: no --guidance, no --label")
    if args.label is not None and not 0 <= args.label < n_classes:
        return fail(f"--label must be in [0, {n_classes})")
    model = wbd.TinyUNet(**config)
    model.load_state_dict(payload["ema_state_dict"] or payload["state_dict"])
    model.eval()
    alpha_bars = wbd.alpha_bars(wbd.schedule_betas(settings["schedule"], T))
    x_T = torch.randn((args.n, 1, 28, 28), generator=torch.Generator().manual_seed(args.seed))
    y = None
    if n_classes is not None:
        y = torch.full((args.n,), args.label) if args.label is not None else torch.arange(args.n) % n_classes
    start = time.perf_counter()
    images = wbd.sample_ddim(model, x_T, alpha_bars, args.steps, y=y, w=args.guidance).clamp(-1, 1)
    seconds = time.perf_counter() - start
    args.out.mkdir(parents=True, exist_ok=True)
    np.save(args.out / "samples.npy", images.numpy())
    ncol = min(8, args.n)
    nrow = -(-args.n // ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol, nrow), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, img in zip(axes.ravel(), wbd.to_unit(images)):
        ax.imshow(img[0], cmap="gray", vmin=0, vmax=1)
    fig.savefig(args.out / "samples.png", dpi=80)
    params = {"checkpoint": str(args.checkpoint), "checkpoint_sha256": hashlib.sha256(args.checkpoint.read_bytes()).hexdigest(),
              "steps": args.steps, "subsequence": [int(t) for t in wbd.ddim_timesteps(T, args.steps)],
              "seed": args.seed, "guidance": args.guidance, "label": args.label, "n": args.n,
              "seconds": round(seconds, 3), "torch": torch.__version__, "date": datetime.date.today().isoformat()}
    (args.out / "params.json").write_text(json.dumps(params, indent=1), encoding="utf-8")
    print(f"{args.n} images -> {args.out / 'samples.png'} ({seconds:.1f} s)")
    return 0


if __name__ == "__main__":
    sys.exit(main())
"""


import json
import subprocess
import sys


def run_30(*args, out="run"):
    """Run YOUR sample.py in OUT_30 with the arguments; returns (exit code, stderr, folder of the images)."""
    folder = OUT_30 / out
    done = subprocess.run([sys.executable, "sample.py", *map(str, args), "--out", str(folder)], cwd=OUT_30,
                          capture_output=True, text=True, timeout=300)
    return done.returncode, done.stderr.strip().splitlines()[-1:] if done.stderr.strip() else [], folder


script_ok_30 = False
with checked("B5.30"):
    need_model("B5.30")
    OUT_30.mkdir(parents=True, exist_ok=True)
    (OUT_30 / "sample.py").write_text(SAMPLE_PY_30, encoding="utf-8")
    print(f"ℹ️ ton script : {(OUT_30 / 'sample.py').resolve()}")
    ckpt_30 = CKPT["path"]
    code_30, err_30, dir_a = run_30(ckpt_30, "--steps", 10, "--seed", 1, "--n", 4, out="a")
    ok_30 = code_30 == 0
    verdict("B5.30", ok_30, "python sample.py <checkpoint> --steps 10 --seed 1 --n 4 : code de sortie 0.",
            f"python sample.py <checkpoint> --steps 10 --seed 1 --n 4 : code de sortie {code_30} ; {err_30}")
    if not ok_30:
        raise StopCheck
    files_30 = {p.name for p in dir_a.iterdir()}
    verdict("B5.30", {"samples.npy", "samples.png", "params.json"} <= files_30,
            "le dossier --out contient samples.npy, samples.png et params.json.",
            f"le dossier --out doit contenir samples.npy, samples.png et params.json ; il contient {sorted(files_30)}.")
    images_a = np.load(dir_a / "samples.npy")
    want_30 = wbd.sample_ddim(MODEL, torch.randn((4, 1, 28, 28), generator=torch.Generator().manual_seed(1)), AB, 10)
    ok_30 &= same("B5.30", "samples.npy : les 4 images de DDIM à 10 pas du bruit de graine 1 (copie EMA)", images_a,
                  want_30.clamp(-1, 1).numpy(), rtol=1e-4, atol=1e-4)
    params_30 = json.loads((dir_a / "params.json").read_text(encoding="utf-8"))
    need_30 = {"checkpoint_sha256", "steps", "seed", "guidance", "n", "subsequence"}
    ok_params_30 = need_30 <= set(params_30) and params_30.get("steps") == 10 and params_30.get("seed") == 1 and \
        params_30.get("subsequence") == wbd.ddim_timesteps(T_STEPS, 10).tolist()
    verdict("B5.30", ok_params_30, "params.json : l'empreinte du checkpoint, les pas, leur sous-séquence, la graine, la guidance.",
            f"params.json doit contenir au moins {sorted(need_30)}, avec les valeurs de l'appel ; reçu {params_30!r:.200}.")
    code_b, _, dir_b = run_30(ckpt_30, "--steps", 10, "--seed", 1, "--n", 4, out="b")
    same_seed_30 = code_b == 0 and np.array_equal(np.load(dir_b / "samples.npy"), images_a)
    verdict("B5.30", same_seed_30, "même graine, mêmes images, au bit près.", "deux appels de même graine doivent rendre les "
            "mêmes images, au bit près.")
    code_c, _, dir_c = run_30(ckpt_30, "--steps", 10, "--seed", 2, "--n", 4, out="c")
    other_seed_30 = code_c == 0 and not np.allclose(np.load(dir_c / "samples.npy"), images_a)
    verdict("B5.30", other_seed_30, "une autre graine donne d'autres images.", "--seed 2 doit donner d'autres images que --seed 1.")
    code_d, err_d, _ = run_30(ckpt_30, "--guidance", 2, "--n", 4, out="d")
    verdict("B5.30", code_d != 0, f"--guidance sur un modèle sans classes : refusé (code {code_d}, {err_d}).",
            "--guidance 2 sur un checkpoint sans classes doit échouer avec un message, pas générer.")
    code_e, err_e, _ = run_30(ckpt_30, "--steps", 0, out="e")
    verdict("B5.30", code_e != 0, f"--steps 0 : refusé (code {code_e}, {err_e}).", "--steps 0 doit échouer avec un message.")
    ok_30 &= ok_params_30 and same_seed_30 and other_seed_30 and code_d != 0 and code_e != 0
    ckpt27_30 = wbd.load_checkpoint(key_27(globals().get("CKPT_SOURCE_23", "learner")))
    if ckpt27_30 is None:
        print("⏳ Ex B5.30 : --guidance et --label se vérifient avec le modèle conditionnel de B5.27 (pas encore sauvegardé).")
    else:
        code_f, err_f, dir_f = run_30(ckpt27_30["path"], "--steps", 10, "--seed", 3, "--n", 4, "--guidance", 3, "--label", 7,
                                      out="f")
        model27_30 = wbd.build_from_checkpoint(wbd.TinyUNet, ckpt27_30)
        want_f = wbd.sample_ddim(model27_30, torch.randn((4, 1, 28, 28), generator=torch.Generator().manual_seed(3)), AB,
                                 10, y=torch.full((4,), 7), w=3.0).clamp(-1, 1).numpy()
        ok_30 &= code_f == 0 and same("B5.30", "--guidance 3 --label 7 sur le modèle de B5.27 : la guidance de la fiche "
                                      "(w = 3)", np.load(dir_f / "samples.npy") if code_f == 0 else None, want_f,
                                      rtol=1e-4, atol=1e-4)
    script_ok_30 = bool(ok_30)

### Ex B5.31 — Défi : la plus petite distance de Fréchet sur Fashion-MNIST en au plus 50 pas d'échantillonnage 🏆 ★★★★ ⏱️ 120 min 🚀
**Objectif :** concevoir de bout en bout un générateur de Fashion-MNIST qui s'entraîne dans un temps donné et génère en au plus 50 passes de réseau par image.  
**Prérequis :** Ex B5.25, B5.27, B5.28, B5.29 · tout le chapitre · **Fil rouge :** Fashion-MNIST · **Parcours :** C

**Le défi** : écris `method_31(images, labels, seconds)`, qui construit et entraîne **un modèle neuf**, en au plus `seconds` secondes, sur les seules images d'entraînement de Fashion-MNIST qu'elle reçoit (`images`, float32 `(N, 1, 28, 28)` dans $[-1, 1]$ ; `labels`, int64 `(N,)`), et rend une fonction `generate(n, generator)` qui rend `n` images, un tenseur float32 `(n, 1, 28, 28)` dans $[-1, 1]$. La mesure : la distance de Fréchet de `N_31` images générées aux 2 000 images de **test** de Fashion-MNIST, que ta méthode ne voit jamais (features de `CLF_F`). Objectif : au plus `GOAL_31`.

**Les règles**, contrôlées par `evaluate_31(method_31)` :
- **au plus 50 passes de réseau par image** : chaque appel d'un réseau (un `nn.Module`, quel qu'il soit, appelé comme une fonction, `model(x, …)`) pendant `generate` compte autant de passes que d'images dans son batch ; la guidance compte donc double (25 pas DDIM guidés = 50 passes, 50 pas guidés = 100) ; `generate` doit appeler au moins un réseau, et tirer son bruit du `generator` reçu (deux generators différents, deux séries d'images différentes) ;
- l'entraînement tient dans `seconds` (`LIMIT_31` : une course réduite de 45 s en mode rapide, sur 10 000 images ; **le défi** en mode complet, 🚀, 10 min sur les 60 000 images) ; la génération n'est pas chronométrée, les 50 passes la bornent ;
- **à froid** et sans fuite : pendant la course, les variables du notebook qui contiennent des images, des features, un modèle ou une sauvegarde (directement, ou dans une liste, un tuple ou un dictionnaire) sont masquées, les chargeurs de `wb` (`wbd.image_subset`, `wb.datasets`), les sauvegardes, `torch.load` et le classifieur des mesures sont neutralisés ; `method_31` ne garde rien de tel dans ses arguments par défaut ; `generate` doit appeler un réseau **neuf**, entraîné pendant la course : ni un modèle d'avant la course, ni une copie de ses poids, même réentraînée un peu (corrélation médiane des poids au plus 0,93) ;
- **pas de copie** : la part des images générées anormalement proches d'une image d'entraînement ($r < 1/3$, 🔬 B5.28) ne dépasse pas `COPIES_31` (celle de la référence plus une marge) ; vues des images d'entraînement et des images de test, tes images ne sont pas plus proches de l'un des deux ensembles que des images réelles inédites (des images de test) : un $r$ médian d'au moins 0,9 fois le leur, et un rapport médian des distances au plus proche voisin, test sur entraînement, entre 0,8 et 1,2 fois le leur ; aucune n'est une image de test.

La cellule suivante donne `evaluate_31`, `LIMIT_31`, `N_31`, `GOAL_31`, `REFERENCE_31` (la meilleure et la pire graine de la solution de référence) et `COPIES_31`. Plus la distance est petite, meilleures sont les images : `GOAL_31` est un **plafond**, fixé un peu au-dessus de la référence (sa pire graine plus 15 %), pour qu'une méthode aussi bonne qu'elle le passe quelle que soit sa graine. La référence : un U-Net conditionné de canaux (16, 32, 64), entraîné pendant le temps donné, avec une EMA, et échantillonné par DDIM guidé en 25 pas ; elle a été calibrée sur un CPU à deux cœurs (environ 500 pas d'optimisation en 45 s) : sur une machine plus lente ou chargée, la course fait moins de pas, et un GPU en fait bien plus. L'objectif du mode complet n'a été mesuré que sur deux graines. Tu peux reprendre tes fonctions du chapitre, ou faire autrement (un autre planning, une autre paramétrisation, un réseau plus petit ou plus gros, moins de pas mais une guidance différente…).

Dans tes notes : ta méthode, ce que tu as essayé, ce qui a aidé et ce qui n'a rien changé, et ta distance. Combien de pas d'optimisation ta méthode fait-elle dans le temps donné ? À 50 passes, vaut-il mieux 50 pas sans guidance ou 25 pas guidés ? Et la course réduite du mode rapide, prédit-elle le classement du défi complet ? La réussite est affichée ; elle n'est pas enregistrée : c'est une règle d'honneur.

In [ ]:
# The challenge: the smallest Fréchet distance on Fashion-MNIST with at most 50 network passes per image
MAX_PASSES_31 = 50
LIMIT_31 = 45.0 if FAST_MODE else 600.0   # seconds of training: a reduced run in fast mode, THE challenge in full mode
N_31 = 256 if FAST_MODE else 1000                     # generated images, compared to 2 000 test images
GOAL_31 = 56.0 if FAST_MODE else 12.0    # the distance to reach (worst seed of the reference + 15 %)
REFERENCE_31 = (38.5, 48.3) if FAST_MODE else (8.4, 10.1)   # the reference solution: (best, worst) of its seeds
COPIES_31 = 0.05 if FAST_MODE else 0.05   # largest share of copies (r < 1/3) allowed: reference + margin


def _make_evaluate_31():
    """evaluate_31, with the test images and their features kept here only (the global variables that hold images,
    models or checkpoints are replaced by a decoy while your method runs)."""
    import gc
    import re
    import tempfile
    import warnings
    from pathlib import Path
    from wb import datasets as wbdata

    train = wbd.image_subset("fashion", wbd.n_train(FAST_MODE))
    test = wbd.image_subset("fashion", 2000, split="test")
    clf = wbd.classifier("fashion")
    test_feats = wbd.features(clf, test.images)
    # What real images that no method has seen look like from the training images, the yardstick of two rules: 256 test
    # images, their median r (B5.28) against the training images, and the median ratio of their distance to the other
    # test images over their distance to the training images (fast mode 1.09 and 1.11; full mode 1.24 for the ratio)
    probe = test.images[:256].reshape(256, -1).double()
    real_r = float(np.median(wbd.copy_ratio(test.images[:256], train.images)))
    flat_train = train.images.reshape(len(train), -1).double()
    near_train = torch.cat([torch.cdist(probe[i:i + 64], flat_train).min(dim=1).values for i in range(0, 256, 64)])
    real_side = float(torch.median(torch.cdist(probe, test.images[256:].reshape(len(test) - 256, -1).double())
                                   .min(dim=1).values / near_train))
    del probe, near_train, flat_train

    class Hidden:
        """What a hidden global variable holds during the run: any use of it says why."""

        def __init__(self, name):
            self._name = name

        def _refuse(self, *args, **kwargs):
            raise RuntimeError(f"pendant le défi, {self._name} est masqué : ta méthode ne reçoit que images, labels et "
                               "seconds")

        __getattr__ = __getitem__ = __call__ = __iter__ = __len__ = _refuse

    ipython_names = re.compile(r"^(_+|_i+|_i?\d+|_[ido]h|In|Out|exit|quit|get_ipython)$")

    def hideable(v, level=0):
        """True for what could hold images, features, a trained model or a checkpoint (also inside a list, a tuple,
        a set or a dict, up to three levels down)."""
        if isinstance(v, wbd.ImageSet):
            return True
        if isinstance(v, nn.Module):
            return any(p.numel() >= 1000 for p in v.parameters())
        if isinstance(v, torch.Tensor):
            return v.numel() >= 784 * 10
        if isinstance(v, np.ndarray):
            return v.size >= 784 * 10
        if isinstance(v, dict):
            if "state_dict" in v or "ema_state_dict" in v:
                return True
            return level < 3 and any(hideable(x, level + 1) for x in list(v.values())[:10_000])
        if isinstance(v, (list, tuple, set, frozenset)):
            return level < 3 and any(hideable(x, level + 1) for x in list(v)[:10_000])
        return False

    def hidden_globals():
        """The global variables that could hold images, features, a trained model or a checkpoint."""
        g, found = globals(), {}
        for n, v in list(g.items()):
            if ipython_names.match(n) or n.startswith("_"):
                continue
            if hideable(v):
                found[n] = Hidden(n)
        return found

    def captured(method):
        """What method keeps in its default arguments or its closure that a hidden variable would hold."""
        kept = list(getattr(method, "__defaults__", None) or ()) + list((getattr(method, "__kwdefaults__", None)
                                                                          or {}).values())
        kept += [cell.cell_contents for cell in (getattr(method, "__closure__", None) or ())
                 if not isinstance(cell.cell_contents, type(method))]
        return any(hideable(v) for v in kept)

    def refused(*args, **kwargs):
        raise RuntimeError("pendant le défi, ta méthode ne reçoit que images, labels et seconds (ni données du workbook, "
                           "ni sauvegarde, ni modèle déjà entraîné, ni classifieur des mesures)")

    def tensors_alive():
        """Every floating-point tensor of the session before the run (parameters, buffers, copies kept anywhere), by
        shape, and their memory addresses: a network of the run must not start from one of them."""
        by_shape, pointers = {}, set()
        with warnings.catch_warnings():               # isinstance on every object wakes deprecated torch aliases
            warnings.simplefilter("ignore")
            for obj in gc.get_objects():
                if isinstance(obj, torch.Tensor) and obj.is_floating_point() and 1000 < obj.numel() <= 4_000_000:
                    by_shape.setdefault(tuple(obj.shape), []).append(obj)
                    pointers.add(obj.data_ptr())
        return by_shape, pointers

    def from_before(p, by_shape, pointers):
        """The largest correlation of a parameter with a tensor of the same shape that existed before the run (1 for
        the same memory): about 0.96 for a model of the notebook fine-tuned 400 steps, below 0.91 for a new model,
        even with the seed and the initialisation of a model of the notebook (history of session 72)."""
        if p.data_ptr() in pointers:
            return 1.0
        x = p.detach().flatten().double()
        best = 0.0
        for q in by_shape.get(tuple(p.shape), []):
            c = torch.corrcoef(torch.stack([x, q.detach().flatten().double()]))[0, 1]
            if torch.isfinite(c):
                best = max(best, float(c))
        return best

    def run(method, seconds):
        """Train the method cold, then generate N_31 images, counting the passes of every network."""
        g = globals()
        if captured(method):
            print("❌ Ex B5.31 : method_31 garde, dans ses arguments par défaut ou sa fermeture, des images, des features, "
                  "un modèle ou une sauvegarde du notebook : elle ne doit utiliser que images, labels et seconds.")
            return None
        hidden = hidden_globals()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            modules_before = {id(m): m for m in gc.get_objects() if isinstance(m, nn.Module)}   # kept alive: ids stay unique
        by_shape, pointers = tensors_alive()
        saved = {name: getattr(wbd, name) for name in ("image_subset", "raw_subset", "_raw", "load_checkpoint",
                                                        "reference_model", "classifier", "features", "predict",
                                                        "checkpoints_dir")}
        saved_data = {name: getattr(wbdata, name) for name in ("load_mnist", "load_fashion_mnist")}
        saved_torch = (torch.load, torch.serialization.load)
        empty = Path(tempfile.mkdtemp(prefix="wb_b5_31_"))
        originals = {n: g[n] for n in hidden}
        caches = [(cache, dict(cache)) for cache in (wbd._RAW, wbd._CLASSIFIERS, wbdata._ARRAYS)]
        counts, depth, used, pending = {"passes": 0}, [0], [], [False]

        def pre(module, args):
            # a pass: one call of a network from outside any network, as many passes as images in its batch (the first
            # tensor among its positional arguments; for a call by keywords, the batch of its first inner module)
            first = next((a for a in args if torch.is_tensor(a) and a.ndim > 0), None)
            if depth[0] == 0:
                used.append(module)
                if first is not None:
                    counts["passes"] += int(first.shape[0])
                else:
                    pending[0] = True
            elif pending[0] and first is not None:
                counts["passes"] += int(first.shape[0])
                pending[0] = False
            depth[0] += 1

        def post(module, args, output):
            depth[0] -= 1
            if depth[0] == 0:
                pending[0] = False

        out = again = elapsed = None
        try:
            g.update(hidden)
            for cache, _ in caches:                           # the images and models kept in memory by wb
                cache.clear()
            for name in saved:
                setattr(wbd, name, refused)
            wbd.checkpoints_dir = lambda: empty
            for name in saved_data:
                setattr(wbdata, name, refused)
            torch.load = torch.serialization.load = refused
            clf.forward = clf.features = refused              # the classifier of the measures, wherever it is kept
            torch.manual_seed(3131)
            start = time.perf_counter()
            generate = method(train.images.clone(), train.labels.clone(), seconds)
            elapsed = time.perf_counter() - start
            if not callable(generate):
                print(f"❌ Ex B5.31 : method_31 doit rendre une fonction generate(n, generator) ; reçu {type(generate).__name__}.")
                return None
            handles = (nn.modules.module.register_module_forward_pre_hook(pre),
                       nn.modules.module.register_module_forward_hook(post))
            try:
                with torch.no_grad():
                    out = generate(N_31, torch.Generator().manual_seed(3132))
            finally:
                for h in handles:
                    h.remove()
            with torch.no_grad():
                again = generate(16, torch.Generator().manual_seed(3133))    # another generator: other images
        finally:
            g.update(originals)
            for cache, content in caches:
                cache.clear()
                cache.update(content)
            for name, f in saved.items():
                setattr(wbd, name, f)
            for name, f in saved_data.items():
                setattr(wbdata, name, f)
            torch.load, torch.serialization.load = saved_torch
            for name in ("forward", "features"):
                clf.__dict__.pop(name, None)
        if not (torch.is_tensor(out) and tuple(out.shape) == (N_31, 1, 28, 28) and torch.isfinite(out).all()):
            print(f"❌ Ex B5.31 : generate(n, generator) doit rendre un tenseur fini de forme ({N_31}, 1, 28, 28) ; reçu "
                  f"{type(out).__name__} {tuple(getattr(out, 'shape', ()))}.")
            return None
        if not counts["passes"]:
            print("❌ Ex B5.31 : generate n'a appelé aucun réseau : les images doivent venir d'un réseau entraîné pendant "
                  "la course, appelé comme une fonction, model(x, …).")
            return None
        if torch.is_tensor(again) and again.shape == out[:16].shape and torch.allclose(again.float(), out[:16].float()):
            print("❌ Ex B5.31 : generate rend les mêmes images avec deux generators différents : le bruit de départ doit "
                  "être tiré du generator reçu.")
            return None
        params = [p for m in used for p in m.parameters() if p.numel() > 1000]
        median_close = float(np.median([from_before(p, by_shape, pointers) for p in params])) if params else 0.0
        if any(id(m) in modules_before for m in used) or median_close > 0.93:
            print("❌ Ex B5.31 : generate doit utiliser un modèle neuf, entraîné pendant la course : un réseau appelé existait "
                  f"déjà, ou ses poids viennent d'un modèle d'avant la course (corrélation médiane {fr(median_close, 2)}, "
                  "au plus 0,93).")
            return None
        return out.float().clamp(-1, 1), elapsed, counts["passes"] / N_31

    def evaluate_31(method):
        """method(images, labels, seconds) -> generate(n, generator): the rules of the statement, then the distance."""
        result = run(method, LIMIT_31)
        if result is None:
            return None
        images, seconds, passes = result
        feats = wbd.features(clf, images)
        fd = float(mylearn.gen_metrics.frechet_distance_from_features(test_feats, feats))
        r = wbd.copy_ratio(images, train.images)
        copies, median_r = float(np.mean(r < 1 / 3)), float(np.median(r))
        flat = images.reshape(N_31, -1).double()
        d_test = torch.cdist(flat, test.images.reshape(len(test), -1).double()).min(dim=1).values
        flat_train = train.images.reshape(len(train), -1).double()
        d_train = torch.cat([torch.cdist(flat[i:i + 64], flat_train).min(dim=1).values for i in range(0, N_31, 64)])
        del flat_train
        to_test = float(d_test.min())
        test_side = float(torch.median(d_test / d_train))
        show_images(images[:32], ncol=16, size=0.75, suptitle="32 des images générées par ta méthode")
        print(f"entraînement en {fr(seconds, 1)} s ; {fr(passes, 1)} passes de réseau par image ; distance de Fréchet "
              f"{fr(fd, 2)} ({N_31} images contre 2 000 images de test) ; copies (r < 1/3) : {fr(100 * copies, 1)} %.")
        rules = [(seconds <= LIMIT_31 * 1.05 + 2, f"entraînement en {fr(seconds, 1)} s, dans les {LIMIT_31:.0f} s.",
                  f"entraînement en {fr(seconds, 1)} s : le défi en permet {LIMIT_31:.0f} (ta méthode doit respecter seconds)."),
                 (passes <= MAX_PASSES_31, f"{fr(passes, 1)} passes de réseau par image, au plus {MAX_PASSES_31}.",
                  f"{fr(passes, 1)} passes de réseau par image (la guidance compte double) : le défi en permet "
                  f"{MAX_PASSES_31}."),
                 (copies <= COPIES_31, f"contrôle des copies : {fr(100 * copies, 1)} % d'images de r < 1/3 (au plus "
                                       f"{fr(100 * COPIES_31, 0)} %, la référence plus une marge).",
                  f"contrôle des copies : {fr(100 * copies, 1)} % des images sont anormalement proches d'une image "
                  f"d'entraînement (r < 1/3 ; au plus {fr(100 * COPIES_31, 0)} %) : ta méthode recopie-t-elle ?"),
                 (median_r >= 0.9 * real_r and test_side <= 1.2 * real_side,
                  f"tes images ne sont pas plus proches des images d'entraînement que des images réelles inédites (r médian "
                  f"{fr(median_r, 2)}, contre {fr(real_r, 2)} ; rapport des distances test / entraînement "
                  f"{fr(test_side, 2)}, contre {fr(real_side, 2)}).",
                  f"tes images sont plus proches des images d'entraînement que des images réelles inédites (r médian "
                  f"{fr(median_r, 2)}, au moins {fr(0.9 * real_r, 2)} ; rapport des distances test / entraînement "
                  f"{fr(test_side, 2)}, au plus {fr(1.2 * real_side, 2)}) : ta méthode recopie-t-elle ?"),
                 (to_test > 1.0 and test_side >= 0.8 * real_side, "aucune image n'est une image de test, ni plus proche "
                                                                  "du test que ne l'est une image réelle inédite.",
                  f"tes images sont plus proches des images de test que des images réelles inédites (rapport des distances "
                  f"test / entraînement {fr(test_side, 2)}, au moins {fr(0.8 * real_side, 2)} ; plus petite distance au "
                  f"test {fr(to_test, 2)}) : le test ne doit jamais servir.")]
        for ok, success, failure in rules:
            verdict("B5.31", ok, success, failure)
        ok = all(rule[0] for rule in rules)
        verdict("B5.31", ok and fd <= GOAL_31,
                f"défi réussi : distance {fr(fd, 2)} (objectif {fr(GOAL_31, 1)} ; référence de {fr(REFERENCE_31[0], 1)} à "
                f"{fr(REFERENCE_31[1], 1)}).",
                f"pas encore : distance {fr(fd, 2)}, il en faut au plus {fr(GOAL_31, 1)} en respectant les règles.")
        return {"fd": fd, "seconds": seconds, "passes": passes, "copies": copies, "ok": ok and fd <= GOAL_31}

    return evaluate_31


evaluate_31 = _make_evaluate_31()
print(f"Course de {LIMIT_31:.0f} s d'entraînement, {MAX_PASSES_31} passes par image ; objectif : une distance de Fréchet "
      f"d'au plus {fr(GOAL_31, 1)} sur {N_31} images.")

In [ ]:
SEED_31 = 3100          # the seed of the reference (history_chB5.py calibrates the goal on several seeds)
W_31 = 2.0 if FAST_MODE else 1.0          # the guidance of the reference (history_chB5.py: the best worst seed)
STEPS_31 = 25   # DDIM steps: 2 passes per step with guidance, so 25 steps = 50 passes


def method_31(images, labels, seconds):
    """Your method: build and train, in at most `seconds` seconds, a NEW model on images (float32 (N, 1, 28, 28) in
    [-1, 1], Fashion-MNIST) and labels (int64 (N,)) ONLY; return a function generate(n, generator) that returns n
    images, a float32 tensor (n, 1, 28, 28) in [-1, 1], with at most 50 network passes per image."""
    start = time.perf_counter()
    torch.manual_seed(SEED_31)
    generator = torch.Generator().manual_seed(SEED_31 + 1)
    T = T_STEPS
    ab_np = AB                                                         # the schedule of the notebook
    ab = torch.tensor(ab_np, dtype=torch.float32)
    model = UNET_CLASS(T, channels=(16, 32, 64), n_classes=10)
    ema = copy.deepcopy(model)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
    budget, step = 0.95 * seconds, 0
    model.train()
    while time.perf_counter() - start < budget:
        idx = torch.randint(len(images), (32,), generator=generator)
        x0, y = images[idx], labels[idx]
        t = torch.randint(0, T, (32,), generator=generator)
        eps = torch.randn(x0.shape, generator=generator)
        y = torch.where(torch.rand(32, generator=generator) < 0.1, torch.full_like(y, 10), y)
        a = ab[t].view(-1, 1, 1, 1)
        loss = F.mse_loss(model(a.sqrt() * x0 + (1 - a).sqrt() * eps, t, y), eps)
        opt.zero_grad()
        loss.backward()
        opt.step()
        ema_update_23(ema, model, min(0.99, (1 + step) / (10 + step)))    # an EMA that warms up
        step += 1
    ema.eval()

    def generate(n, generator):
        x_T = torch.randn((n, 1, 28, 28), generator=generator)
        y = torch.arange(n) % 10                                        # as many images of each class
        return sample_cond_27(ema, x_T, y, W_31, ab_np, STEPS_31).clamp(-1, 1)

    return generate


result_31 = None
with checked("B5.31"):
    if globals().get("method_31") is None:
        raise StopCheck
    result_31 = evaluate_31(method_31)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer à la main un planning de bruit, ses $\bar\alpha_t$ et son rapport signal sur bruit ?
2. Sais-tu dériver la forme fermée et le posterior d'un pas, puis calculer un pas DDPM et un pas DDIM ?
3. Sais-tu appliquer la guidance sans classifieur et traduire `guidance_scale` en $w$ ?
4. Sais-tu estimer le coût d'un échantillonnage, et ce que gagnent DDIM, la distillation et l'espace latent ?

**Pour aller plus loin** : l'article de Ho, Jain et Abbeel et le billet de L. Weng, cités dans la fiche. La suite : le bonus B6, l'explicabilité et l'équité.